# 13_VIB60ABC — VIB A / B-sym / B analyses for ELAPS

This notebook produces the paired VIB-labelled analyses reported in the manuscript. Run the cells of **Section A** in the order shown; Section B is independent. Code comments are partly in Vietnamese.

**Section A — day-60 cohort with a common administrative endpoint (94,453 customers)**
1. *ELAPS — RUN_AUDIT CHO VIB*: with `RUN = ()` it only loads the data and the feature builder used by Steps 2–3. Set `RUN = ('H180', 'ADMIN')` to reproduce B-sym-rs, the lag curve and the train-on-B/score-on-A check for both VIB cohorts (Results text, Table 6).
2. *HƯỚNG 2 — BƯỚC 1* and *BƯỚC 1b*: rebuild VIB Branch B with the Branch A feature pipeline and write `VIB_FLDC_60D_B_REBUILT.csv`; Step 1b must print `AUC B mới = 0.922914` and `KHỚP`.
3. *VIB60ABC — A / B / B-sym + R50 + H=180* and `run_all()`: four-learner 50-refit study (Table 7, Supplementary Table S18). Its Part II (H180 from processed files) needs the earlier H180 run folder; H180 is reproduced in Section B instead.
4. *HƯỚNG 2 — BƯỚC 3: TABLE 5*: seed-42 split, XGBoost, 1,000 paired bootstrap resamples (Table 5).

**Section B — H180 cohort (31,149 customers, equal 180-day follow-up)**: A / B-sym / B with 1,000 paired bootstrap resamples (Table 6, Supplementary Table S19).

**Section L — legacy cells**: built on the historical Branch B matrix (`VIB_FLDC_60D.csv`, AUC 0.960688), whose aggregation rules differ from Branch A. They are kept only for transparency and are **not** used for any reported result.

## Section A — Day-60 cohort, common administrative endpoint
Step 1 (load data and feature builder; optional probes), Steps 2a–2b (rebuild Branch B), Step 3 (R50), Step 4 (Table 5).

In [2]:
# -*- coding: utf-8 -*-
# =====================================================================
# ELAPS — RUN_AUDIT CHO VIB (dán vào MỘT ô Colab, CHẠY ĐỘC LẬP)
# Không cần chạy ô nào khác trước. Hai cohort:
#   H180  : landmark 60 ngày, theo dõi đủ 180 ngày (31.149 khách)
#   ADMIN : mốc hành chính cuối 2019 (94.453 khách)
# Nhánh: A, B, B->A, Bsym_rs, Bsym_rs->A, lag7, lag30, lag60
#   Bsym_rs : negative mượn độ trễ (e - r) của positive TRAIN cùng nhóm
#             thập phân ngày mở quan hệ; cắt tại T
#   lag k   : positive cắt tại max(e - k ngày, r + 1 ngày); negative ghép
#             theo thời điểm vào với độ trễ đã dời
# KHÔNG tạo lại B-sym gốc (giữ số đã công bố).
#
# CỔNG KIỂM TRA (dừng nếu không qua, không nới sai số):
#   ADMIN: split E2/nhãn đúng; A, B ra đúng AUC 0.910242 / 0.960688;
#          hàm dựng đặc trưng tái tạo đúng từng giá trị A và B đã lưu.
#   H180 : cohort đúng 31.149 / 1.454 sự kiện; split train/test đúng;
#          A, B (B dựng lại từ dữ liệu gốc) ra đúng AUC 0.891002 / 0.906161.
# KẾT QUẢ: mỗi cohort 3 CSV (_results, _boot, _balance) + manifest.json
# =====================================================================
from __future__ import annotations

import gc
import importlib.metadata
import inspect
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

# ---------------------------------------------------------------- cấu hình
RUN = ()                         # chỉ nạp hàm + dữ liệu; ô kế tiếp tạo file B mới
ROOT = Path('/content/drive/MyDrive/Fintect')
A_FILE = ROOT / 'final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv'
B_FILE = ROOT / 'VIB_FLDC_60D.csv'
SPLIT_FILE = ROOT / 'FINAL_ABC_MATCHED_E2/exact_E2_shared_train_test_split.csv'
CV_FILE = ROOT / 'FINAL_ABC_MATCHED_E2/exact_E2_shared_CV_folds.csv'
CUSTOMER_FILE = ROOT / '1.Data_Customer.csv'
CARD_FILE = ROOT / '6.Data_Card.xlsx'
TRANSACTION_FILE = ROOT / '2.Data_MyVIB_Transaction.csv'
ACTIVITY_FILE = ROOT / '3.Data_MyVIB_Activity.csv'
DEPOSIT_FILE = ROOT / '4.Data_Deposit.csv'
LENDING_FILE = ROOT / '5.Data_Lending.xlsx'
VERIFIED_DATES_CSV = ''          # tuỳ chọn: CUSTOMER_NUMBER + RELATIONSHIP_START_DATE
AUDIT_ROOT = ROOT / 'ELAPS_RUN_AUDIT'

ID = 'CUSTOMER_NUMBER'
TARGET = 'TARGET_60D'
START = 'RELATIONSHIP_START_DATE'
EVENT = 'FIRST_CARD_MONTH'
CUTOFF = 'FEATURE_CUTOFF_DATE'
DATE_END_EXCLUSIVE = pd.Timestamp('2020-01-01')
OBSERVATION_END = pd.Timestamp('2019-12-31 23:59:59.999999999')
DAY60, HORIZON = 60, 180
CHUNK = 200_000
VERIFIED_DAYFIRST = {'activity': False, 'deposit': False}
VERIFY_ADMIN_BUILDER = True      # bắt buộc trước khi tin các nhánh mới

LAGS = (7, 30, 60)
N_STRATA = 10
SEED_RS = 20261120               # lần rút mới, KHÔNG thay B-sym đã công bố
N_BOOT = 1000
SEED_BOOT = 42
BASELINE_TOL = 5e-6
REFERENCE = {'H180': {'A': 0.891002, 'B': 0.906161},
             'ADMIN': {'A': 0.910242, 'B': 0.960688}}

STATIC = ['CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'AGE']
ACTIVITY = ['LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO']
TRANSACTION = ['TRANS_LV1_MODE', 'TRANS_LV2_MODE', 'TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']
DEPOSIT = ['COUNT_CA_ACCT', 'AVG_CA_BALANCE', 'COUNT_TD_ACCT', 'AVG_TD_BALANCE']
LENDING = ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT', 'TOTAL_LOAN_AMOUNT']
FEATURES = STATIC + ACTIVITY + TRANSACTION + DEPOSIT + LENDING
DYNAMIC = ACTIVITY + TRANSACTION + DEPOSIT + LENDING
assert len(FEATURES) == 18
XGB_ADMIN = dict(n_estimators=300, max_depth=6, learning_rate=.05, subsample=.8,
                 colsample_bytree=.8, objective='binary:logistic', eval_metric='logloss',
                 random_state=42, n_jobs=2)
XGB_H180 = dict(XGB_ADMIN, n_jobs=-1)
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


def fail(msg):
    raise RuntimeError('DỪNG: ' + msg)


# ================================================================ ĐỌC DỮ LIỆU
def standardize_id(s):
    x = s.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    x = x.mask(x.str.upper().isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)
    nums = x.str.fullmatch(r'\d+', na=False)
    x.loc[nums] = x.loc[nums].str.lstrip('0').replace('', '0')
    return x


def clean_cat(s):
    x = s.astype('string').str.strip().str.upper()
    return x.mask(x.isin(['', 'NAN', 'NA', '<NA>', 'NULL', 'NONE']), pd.NA)


def read_csv_exact(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(str(path))
    h = pd.read_csv(path, nrows=0)
    fields = {str(c).strip().upper(): c for c in h.columns}
    if ID not in fields:
        fail(f'{path}: thiếu {ID}')
    d = pd.read_csv(path, dtype={fields[ID]: str}, low_memory=False)
    d.columns = d.columns.str.strip().str.upper()
    d[ID] = standardize_id(d[ID])
    if d[ID].isna().any() or d[ID].duplicated().any():
        fail(f'{ID} thiếu/trùng trong {path}')
    return d


def date_parse(series, name, source=None):
    dfirst = VERIFIED_DAYFIRST.get(source or 'generic')
    textual = series.astype('string').str.strip()
    parts = textual.str.extract(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)')
    first = pd.to_numeric(parts[0], errors='coerce')
    second = pd.to_numeric(parts[1], errors='coerce')
    ambiguous = first.between(1, 12) & second.between(1, 12) & first.ne(second)
    if dfirst is None and ambiguous.fillna(False).any():
        fail(f'{name}: ngày mơ hồ dd/mm hoặc mm/dd: {textual[ambiguous.fillna(False)].head().tolist()}')
    if dfirst is False and (first > 12).fillna(False).any():
        fail(f'{name}: ngày trái quy ước mm/dd/yyyy')
    if dfirst is True and (second > 12).fillna(False).any():
        fail(f'{name}: ngày trái quy ước dd/mm/yyyy')
    s = pd.to_datetime(series, format='mixed', dayfirst=bool(dfirst), errors='coerce')
    missing = series.isna() | clean_cat(series).isna()
    if (s.isna() & ~missing).any():
        fail(f'{name}: ngày không đọc được {series[s.isna() & ~missing].head().tolist()}')
    return s


def iter_table(path, columns):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu nguồn gốc: {path}')
    if path.suffix.lower() in ('.xlsx', '.xls'):
        header = pd.read_excel(path, nrows=0)
        mapping = {str(x).strip().upper(): x for x in header.columns}
        if any(c not in mapping for c in columns):
            fail(f'{path}: thiếu {[c for c in columns if c not in mapping]}')
        dat = pd.read_excel(path, usecols=[mapping[c] for c in columns], dtype={mapping[ID]: str})
        dat.columns = dat.columns.str.strip().str.upper()
        dat[ID] = standardize_id(dat[ID])
        yield dat[columns]
    else:
        header = pd.read_csv(path, nrows=0)
        mapping = {str(x).strip().upper(): x for x in header.columns}
        if any(c not in mapping for c in columns):
            fail(f'{path}: thiếu {[c for c in columns if c not in mapping]}')
        for dat in pd.read_csv(path, usecols=[mapping[c] for c in columns],
                               dtype={mapping[ID]: str}, chunksize=CHUNK, low_memory=False):
            dat.columns = dat.columns.str.strip().str.upper()
            dat[ID] = standardize_id(dat[ID])
            yield dat[columns]


def load_inputs():
    a, b = read_csv_exact(A_FILE), read_csv_exact(B_FILE)
    s, cv = read_csv_exact(SPLIT_FILE), read_csv_exact(CV_FILE)
    for name, df in (('A', a), ('B', b)):
        if [f for f in FEATURES if f not in df]:
            fail(f'{name} thiếu biến')
        if len(df) != 94453:
            fail(f'{name}: N={len(df)}, cần 94453')
    if set(a[ID]) != set(b[ID]) or set(a[ID]) != set(s[ID]):
        fail('A/B/split khác tập khách hàng')
    a = a.reset_index(drop=True)
    b = b.set_index(ID).loc[a[ID]].reset_index()
    s = s.set_index(ID).loc[a[ID]].reset_index()
    y = pd.to_numeric(a[TARGET], errors='raise').to_numpy(dtype=np.int8)
    if int(y.sum()) != 4374:
        fail(f'nhãn E2 sai: {int(y.sum())}')
    if TARGET in b and not np.array_equal(pd.to_numeric(b[TARGET]).to_numpy(dtype=np.int8), y):
        fail('nhãn B khác A')
    split = s['SPLIT'].astype(str).str.strip().str.upper()
    train_ids = s.loc[split == 'TRAIN', ID].tolist()
    test_ids = s.loc[split.isin(['TEST', 'HOLDOUT']), ID].tolist()
    if (len(train_ids), len(test_ids)) != (75562, 18891):
        fail('kích thước train/test E2 sai')
    if '_E2_ORDER' not in cv:
        fail('file CV thiếu _E2_ORDER')
    order = cv.sort_values('_E2_ORDER')[ID].tolist()
    if set(order) != set(train_ids):
        fail('ID train trong CV khác split')
    ids = a[ID].tolist()
    index = pd.Series(np.arange(len(a)), index=ids)
    tr = index.loc[order].to_numpy(dtype=int)
    te = index.loc[test_ids].to_numpy(dtype=int)
    if int(y[tr].sum()) != 3499 or int(y[te].sum()) != 875:
        fail('số sự kiện train/test E2 sai')
    print(f'E2: N={len(ids):,}; sự kiện={y.sum():,}; train/test={len(tr):,}/{len(te):,}')
    return a, b, ids, y, tr, te


def make_master(a, ids, y):
    if VERIFIED_DATES_CSV:
        dates = read_csv_exact(VERIFIED_DATES_CSV)
        starts = date_parse(dates.set_index(ID)[START].reindex(ids).reset_index(drop=True), START)
    elif START in a:
        starts = date_parse(a[START], START)
    elif 'CLIENT_CREATE_DATE' in a:
        starts = date_parse(a['CLIENT_CREATE_DATE'], 'A.CLIENT_CREATE_DATE')
    else:
        print('Đọc ngày mở quan hệ từ dữ liệu khách hàng...')
        chunks, idset = [], set(ids)
        for d in iter_table(CUSTOMER_FILE, [ID, 'CLIENT_CREATE_DATE']):
            d = d[d[ID].isin(idset)].copy()
            d['CLIENT_CREATE_DATE'] = date_parse(d['CLIENT_CREATE_DATE'], 'customer.CLIENT_CREATE_DATE')
            chunks.append(d)
        starts = pd.Series(ids).map(pd.concat(chunks).groupby(ID)['CLIENT_CREATE_DATE'].min())
    if starts.isna().any():
        fail(f'{starts.isna().sum()} khách thiếu ngày mở quan hệ')
    master = pd.DataFrame({ID: ids, START: pd.to_datetime(starts).to_numpy(), TARGET: y})
    master['A_CUTOFF'] = master[START] + pd.to_timedelta(DAY60, unit='D')
    print('Đọc ngày mở thẻ đầu tiên...')
    parts, idset = [], set(ids)
    for d in iter_table(CARD_FILE, [ID, 'MONTH', 'COUNT_CREDITCARD']):
        d = d[d[ID].isin(idset)].copy()
        if d.empty:
            continue
        d['MONTH'] = date_parse(d['MONTH'], 'card.MONTH')
        d['COUNT_CREDITCARD'] = pd.to_numeric(d['COUNT_CREDITCARD'], errors='coerce')
        if d['MONTH'].isna().any() or d['COUNT_CREDITCARD'].isna().any():
            fail('dữ liệu thẻ thiếu ngày/trạng thái')
        d = d[d['COUNT_CREDITCARD'] > 0]
        if not d.empty:
            parts.append(d.groupby(ID)['MONTH'].min())
    event = pd.concat(parts).groupby(level=0).min() if parts else pd.Series(dtype='datetime64[ns]')
    master[EVENT] = master[ID].map(event)
    obs = ((master[EVENT].notna()) & (master[EVENT] > master['A_CUTOFF'])
           & (master[EVENT] < DATE_END_EXCLUSIVE)).astype(np.int8)
    if not np.array_equal(obs.to_numpy(), y):
        fail(f'nhãn tính từ ngày thẻ lệch {int((obs.to_numpy() != y).sum())} khách')
    master['B_CUTOFF'] = master[EVENT].where(master[TARGET].eq(1), DATE_END_EXCLUSIVE)
    return master


# ================================================================ DỰNG ĐẶC TRƯNG
def rows_for_window(path, cols, master, date_col):
    ids, pieces = set(master[ID]), []
    for d in iter_table(path, cols):
        d = d[d[ID].isin(ids)]
        if d.empty:
            continue
        df = d.merge(master[[ID, START, CUTOFF]], on=ID, how='inner', validate='many_to_one')
        source = ('activity' if date_col == 'ACTIVITY_DATE' else
                  'deposit' if date_col == 'MONTH' and 'COUNT_CA_ACCT' in df.columns else 'generic')
        df[date_col] = date_parse(df[date_col], date_col, source)
        df = df[(df[date_col] >= df[START]) & (df[date_col] < df[CUTOFF])]
        if not df.empty:
            pieces.append(df.copy())
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=cols + [START, CUTOFF])


def mode_by_customer(df, field, label):
    if df.empty:
        return pd.Series(dtype=object, name=label)
    d = df[[ID]].copy()
    d['cat'] = clean_cat(df[field])
    d = d.dropna()
    counts = d.groupby([ID, 'cat']).size().rename('n').reset_index()
    win = counts.sort_values([ID, 'n', 'cat'], ascending=[True, False, True],
                             kind='stable').drop_duplicates(ID)
    return win.set_index(ID)['cat'].rename(label)


def build_features(master, base_a, label):
    """13 biến động từ dữ liệu gốc với r <= thời điểm < cutoff; 5 biến tĩnh lấy từ A."""
    print(f'  Dựng {label}: giao dịch / hoạt động / tiền gửi / khoản vay')
    m = master[[ID, START, CUTOFF]].copy()
    ids = m[ID].tolist()
    base = pd.DataFrame({ID: ids}).set_index(ID)
    tr = rows_for_window(TRANSACTION_FILE, [ID, 'TRANS_DATE', 'TRANS_LV1', 'TRANS_LV2', 'TRANS_AMOUNT'],
                         m, 'TRANS_DATE')
    if len(tr):
        amt = pd.to_numeric(tr['TRANS_AMOUNT'].astype('string').str.replace(',', '', regex=False),
                            errors='coerce')
        if (amt.isna() & tr['TRANS_AMOUNT'].notna()).any():
            fail('số tiền giao dịch không hợp lệ')
        tr['AMOUNT'] = amt
        num = tr.groupby(ID)['AMOUNT'].agg(['max', 'mean'])
        base['TRANS_AMOUNT_MAX'] = num['max'].reindex(base.index)
        base['TRANS_AMOUNT_MEAN'] = num['mean'].reindex(base.index)
        base['TRANS_LV1_MODE'] = mode_by_customer(tr, 'TRANS_LV1', 'TRANS_LV1_MODE')
        base['TRANS_LV2_MODE'] = mode_by_customer(tr, 'TRANS_LV2', 'TRANS_LV2_MODE')
    else:
        base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN', 'TRANS_LV1_MODE', 'TRANS_LV2_MODE']] = np.nan
    base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']] = base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']].fillna(0)
    base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']] = base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']].fillna('NO_TRANSACTION')
    del tr
    act = rows_for_window(ACTIVITY_FILE, [ID, 'ACTIVITY_DATE', 'ACTIVITY_NAME'], m, 'ACTIVITY_DATE')
    if len(act):
        cat = clean_cat(act['ACTIVITY_NAME'])
        act['IS_LOGIN'] = cat.str.contains('LOGIN', regex=False, na=False).astype(int)
        act['IS_INTEREST'] = cat.str.contains('INTEREST_RATE', regex=False, na=False).astype(int)
        act['DAY'] = act['ACTIVITY_DATE'].dt.normalize()
        g = act.groupby(ID)
        ndays = act[[ID, 'DAY']].drop_duplicates().groupby(ID).size()
        base['LOGIN_PER_ACTIVE_DAY'] = (g['IS_LOGIN'].sum() / ndays).reindex(base.index)
        base['INTEREST_RATE_RATIO'] = (g['IS_INTEREST'].sum() / g.size()).reindex(base.index)
    else:
        base[ACTIVITY] = np.nan
    base[ACTIVITY] = base[ACTIVITY].fillna(0)
    del act
    for source, path, cols in (('deposit', DEPOSIT_FILE, ['COUNT_CA_ACCT', 'AVG_CA_BALANCE',
                                                          'COUNT_TD_ACCT', 'AVG_TD_BALANCE']),
                               ('lending', LENDING_FILE, ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT'])):
        d = rows_for_window(path, [ID, 'MONTH'] + cols, m, 'MONTH')
        if len(d):
            for col in cols:
                val = pd.to_numeric(d[col].astype('string').str.replace(',', '', regex=False), errors='coerce')
                if (val.isna() & d[col].notna()).any():
                    fail(f'{source}/{col} không hợp lệ')
                d[col] = val
            d = d.sort_values([ID, 'MONTH'], kind='stable')
            ties = d[d.duplicated([ID, 'MONTH'], keep=False)]
            if not ties.empty and ties.groupby([ID, 'MONTH'])[cols].nunique(dropna=False).gt(1).any(axis=1).any():
                fail(f'{source}: snapshot cùng tháng mâu thuẫn')
            latest = d.drop_duplicates([ID], keep='last').set_index(ID)
            for col in cols:
                base[col] = latest[col].reindex(base.index)
        else:
            base[cols] = np.nan
        base[cols] = base[cols].fillna(0)
        del d
    for col in ('COUNT_CA_ACCT', 'COUNT_TD_ACCT', 'COUNT_OF_LOAN'):
        if (base[col] < 0).any():
            fail('số đếm âm: ' + col)
        base[col] = base[col].round().astype(int)
    base['TOTAL_LOAN_AMOUNT'] = base['COUNT_OF_LOAN'] * base['AVG_LOAN_AMOUNT']
    static = base_a.set_index(ID).loc[ids, STATIC]
    out = pd.concat([static, base[DYNAMIC]], axis=1).loc[:, FEATURES].reset_index(drop=True)
    out.insert(0, ID, ids)
    gc.collect()
    return out


def verify_rebuild(label, saved, rebuilt, numeric):
    bad = []
    for col in DYNAMIC:
        x, z = saved[col], rebuilt[col]
        if col in numeric:
            neq = ~np.isclose(pd.to_numeric(x, errors='coerce').to_numpy(float),
                              pd.to_numeric(z, errors='coerce').to_numpy(float),
                              atol=1e-8, rtol=1e-12, equal_nan=True)
        else:
            neq = (clean_cat(x).fillna('MISSING') != clean_cat(z).fillna('MISSING')).to_numpy()
        if neq.sum():
            bad.append((col, int(neq.sum())))
    print(f'  {label}: số giá trị lệch = {bad or 0}')
    return bad


def frame_aligned(df, ids):
    if df[ID].duplicated().any() or set(df[ID]) != set(ids):
        fail('sai tập khách hàng trong ma trận')
    return df.set_index(ID).loc[ids].reset_index()


# ================================================================ MÔ HÌNH
def onehot():
    kw = {'handle_unknown': 'ignore'}
    kw['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse'] = False
    return OneHotEncoder(**kw)


# ADMIN: đúng pipeline ô "VIB day-60 audit" (clean_cat + ép kiểu)
def coercion(df, numeric):
    x = df[FEATURES].copy()
    for f in FEATURES:
        if f in numeric:
            x[f] = pd.to_numeric(x[f].astype('string').str.replace(',', '', regex=False),
                                 errors='coerce').astype(float)
        else:
            c = clean_cat(x[f])
            x[f] = c.astype(object).where(c.notna(), np.nan)
    return x


def admin_model(numeric):
    cat = [c for c in FEATURES if c not in numeric]
    trans = []
    if numeric:
        trans.append(('numeric', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value=0))]), numeric))
    if cat:
        trans.append(('categorical', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='MISSING')),
                                               ('encode', onehot())]), cat))
    return Pipeline([('prep', ColumnTransformer(trans, remainder='drop', sparse_threshold=0)),
                     ('xgb', XGBClassifier(**XGB_ADMIN))])


# H180: đúng pipeline ô "VIB H=180 — A / B / B-sym"
def h180_model(numeric):
    cat = [c for c in FEATURES if c not in numeric]
    trans = []
    if numeric:
        trans.append(('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))]), numeric))
    if cat:
        trans.append(('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
                                       ('onehot', onehot())]), cat))
    return Pipeline([('preprocess', ColumnTransformer(trans)), ('xgb', XGBClassifier(**XGB_H180))])


def h180_types(mat, numeric):
    mat = mat.copy()
    for col in FEATURES:
        if col in numeric:
            mat[col] = pd.to_numeric(mat[col], errors='raise')
        else:
            s = mat[col]
            mat[col] = s.astype(str).where(s.notna(), np.nan)
    return mat


# ================================================================ NHÁNH MỚI + BÁO CÁO
def rs_cutoffs(start, pos_cut, T, y, train_idx, seed, cap_inclusive):
    start = pd.to_datetime(pd.Series(start)).reset_index(drop=True)
    pos_cut = pd.to_datetime(pd.Series(pos_cut)).reset_index(drop=True)
    T = pd.to_datetime(pd.Series(T)).reset_index(drop=True)
    y = np.asarray(y)
    pos_tr = np.asarray(train_idx)[y[np.asarray(train_idx)] == 1]
    r_ns = start.to_numpy('datetime64[ns]').astype('int64').astype(float)
    strata = np.digitize(r_ns, np.quantile(r_ns[pos_tr], np.linspace(0, 1, N_STRATA + 1)[1:-1]))
    delay = (pos_cut - start).to_numpy('timedelta64[ns]')
    if pd.isna(delay[pos_tr]).any() or (delay[pos_tr] <= np.timedelta64(0, 'ns')).any():
        fail('độ trễ positive TRAIN không hợp lệ')
    cut = pos_cut.to_numpy('datetime64[ns]').copy()
    capped = np.zeros(len(y), bool)
    rng = np.random.default_rng(seed)
    for b in range(N_STRATA):
        m = (y == 0) & (strata == b)
        if not m.any():
            continue
        pool = delay[pos_tr[strata[pos_tr] == b]]
        if len(pool) == 0:
            pool = delay[pos_tr]
        prop = start.to_numpy('datetime64[ns]')[m] + rng.choice(pool, m.sum(), replace=True)
        Tm = T.to_numpy('datetime64[ns]')[m]
        capped[m] = prop >= Tm if cap_inclusive else prop > Tm
        cut[m] = np.minimum(prop, Tm)
    cut = pd.Series(cut)
    if pd.isna(cut).any() or not (cut > start).all() or not (cut <= T).all():
        fail('cutoff mới nằm ngoài [r, T]')
    return cut, capped


def lag_positive(start, event, k):
    start = pd.to_datetime(pd.Series(start)).reset_index(drop=True)
    t = pd.to_datetime(pd.Series(event)).reset_index(drop=True) - pd.Timedelta(days=k)
    floor = start + pd.Timedelta(days=1)
    return t.where(t > floor, floor)


def balance_row(name, start, cut, y, capped):
    start = pd.to_datetime(pd.Series(start)).reset_index(drop=True)
    d = (pd.to_datetime(cut).reset_index(drop=True) - start).dt.total_seconds().to_numpy() / 86400
    pos = np.asarray(y) == 1
    return dict(branch=name, trunc_rate=float(capped[~pos].mean()),
                median_pos=float(np.median(d[pos])), median_neg=float(np.median(d[~pos])),
                ks=float(ks_2samp(d[pos], d[~pos]).statistic),
                n_pos=int(pos.sum()), n_neg=int((~pos).sum()))


def gate(tag, auc_a, auc_b):
    ref = REFERENCE[tag]
    print(f'{tag}: A={auc_a:.6f} (công bố {ref["A"]:.6f}); B={auc_b:.6f} (công bố {ref["B"]:.6f})')
    if abs(auc_a - ref['A']) > BASELINE_TOL or abs(auc_b - ref['B']) > BASELINE_TOL:
        fail(f'{tag}: A/B không khớp số công bố. Không dùng kết quả mới.')


def environment(t0):
    ram = None
    try:
        with open('/proc/meminfo') as fh:
            ram = round(int(fh.readline().split()[1]) / 1024 ** 2, 1)
    except OSError:
        pass
    cpu = platform.processor()
    try:
        with open('/proc/cpuinfo') as fh:
            cpu = next((l.split(':', 1)[1].strip() for l in fh if l.startswith('model name')), cpu)
    except OSError:
        pass
    pk = {}
    for p in ('numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'lightgbm'):
        try:
            pk[p] = importlib.metadata.version(p)
        except importlib.metadata.PackageNotFoundError:
            pk[p] = 'not installed'
    return dict(os=platform.platform(), cpu=cpu, n_cpu=os.cpu_count(), ram_gb=ram,
                python=platform.python_version(), packages=pk,
                elapsed_hours=round((time.monotonic() - t0) / 3600, 3))


def finalize(tag, y_te, scores, balance, info, t0, extra_auc=(), extra_bss=()):
    out = AUDIT_ROOT / f'{tag}_{STAMP}'
    out.mkdir(parents=True, exist_ok=False)
    y_te = np.asarray(y_te)
    base = y_te.mean() * (1 - y_te.mean())        # BSS so với tỷ lệ positive của HOLDOUT
    res = pd.DataFrame([dict(branch=k, auc=float(roc_auc_score(y_te, p)),
                             brier=float(brier_score_loss(y_te, p)),
                             bss=float(1 - brier_score_loss(y_te, p) / base),
                             n_test=len(y_te), events_test=int(y_te.sum())) for k, p in scores.items()])
    auc_pairs = [('B', 'A'), ('Bsym_rs', 'A'), ('B', 'Bsym_rs'), ('B->A', 'A'), ('Bsym_rs->A', 'A')] + \
                [(f'lag{k}', 'A') for k in LAGS] + list(extra_auc)
    bss_pairs = [('B->A', 'A'), ('Bsym_rs->A', 'A')] + list(extra_bss)
    bss_keys = sorted({x for p in bss_pairs for x in p})
    draws = {('AUC',) + p: [] for p in auc_pairs}
    draws.update({('BSS',) + p: [] for p in bss_pairs})
    rng = np.random.default_rng(SEED_BOOT)
    done = 0
    while done < N_BOOT:
        i = rng.integers(0, len(y_te), len(y_te))
        yy = y_te[i]
        if yy.min() == yy.max():
            continue
        auc = {k: roc_auc_score(yy, scores[k][i]) for k in scores}
        bb = yy.mean() * (1 - yy.mean())
        bss = {k: 1 - np.mean((scores[k][i] - yy) ** 2) / bb for k in bss_keys}
        for a, b in auc_pairs:
            draws[('AUC', a, b)].append(auc[a] - auc[b])
        for a, b in bss_pairs:
            draws[('BSS', a, b)].append(bss[a] - bss[b])
        done += 1
        if done % 250 == 0:
            print(f'  bootstrap {done}/{N_BOOT}')
    rows = []
    for (metric, a, b), v in draws.items():
        if metric == 'AUC':
            est = roc_auc_score(y_te, scores[a]) - roc_auc_score(y_te, scores[b])
        else:
            est = (1 - np.mean((scores[a] - y_te) ** 2) / base) - (1 - np.mean((scores[b] - y_te) ** 2) / base)
        lo, hi = np.percentile(v, [2.5, 97.5])
        rows.append(dict(metric=metric, contrast=f'{a} - {b}', estimate=float(est), lo=float(lo),
                         hi=float(hi), n_bootstrap=N_BOOT, seed=SEED_BOOT))
    boot, bal = pd.DataFrame(rows), pd.DataFrame(balance)
    res.to_csv(out / f'{tag}_results.csv', index=False)
    boot.to_csv(out / f'{tag}_boot.csv', index=False)
    bal.to_csv(out / f'{tag}_balance.csv', index=False)
    info.update(environment=environment(t0), bss_reference='holdout prevalence', seed_rs=SEED_RS,
                n_strata=N_STRATA, lags_days=list(LAGS), n_bootstrap=N_BOOT, bootstrap_seed=SEED_BOOT,
                note='Bsym_rs and lag negatives are NEW draws; published B-sym is not re-estimated.')
    (out / 'manifest.json').write_text(json.dumps(info, ensure_ascii=False, indent=2, default=str),
                                       encoding='utf-8')
    print('\nRESULTS\n', res.round(6).to_string(index=False))
    print('\nBOOTSTRAP\n', boot.round(6).to_string(index=False))
    print('\nBALANCE\n', bal.round(4).to_string(index=False))
    print('\nĐã lưu tại:', out)
    return res, boot, bal


# ================================================================ CHUẨN BỊ CHUNG
T0 = time.monotonic()
A_ALL, B_ALL, IDS, Y60, TR60, TE60 = load_inputs()
NUM_ADMIN = A_ALL.iloc[TR60][FEATURES].select_dtypes(include=np.number).columns.tolist()
NUM_H180 = A_ALL[FEATURES].select_dtypes(include=np.number).columns.tolist()
A_ALL, B_ALL = frame_aligned(A_ALL, IDS), frame_aligned(B_ALL, IDS)
MASTER = make_master(A_ALL, IDS, Y60)


# ================================================================ H180
def run_h180():
    t0 = time.monotonic()
    print('\n' + '=' * 70 + '\nVIB H180\n' + '=' * 70)
    m = MASTER.copy()
    m['LANDMARK_DATE'] = m[START] + pd.Timedelta(days=DAY60)
    m['HORIZON_END_DATE'] = m['LANDMARK_DATE'] + pd.Timedelta(days=HORIZON)
    m['Y_H'] = (m[EVENT].notna() & (m[EVENT] > m['LANDMARK_DATE'])
                & (m[EVENT] <= m['HORIZON_END_DATE'])).astype(np.int8)
    c = m.loc[m['HORIZON_END_DATE'] <= OBSERVATION_END].sort_values(ID, kind='stable').reset_index(drop=True)
    y = c['Y_H'].to_numpy(np.int8)
    if len(c) != 31_149 or int(y.sum()) != 1_454:
        fail(f'cohort H180 = {len(c):,}/{int(y.sum()):,}, cần 31.149/1.454')
    tr, te = train_test_split(np.arange(len(c)), test_size=0.20, stratify=y, random_state=42)
    if int(y[tr].sum()) != 1_163 or int(y[te].sum()) != 291:
        fail('split H180 khác bản gốc')
    print(f'H180: N={len(c):,}; sự kiện={int(y.sum()):,}; train/test sự kiện={int(y[tr].sum())}/{int(y[te].sum())}')
    c[ID] = c[ID].astype(str)
    mat_a = A_ALL.set_index(ID).loc[c[ID].tolist(), FEATURES].reset_index()
    mat_a[ID] = mat_a[ID].astype(str)
    mat_a = h180_types(mat_a, NUM_H180)

    def matrix_at(cut, label):
        mm = c[[ID, START]].copy()
        mm[CUTOFF] = pd.to_datetime(cut).to_numpy()
        built = build_features(mm, A_ALL, label)
        built[ID] = built[ID].astype(str)
        mat = mat_a[[ID] + STATIC].merge(built[[ID] + DYNAMIC], on=ID, how='left', validate='one_to_one')
        mat = h180_types(mat[[ID] + FEATURES], NUM_H180)
        # so sánh theo giá trị (không theo kiểu dữ liệu của cột ID)
        if mat[ID].astype(str).tolist() != mat_a[ID].astype(str).tolist():
            fail(f'{label}: thứ tự ID khác A')
        if mat[DYNAMIC].isna().any().any():
            fail(f'{label}: có khách không ghép được đặc trưng động')
        return mat

    def fit_pred(mat, extra=None):
        model = h180_model(NUM_H180).fit(mat.loc[tr, FEATURES], y[tr])
        own = np.asarray(model.predict_proba(mat.loc[te, FEATURES])[:, 1], float)
        other = None if extra is None else np.asarray(model.predict_proba(extra.loc[te, FEATURES])[:, 1], float)
        return own, other

    start, event = c[START], c[EVENT]
    T = c['HORIZON_END_DATE']
    cut_b = event.where(c['Y_H'].eq(1), T)
    mat_b = matrix_at(cut_b, 'B_H180')
    scores = {}
    scores['A'], _ = fit_pred(mat_a)
    scores['B'], scores['B->A'] = fit_pred(mat_b, extra=mat_a)
    gate('H180', roc_auc_score(y[te], scores['A']), roc_auc_score(y[te], scores['B']))
    del mat_b
    balance = []
    specs = [('Bsym_rs', event, SEED_RS)] + [(f'lag{k}', lag_positive(start, event, k), SEED_RS + k) for k in LAGS]
    for name, pos_cut, seed in specs:
        print(f'\n=== H180 {name} ===')
        cut, capped = rs_cutoffs(start, pos_cut, T, y, tr, seed, cap_inclusive=False)
        balance.append(balance_row(name, start, cut, y, capped))
        mat = matrix_at(cut, name)
        if name == 'Bsym_rs':
            scores[name], scores['Bsym_rs->A'] = fit_pred(mat, extra=mat_a)
        else:
            scores[name], _ = fit_pred(mat)
        print(f'{name}: AUC={roc_auc_score(y[te], scores[name]):.6f}')
        del mat
        gc.collect()
    info = dict(audit='VIB H180', cohort_n=len(c), events=int(y.sum()), train=len(tr), test=len(te),
                test_events=int(y[te].sum()), T='r + 240 days', capped_definition='proposal > T')
    return finalize('VIB_H180', y[te], scores, balance, info, t0)


# ================================================================ ADMIN
def run_admin():
    t0 = time.monotonic()
    print('\n' + '=' * 70 + '\nVIB MỐC HÀNH CHÍNH\n' + '=' * 70)
    y, tr, te = Y60, TR60, TE60

    def fit_pred(mat, extra=None):
        model = admin_model(NUM_ADMIN).fit(coercion(mat, NUM_ADMIN).iloc[tr], y[tr])
        own = np.asarray(model.predict_proba(coercion(mat, NUM_ADMIN).iloc[te])[:, 1], float)
        other = None if extra is None else np.asarray(
            model.predict_proba(coercion(extra, NUM_ADMIN).iloc[te])[:, 1], float)
        return own, other

    scores = {}
    scores['A'], _ = fit_pred(A_ALL)
    scores['B'], scores['B->A'] = fit_pred(B_ALL, extra=A_ALL)
    gate('ADMIN', roc_auc_score(y[te], scores['A']), roc_auc_score(y[te], scores['B']))
    # Kiểm tra hàm dựng: A PHẢI khớp tuyệt đối. B đã lưu (VIB_FLDC_60D.csv) được tạo
    # bằng quy trình khác, nên dựng lại B theo đúng quy tắc của A ("B_rebuilt") và
    # so các nhánh mới với B_rebuilt trong cùng một cách dựng.
    print('\nKiểm tra hàm dựng đặc trưng với A đã lưu...')
    mm = MASTER[[ID, START]].copy()
    mm[CUTOFF] = MASTER['A_CUTOFF'].values
    bad_a = verify_rebuild('A', A_ALL, frame_aligned(build_features(mm, A_ALL, 'CHECK_A'), IDS), NUM_ADMIN)
    if bad_a:
        fail(f'hàm dựng không tái tạo đúng A: {bad_a}')
    mm = MASTER[[ID, START]].copy()
    mm[CUTOFF] = MASTER['B_CUTOFF'].values
    b_rebuilt = frame_aligned(build_features(mm, A_ALL, 'B_rebuilt'), IDS)
    bad_b = verify_rebuild('B (so với file đã lưu)', B_ALL, b_rebuilt, NUM_ADMIN)
    scores['B_rebuilt'], scores['B_rebuilt->A'] = fit_pred(b_rebuilt, extra=A_ALL)
    print(f"B_rebuilt: AUC={roc_auc_score(y[te], scores['B_rebuilt']):.6f}")
    del b_rebuilt
    gc.collect()
    start, event = MASTER[START], MASTER[EVENT]
    T = pd.Series(DATE_END_EXCLUSIVE, index=MASTER.index)
    balance = []
    specs = [('Bsym_rs', event, SEED_RS)] + [(f'lag{k}', lag_positive(start, event, k), SEED_RS + k) for k in LAGS]
    for name, pos_cut, seed in specs:
        print(f'\n=== ADMIN {name} ===')
        cut, capped = rs_cutoffs(start, pos_cut, T, y, tr, seed, cap_inclusive=True)
        balance.append(balance_row(name, start, cut, y, capped))
        mm = MASTER[[ID, START]].copy()
        mm[CUTOFF] = cut.values
        mat = frame_aligned(build_features(mm, A_ALL, name), IDS)
        if name == 'Bsym_rs':
            scores[name], scores['Bsym_rs->A'] = fit_pred(mat, extra=A_ALL)
        else:
            scores[name], _ = fit_pred(mat)
        print(f'{name}: AUC={roc_auc_score(y[te], scores[name]):.6f}')
        del mat
        gc.collect()
    info = dict(audit='VIB day-60 administrative endpoint', cohort_n=len(IDS), events=int(y.sum()),
                train=len(tr), test=len(te), test_events=int(y[te].sum()),
                T='2020-01-01 (exclusive)', capped_definition='proposal >= T',
                saved_B_vs_rebuilt_mismatches=bad_b,
                B_note='B = saved canonical matrix; B_rebuilt = B cutoffs with the A feature builder')
    return finalize('VIB_ADMIN', y[te], scores, balance, info, t0,
                    extra_auc=[('B_rebuilt', 'A'), ('B_rebuilt', 'Bsym_rs'), ('B_rebuilt', 'B'),
                               ('B_rebuilt->A', 'A')],
                    extra_bss=[('B_rebuilt->A', 'A')])


# ================================================================ CHẠY
AUDIT_RESULTS = {}
if 'H180' in RUN:
    AUDIT_RESULTS['H180'] = run_h180()
if 'ADMIN' in RUN:
    AUDIT_RESULTS['ADMIN'] = run_admin()

Mounted at /content/drive
E2: N=94,453; sự kiện=4,374; train/test=75,562/18,891
Đọc ngày mở quan hệ từ dữ liệu khách hàng...
Đọc ngày mở thẻ đầu tiên...


In [ ]:
# =====================================================================
# HƯỚNG 2 — BƯỚC 1: TẠO FILE B ĐÚNG (VIB_FLDC_60D_B_REBUILT.csv)
# Chạy SAU ô "ELAPS — RUN_AUDIT CHO VIB" ngay phía trên (ô đó đã đặt RUN = ()
# nên chỉ nạp hàm + dữ liệu). File B mới = bản sao file B cũ, chỉ thay 13 biến
# động bằng giá trị dựng theo ĐÚNG công thức của A.
# BƯỚC 2: chạy lại ô "VIB60ABC — A / B / B-sym + R50 + H=180" (dòng 39 đã trỏ
# sang file mới) và ô run_all() ngay dưới nó.
# =====================================================================
OUT_B = ROOT / 'VIB_FLDC_60D_B_REBUILT.csv'
print('\nKiểm tra hàm dựng với A đã lưu (bắt buộc khớp 100%)...')
_mm = MASTER[[ID, START]].copy()
_mm[CUTOFF] = MASTER['A_CUTOFF'].values
if verify_rebuild('A', A_ALL, frame_aligned(build_features(_mm, A_ALL, 'CHECK_A'), IDS), NUM_ADMIN):
    fail('hàm dựng không tái tạo đúng A; không tạo file B mới.')
print('\nDựng B theo đúng quy tắc của A...')
_mm = MASTER[[ID, START]].copy()
_mm[CUTOFF] = MASTER['B_CUTOFF'].values
_b_new = frame_aligned(build_features(_mm, A_ALL, 'B_rebuilt'), IDS).set_index(ID)
_raw = pd.read_csv(B_FILE, dtype=str, keep_default_na=False, low_memory=False)
_cols = {str(c).strip().upper(): c for c in _raw.columns}
_key = standardize_id(_raw[_cols[ID]]).tolist()
_vals = _b_new.loc[_key, DYNAMIC]
for _f in DYNAMIC:
    _raw[_cols[_f]] = _vals[_f].to_numpy()
_raw.to_csv(OUT_B, index=False, encoding='utf-8-sig')
print('Đã lưu:', OUT_B)

# kiểm tra: đọc lại file mới, fit như Table 5 -> phải ra AUC 0.922914
_chk = frame_aligned(read_csv_exact(OUT_B), IDS)
_m = admin_model(NUM_ADMIN).fit(coercion(_chk, NUM_ADMIN).iloc[TR60], Y60[TR60])
_auc = roc_auc_score(Y60[TE60], _m.predict_proba(coercion(_chk, NUM_ADMIN).iloc[TE60])[:, 1])
print(f'AUC B mới = {_auc:.6f} (lần chạy audit: 0.922914)')
if abs(_auc - 0.922914) > BASELINE_TOL:
    print('CẢNH BÁO: AUC khác lần chạy audit; dừng lại, chưa chạy Bước 2.')
else:
    print('KHỚP. Chạy tiếp Bước 2.')


In [ ]:
# HƯỚNG 2 — BƯỚC 1b: SỬA FILE B MỚI (chạy ngay sau ô BƯỚC 1 ở trên)
# Chẩn đoán: file B cũ còn lệch A cả ở biến tĩnh (thiếu giá trị mã '0' thay vì 'UNKNOWN'
# ở CLIENT_SEX / EB_REGISTER_CHANNEL / SMS / VERIFY_METHOD; AGE tính ở mốc khác).
# Sửa: thay TOÀN BỘ biến đầu vào (tĩnh + động) bằng B dựng theo đúng công thức A.
# Kết quả đã chạy 21/11/2026: AUC B mới = 0.922914 -> KHỚP.
_feat = list(coercion(frame_aligned(_b_new.reset_index(), IDS), NUM_ADMIN).columns)
_raw = pd.read_csv(B_FILE, dtype=str, keep_default_na=False, low_memory=False)
_cols = {str(c).strip().upper(): c for c in _raw.columns}
_miss = [f for f in _feat if f not in _cols]
if _miss:
    fail(f'file B cũ thiếu cột {_miss}')
_key = standardize_id(_raw[_cols[ID]]).tolist()
_vals = _b_new.loc[_key, _feat]
for _f in _feat:
    _raw[_cols[_f]] = _vals[_f].to_numpy()
_raw.to_csv(OUT_B, index=False, encoding='utf-8-sig', float_format='%.17g')
print('Đã ghi lại:', OUT_B, '| số biến thay:', len(_feat))
_chk = frame_aligned(read_csv_exact(OUT_B), IDS)
_X1 = coercion(_chk, NUM_ADMIN).reset_index(drop=True)
_m = admin_model(NUM_ADMIN).fit(_X1.iloc[TR60], Y60[TR60])
_auc = roc_auc_score(Y60[TE60], _m.predict_proba(_X1.iloc[TE60])[:, 1])
print(f'AUC B mới = {_auc:.6f} (lần chạy audit: 0.922914)')
print('KHỚP. Chạy tiếp Bước 2.' if abs(_auc - 0.922914) <= BASELINE_TOL else 'CẢNH BÁO: vẫn lệch; dừng lại.')


### Step 3 — Four-learner R50 (Table 7, S18)
Line 39 reads `VIB_FLDC_60D_B_REBUILT.csv` produced in Step 2.

VIB60ABC — A / B / B-sym + R50 + H=180
Thiết kế của notebook này
- Primary A/B được đọc trực tiếp từ bộ VIB60ABC / exact E2 đã xác nhận; không dựng lại A/B từ raw.
- C không được fit. VIB_E1_BRANCH_C.csv chỉ cung cấp timeline đã xác nhận (RELATIONSHIP_DATE, FIRST_CARD_DATE, TENURE_AT_CUTOFF) để dựng B-sym.
- Primary canonical dùng đúng exact E2 train order từ _E2_ORDER và exact E2 test set.
- R=50 chạy LR / RF / LightGBM / XGBoost trên A / B-sym / B.
- B-sym: positive giữ event cutoff thật; negative lấy pseudo-duration từ phân phối e-r của positive training customers trong chính split, rồi chặn tại T_end.
- H=180 giữ đúng cohort/target V3; A dùng lại đủ 18 feature A60 đã lưu; chỉ B và B-sym được tái dựng theo cutoff mới.
- Canonical Primary và H180 dùng 1,000 paired bootstrap.

In [ ]:

import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "openpyxl": "openpyxl",
}

missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

print("Libraries ready.")


Libraries ready.


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import gc
import inspect
import json
import math
import warnings

import numpy as np
import pandas as pd

from scipy.stats import kendalltau

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1. CONFIG
# ============================================================

ROOT = Path("/content/drive/MyDrive/Fintect")

ID = "CUSTOMER_NUMBER"
TARGET = "TARGET"

PRIMARY_A_FILE = ROOT / "final_landmark_60d" / "final_dataset_landmark_60d_no_auto_job.csv"
PRIMARY_B_FILE = ROOT / "VIB_FLDC_60D_B_REBUILT.csv"  # Hướng 2: B dựng cùng công thức với A
PRIMARY_C_FILE = ROOT / "VIB_E1_BRANCH_C.csv"

E2_SPLIT_FILE = ROOT / "FINAL_ABC_MATCHED_E2" / "exact_E2_shared_train_test_split.csv"

# H180: chỉ đọc lại các file đã xử lý từ run V3 đã xác nhận.
H180_PARENT = ROOT / "VIB_FIXED_HORIZON_H180_A60_B240_V3"
H180_PREFERRED_RUN = H180_PARENT / "run_20261006T034357836181Z"

# Raw longitudinal sources are used ONLY to construct B-sym.
# A/B/C/H180 are NOT rebuilt.
RAW_FILES = {
    "transaction": ROOT / "2.Data_MyVIB_Transaction.csv",
    "activity": ROOT / "3.Data_MyVIB_Activity.csv",
    "deposit": ROOT / "4.Data_Deposit.csv",
    "lending": ROOT / "5.Data_Lending.xlsx",
}

PRIMARY_STUDY_END = pd.Timestamp("2019-12-31")

LANDMARK_DAYS = 60
HORIZON_DAYS = 180

TEST_SIZE = 0.20
N_BOOTSTRAP = 1000
R50_SEEDS = list(range(42, 92))

CSV_CHUNKSIZE = 250_000

# Set False if you only want canonical 4-learner results first.
RUN_R50 = True

# B-sym cache: if Colab disconnects, rerun and completed seeds are reused.
CACHE_ROOT = ROOT / "VIB_BSYM_CACHE_V1"
OUTPUT_PARENT = ROOT / "VIB_REUSE_PROCESSED_ABCH180_BSYM_4LEARNERS"

STATIC_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "AGE",
]

ACTIVITY_FEATURES = [
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
]

TRANSACTION_FEATURES = [
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
]

DEPOSIT_FEATURES = [
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
]

LENDING_FEATURES = [
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT",
]

FEATURES = (
    STATIC_FEATURES
    + ACTIVITY_FEATURES
    + TRANSACTION_FEATURES
    + DEPOSIT_FEATURES
    + LENDING_FEATURES
)

assert len(FEATURES) == 18

# The date formats below reproduce the already-audited H180 V3 rules.
DATE_DAYFIRST = None
SOURCE_DATE_DAYFIRST = {
    "activity": False,
    "deposit": False,
}

# ============================================================
# 2. HELPERS
# ============================================================

def mount_drive():
    if str(ROOT).startswith("/content/"):
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive", force_remount=False)

def clean_id(series):
    s = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    s = s.mask(
        s.str.upper().isin(["", "NAN", "<NA>", "NONE", "NULL"]),
        pd.NA,
    )
    numeric = s.str.fullmatch(r"[0-9]+", na=False)
    s.loc[numeric] = s.loc[numeric].str.lstrip("0").replace("", "0")
    return s

def normalize_columns(df):
    x = df.copy()
    x.columns = x.columns.astype(str).str.strip().str.upper()
    if x.columns.duplicated().any():
        raise ValueError("Có cột trùng sau chuẩn hóa tên.")
    return x

def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: thiếu cột {missing}. Có: {list(df.columns)}")

def clean_category(series):
    x = series.astype("string").str.strip().str.upper()
    return x.mask(x.isin(["", "NAN", "<NA>", "NONE", "NULL", "N/A", "NA"]), pd.NA)

def parse_numbers(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )

def parse_source_date(df, column, source):
    raw = df[column]
    text = clean_category(raw)

    parts = text.str.extract(r"^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)")
    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    ambiguous = (
        first.between(1, 12)
        & second.between(1, 12)
        & first.ne(second)
    ).fillna(False)

    dayfirst = SOURCE_DATE_DAYFIRST.get(source, DATE_DAYFIRST)

    if dayfirst is None and ambiguous.any():
        sample = df.loc[ambiguous, [ID, column]].head(20)
        raise ValueError(
            f"{source}/{column}: có ngày mơ hồ dd/mm hay mm/dd.\n"
            f"{sample.to_string(index=False)}"
        )

    parsed = pd.to_datetime(
        raw,
        errors="coerce",
        format="mixed",
        dayfirst=bool(dayfirst),
    )

    invalid = text.notna() & parsed.isna()
    if invalid.any():
        sample = df.loc[invalid, [ID, column]].head(20)
        raise ValueError(
            f"{source}/{column}: có ngày không đọc được.\n"
            f"{sample.to_string(index=False)}"
        )

    return parsed

def read_csv_preserve_id(path):
    header = pd.read_csv(path, nrows=0)
    names = {str(c).strip().upper(): c for c in header.columns}
    if ID not in names:
        raise ValueError(f"{path.name}: thiếu {ID}.")
    df = pd.read_csv(path, dtype={names[ID]: str}, low_memory=False)
    df = normalize_columns(df)
    df[ID] = clean_id(df[ID])
    return df.loc[df[ID].notna()].copy()

def read_processed(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
        df = normalize_columns(df)
        require_columns(df, [ID], path.name)
        df[ID] = clean_id(df[ID])
        return df.loc[df[ID].notna()].copy()
    return read_csv_preserve_id(path)

def iter_source(path, columns):
    path = Path(path)

    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
        df = normalize_columns(df)
        require_columns(df, columns, path.name)
        df[ID] = clean_id(df[ID])
        yield df.loc[df[ID].notna(), columns].copy()
        return

    header = pd.read_csv(path, nrows=0)
    raw_names = {str(c).strip().upper(): c for c in header.columns}
    missing = [c for c in columns if c not in raw_names]
    if missing:
        raise ValueError(f"{path.name}: thiếu {missing}.")

    usecols = [raw_names[c] for c in columns]

    for chunk in pd.read_csv(
        path,
        usecols=usecols,
        dtype={raw_names[ID]: str},
        chunksize=CSV_CHUNKSIZE,
        low_memory=False,
    ):
        chunk = normalize_columns(chunk)
        chunk[ID] = clean_id(chunk[ID])
        chunk = chunk.loc[chunk[ID].notna(), columns].copy()
        yield chunk

def find_h180_run():
    candidates = []

    if H180_PREFERRED_RUN.is_dir():
        candidates.append(H180_PREFERRED_RUN)

    if H180_PARENT.is_dir():
        candidates += sorted(
            [p for p in H180_PARENT.glob("run_*") if p.is_dir()],
            reverse=True,
        )

    for run in candidates:
        a = run / "MATRICES" / "A_fixed_horizon_matrix.csv"
        b = run / "MATRICES" / "B_fixed_horizon_matrix.csv"
        cohort = run / "cohort_fixed_horizon_master.csv"
        split = run / "shared_H180_train_test_split.csv"

        if all(p.is_file() for p in [a, b, cohort, split]):
            return run

    raise FileNotFoundError(
        "Không tìm thấy run H180 đã xử lý có đủ:\n"
        "MATRICES/A_fixed_horizon_matrix.csv\n"
        "MATRICES/B_fixed_horizon_matrix.csv\n"
        "cohort_fixed_horizon_master.csv\n"
        "shared_H180_train_test_split.csv"
    )

# ============================================================
# 3. LOAD PROCESSED PRIMARY A/B/C
# ============================================================

def load_primary_processed():
    a = read_processed(PRIMARY_A_FILE)
    b = read_processed(PRIMARY_B_FILE)
    c = read_processed(PRIMARY_C_FILE)
    split = read_processed(E2_SPLIT_FILE)

    require_columns(a, [ID, "TARGET_60D"] + FEATURES, "PRIMARY A")
    require_columns(b, [ID, "COUNT_CREDITCARD"] + FEATURES, "PRIMARY B")
    require_columns(
        c,
        [
            ID,
            "COUNT_CREDITCARD",
            "RELATIONSHIP_DATE",
            "FIRST_CARD_DATE",
        ] + FEATURES,
        "PRIMARY C",
    )
    require_columns(split, [ID, "TARGET", "SPLIT"], "E2 split")

    a = a[[ID, "TARGET_60D"] + FEATURES].copy()
    b = b[[ID, "COUNT_CREDITCARD"] + FEATURES].copy()

    a = a.rename(columns={"TARGET_60D": TARGET})
    b = b.rename(columns={"COUNT_CREDITCARD": TARGET})

    a[TARGET] = pd.to_numeric(a[TARGET], errors="raise").astype(np.int8)
    b[TARGET] = pd.to_numeric(b[TARGET], errors="raise").astype(np.int8)

    # Align B to locked A order.
    b = (
        a[[ID]]
        .merge(b, on=ID, how="left", validate="one_to_one", sort=False)
    )

    if b[FEATURES + [TARGET]].isna().all(axis=1).any():
        raise ValueError("PRIMARY B thiếu customer sau khi align với A.")

    if not np.array_equal(
        a[ID].astype(str).to_numpy(),
        b[ID].astype(str).to_numpy(),
    ):
        raise AssertionError("PRIMARY A/B ID order mismatch.")

    if not np.array_equal(
        a[TARGET].to_numpy(),
        b[TARGET].to_numpy(),
    ):
        raise AssertionError("PRIMARY A/B target mismatch.")

    if len(a) != 94_453 or int(a[TARGET].sum()) != 4_374:
        raise ValueError(
            f"PRIMARY cohort không đúng: N={len(a):,}, positives={int(a[TARGET].sum()):,}."
        )

    # Timeline only from C; C is NOT fitted.
    c = c[
        [ID, "COUNT_CREDITCARD", "RELATIONSHIP_DATE", "FIRST_CARD_DATE"]
    ].copy()

    c = c.rename(columns={"COUNT_CREDITCARD": TARGET})
    c[TARGET] = pd.to_numeric(c[TARGET], errors="raise").astype(np.int8)
    c["RELATIONSHIP_DATE"] = pd.to_datetime(
        c["RELATIONSHIP_DATE"], errors="raise"
    )
    c["FIRST_CARD_DATE"] = pd.to_datetime(
        c["FIRST_CARD_DATE"], errors="coerce"
    )

    timeline = (
        a[[ID, TARGET]]
        .merge(
            c,
            on=ID,
            how="left",
            suffixes=("_A", "_C"),
            validate="one_to_one",
            sort=False,
        )
    )

    if timeline["RELATIONSHIP_DATE"].isna().any():
        raise ValueError("PRIMARY C timeline thiếu RELATIONSHIP_DATE.")

    if not np.array_equal(
        timeline[f"{TARGET}_A"].to_numpy(),
        timeline[f"{TARGET}_C"].to_numpy(),
    ):
        raise ValueError("PRIMARY C target không khớp A.")

    timeline = timeline.rename(columns={f"{TARGET}_A": TARGET})
    timeline = timeline.drop(columns=[f"{TARGET}_C"])
    timeline["LANDMARK_DATE"] = (
        timeline["RELATIONSHIP_DATE"] + pd.Timedelta(days=LANDMARK_DAYS)
    )
    timeline["T_END"] = PRIMARY_STUDY_END

    pos = timeline[TARGET].eq(1)
    neg = ~pos

    if timeline.loc[pos, "FIRST_CARD_DATE"].isna().any():
        raise ValueError("PRIMARY positive thiếu FIRST_CARD_DATE.")

    if not (
        timeline.loc[pos, "FIRST_CARD_DATE"]
        > timeline.loc[pos, "LANDMARK_DATE"]
    ).all():
        raise ValueError("PRIMARY positive event không strictly after day 60.")

    # Locked VIB60ABC allows T_end == landmark for boundary negatives.
    if (
        timeline.loc[neg, "T_END"]
        < timeline.loc[neg, "LANDMARK_DATE"]
    ).any():
        raise ValueError("PRIMARY negative T_end before landmark.")

    split["TARGET"] = pd.to_numeric(split["TARGET"], errors="raise").astype(np.int8)
    split["SPLIT"] = split["SPLIT"].astype("string").str.strip().str.upper()

    split = (
        a[[ID, TARGET]]
        .merge(
            split[[ID, "TARGET", "SPLIT"]],
            on=ID,
            how="left",
            suffixes=("_A", "_SPLIT"),
            validate="one_to_one",
            sort=False,
        )
    )

    if split["SPLIT"].isna().any():
        raise ValueError("E2 split thiếu customer.")

    if not np.array_equal(
        split[f"{TARGET}_A"].to_numpy(),
        split[f"{TARGET}_SPLIT"].to_numpy(),
    ):
        raise ValueError("E2 split target mismatch.")

    train_idx = np.where(split["SPLIT"].eq("TRAIN").to_numpy())[0]
    test_idx = np.where(split["SPLIT"].eq("TEST").to_numpy())[0]

    if len(train_idx) != 75_562 or len(test_idx) != 18_891:
        raise ValueError(
            f"E2 split không đúng: train={len(train_idx):,}, test={len(test_idx):,}."
        )

    print(
        f"PRIMARY processed files OK: N={len(a):,}; "
        f"positives={int(a[TARGET].sum()):,}; "
        f"train/test={len(train_idx):,}/{len(test_idx):,}"
    )

    return a, b, timeline, train_idx, test_idx

# ============================================================
# 4. LOAD PROCESSED H180
# ============================================================

def load_h180_processed():
    run = find_h180_run()

    a_file = run / "MATRICES" / "A_fixed_horizon_matrix.csv"
    b_file = run / "MATRICES" / "B_fixed_horizon_matrix.csv"
    cohort_file = run / "cohort_fixed_horizon_master.csv"
    split_file = run / "shared_H180_train_test_split.csv"

    a = read_processed(a_file)
    b = read_processed(b_file)
    cohort = read_processed(cohort_file)
    split = read_processed(split_file)

    require_columns(a, [ID, "TARGET_H"] + FEATURES, "H180 A")
    require_columns(b, [ID, "TARGET_H"] + FEATURES, "H180 B")
    require_columns(
        cohort,
        [
            ID,
            "TARGET_H",
            "RELATIONSHIP_START_DATE",
            "FIRST_CARD_MONTH",
            "LANDMARK_DATE",
            "HORIZON_END_DATE",
        ],
        "H180 cohort",
    )
    require_columns(split, [ID, "TARGET_H", "SPLIT"], "H180 split")

    a = a[[ID, "TARGET_H"] + FEATURES].rename(
        columns={"TARGET_H": TARGET}
    )
    b = b[[ID, "TARGET_H"] + FEATURES].rename(
        columns={"TARGET_H": TARGET}
    )

    a[TARGET] = pd.to_numeric(a[TARGET], errors="raise").astype(np.int8)
    b[TARGET] = pd.to_numeric(b[TARGET], errors="raise").astype(np.int8)

    b = (
        a[[ID]]
        .merge(b, on=ID, how="left", validate="one_to_one", sort=False)
    )

    if not np.array_equal(a[TARGET].to_numpy(), b[TARGET].to_numpy()):
        raise ValueError("H180 A/B target mismatch.")

    if len(a) != 31_149 or int(a[TARGET].sum()) != 1_454:
        raise ValueError(
            f"H180 cohort không đúng: N={len(a):,}, events={int(a[TARGET].sum()):,}."
        )

    cohort = cohort[
        [
            ID,
            "TARGET_H",
            "RELATIONSHIP_START_DATE",
            "FIRST_CARD_MONTH",
            "LANDMARK_DATE",
            "HORIZON_END_DATE",
        ]
    ].copy()

    for col in [
        "RELATIONSHIP_START_DATE",
        "FIRST_CARD_MONTH",
        "LANDMARK_DATE",
        "HORIZON_END_DATE",
    ]:
        cohort[col] = pd.to_datetime(cohort[col], errors="coerce")

    cohort["TARGET_H"] = pd.to_numeric(
        cohort["TARGET_H"], errors="raise"
    ).astype(np.int8)

    timeline = (
        a[[ID, TARGET]]
        .merge(
            cohort,
            on=ID,
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    if timeline["RELATIONSHIP_START_DATE"].isna().any():
        raise ValueError("H180 timeline thiếu relationship start.")

    if not np.array_equal(
        timeline[TARGET].to_numpy(),
        timeline["TARGET_H"].to_numpy(),
    ):
        raise ValueError("H180 cohort target mismatch.")

    timeline = timeline.drop(columns=["TARGET_H"]).rename(
        columns={
            "RELATIONSHIP_START_DATE": "RELATIONSHIP_DATE",
            "FIRST_CARD_MONTH": "FIRST_CARD_DATE",
            "HORIZON_END_DATE": "T_END",
        }
    )

    split["TARGET_H"] = pd.to_numeric(
        split["TARGET_H"], errors="raise"
    ).astype(np.int8)
    split["SPLIT"] = split["SPLIT"].astype("string").str.strip().str.upper()

    split = (
        a[[ID, TARGET]]
        .merge(
            split[[ID, "TARGET_H", "SPLIT"]],
            on=ID,
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    if split["SPLIT"].isna().any():
        raise ValueError("H180 split thiếu customer.")

    if not np.array_equal(
        split[TARGET].to_numpy(),
        split["TARGET_H"].to_numpy(),
    ):
        raise ValueError("H180 split target mismatch.")

    train_idx = np.where(split["SPLIT"].eq("TRAIN").to_numpy())[0]
    test_idx = np.where(split["SPLIT"].eq("TEST").to_numpy())[0]

    if len(train_idx) != 24_919 or len(test_idx) != 6_230:
        raise ValueError(
            f"H180 split không đúng: train={len(train_idx):,}, test={len(test_idx):,}."
        )

    print(
        f"H180 processed files OK: N={len(a):,}; "
        f"events={int(a[TARGET].sum()):,}; "
        f"train/test={len(train_idx):,}/{len(test_idx):,}"
    )
    print("H180 run:", run)

    return a, b, timeline, train_idx, test_idx, run

# ============================================================
# 5. REBUILD ONLY B-SYM DYNAMIC FEATURES
# ============================================================

def rows_in_window(raw, master, date_col, source):
    raw = raw.loc[raw[ID].isin(master[ID])].copy()
    raw[date_col] = parse_source_date(raw, date_col, source)

    joined = raw.merge(
        master[[ID, "RELATIONSHIP_DATE", "BSYM_CUTOFF"]],
        on=ID,
        how="inner",
        validate="many_to_one",
    )

    valid = (
        joined[date_col].notna()
        & joined[date_col].ge(joined["RELATIONSHIP_DATE"])
        & joined[date_col].lt(joined["BSYM_CUTOFF"])
    )

    return joined.loc[valid, raw.columns].copy()

def reduce_mode(parts, name):
    if not parts:
        return pd.Series(dtype=object, name=name)

    counts = (
        pd.concat(parts, ignore_index=True)
        .groupby([ID, "CATEGORY"], as_index=False)["N"]
        .sum()
    )

    winner = (
        counts
        .sort_values(
            [ID, "N", "CATEGORY"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates(ID)
    )

    return winner.set_index(ID)["CATEGORY"].rename(name)

def build_transactions(master):
    numeric_parts = []
    modes1 = []
    modes2 = []

    columns = [ID, "TRANS_DATE", "TRANS_LV1", "TRANS_LV2", "TRANS_AMOUNT"]

    for chunk in iter_source(RAW_FILES["transaction"], columns):
        d = rows_in_window(chunk, master, "TRANS_DATE", "transaction")
        if d.empty:
            continue

        d["TRANS_AMOUNT"] = parse_numbers(d["TRANS_AMOUNT"])

        n = (
            d.groupby(ID)["TRANS_AMOUNT"]
            .agg(["sum", "count", "max"])
            .rename(
                columns={
                    "sum": "AMOUNT_SUM",
                    "count": "AMOUNT_N",
                    "max": "TRANS_AMOUNT_MAX",
                }
            )
        )
        numeric_parts.append(n)

        for col, container in [("TRANS_LV1", modes1), ("TRANS_LV2", modes2)]:
            cat = clean_category(d[col])
            pairs = pd.DataFrame({ID: d[ID], "CATEGORY": cat}).dropna(
                subset=["CATEGORY"]
            )
            container.append(
                pairs.groupby([ID, "CATEGORY"]).size().rename("N").reset_index()
            )

    base = master[[ID]].set_index(ID)

    if numeric_parts:
        n = (
            pd.concat(numeric_parts)
            .groupby(level=0)
            .agg(
                AMOUNT_SUM=("AMOUNT_SUM", "sum"),
                AMOUNT_N=("AMOUNT_N", "sum"),
                TRANS_AMOUNT_MAX=("TRANS_AMOUNT_MAX", "max"),
            )
        )
        n["TRANS_AMOUNT_MEAN"] = (
            n["AMOUNT_SUM"] / n["AMOUNT_N"].replace(0, np.nan)
        )
        base = base.join(n[["TRANS_AMOUNT_MAX", "TRANS_AMOUNT_MEAN"]])
    else:
        base["TRANS_AMOUNT_MAX"] = np.nan
        base["TRANS_AMOUNT_MEAN"] = np.nan

    base = (
        base
        .join(reduce_mode(modes1, "TRANS_LV1_MODE"))
        .join(reduce_mode(modes2, "TRANS_LV2_MODE"))
    )

    base[["TRANS_LV1_MODE", "TRANS_LV2_MODE"]] = (
        base[["TRANS_LV1_MODE", "TRANS_LV2_MODE"]]
        .fillna("NO_TRANSACTION")
    )
    base[["TRANS_AMOUNT_MAX", "TRANS_AMOUNT_MEAN"]] = (
        base[["TRANS_AMOUNT_MAX", "TRANS_AMOUNT_MEAN"]]
        .fillna(0)
    )

    return base.reset_index()[[ID] + TRANSACTION_FEATURES]

def build_activity(master):
    count_parts = []
    day_parts = []

    for chunk in iter_source(
        RAW_FILES["activity"],
        [ID, "ACTIVITY_DATE", "ACTIVITY_NAME"],
    ):
        d = rows_in_window(chunk, master, "ACTIVITY_DATE", "activity")
        if d.empty:
            continue

        cat = clean_category(d["ACTIVITY_NAME"])
        d["LOGIN_N"] = cat.str.contains(
            "LOGIN", na=False, regex=False
        ).astype(int)
        d["INTEREST_N"] = cat.str.contains(
            "INTEREST_RATE", na=False, regex=False
        ).astype(int)
        d["ACTIVITY_N"] = 1

        count_parts.append(
            d.groupby(ID)[["LOGIN_N", "INTEREST_N", "ACTIVITY_N"]].sum()
        )

        day_parts.append(
            pd.DataFrame(
                {
                    ID: d[ID],
                    "DAY": d["ACTIVITY_DATE"].dt.normalize(),
                }
            ).drop_duplicates()
        )

    base = master[[ID]].set_index(ID)

    if count_parts:
        c = pd.concat(count_parts).groupby(level=0).sum()
        active_days = (
            pd.concat(day_parts, ignore_index=True)
            .drop_duplicates()
            .groupby(ID)
            .size()
        )
        c["LOGIN_PER_ACTIVE_DAY"] = c["LOGIN_N"] / active_days
        c["INTEREST_RATE_RATIO"] = c["INTEREST_N"] / c["ACTIVITY_N"]
        base = base.join(c[ACTIVITY_FEATURES])
    else:
        base[ACTIVITY_FEATURES] = np.nan

    return (
        base.fillna(0)
        .reset_index()[[ID] + ACTIVITY_FEATURES]
    )

def build_latest_snapshot(master, source, numeric_columns):
    parts = []

    for chunk in iter_source(
        RAW_FILES[source],
        [ID, "MONTH"] + numeric_columns,
    ):
        d = rows_in_window(chunk, master, "MONTH", source)
        if d.empty:
            continue

        for col in numeric_columns:
            d[col] = parse_numbers(d[col])

        parts.append(d[[ID, "MONTH"] + numeric_columns])

    if parts:
        all_rows = pd.concat(parts, ignore_index=True)
        maximum = all_rows.groupby(ID)["MONTH"].transform("max")
        latest = all_rows.loc[all_rows["MONTH"].eq(maximum)].copy()

        # Keep only exact duplicates; otherwise stop.
        latest = latest.drop_duplicates()
        dup = latest[ID].duplicated(False)
        if dup.any():
            sample = latest.loc[dup].head(30)
            raise ValueError(
                f"{source}: cùng ID/tháng cuối có snapshot khác nhau.\n"
                f"{sample.to_string(index=False)}"
            )

        latest = latest[[ID] + numeric_columns]
    else:
        latest = pd.DataFrame(columns=[ID] + numeric_columns)

    feature = master[[ID]].merge(
        latest,
        on=ID,
        how="left",
        validate="one_to_one",
    )

    for col in numeric_columns:
        feature[col] = pd.to_numeric(feature[col], errors="coerce").fillna(0)

    if source == "deposit":
        for col in ["COUNT_CA_ACCT", "COUNT_TD_ACCT"]:
            feature[col] = feature[col].round().astype(int)
        return feature[[ID] + DEPOSIT_FEATURES]

    feature["COUNT_OF_LOAN"] = feature["COUNT_OF_LOAN"].round().astype(int)
    feature["TOTAL_LOAN_AMOUNT"] = (
        feature["COUNT_OF_LOAN"] * feature["AVG_LOAN_AMOUNT"]
    )
    return feature[[ID] + LENDING_FEATURES]

def rebuild_bsym_dynamic(master):
    combined = master[[ID]].copy()

    pieces = [
        build_activity(master),
        build_transactions(master),
        build_latest_snapshot(
            master,
            "deposit",
            ["COUNT_CA_ACCT", "AVG_CA_BALANCE", "COUNT_TD_ACCT", "AVG_TD_BALANCE"],
        ),
        build_latest_snapshot(
            master,
            "lending",
            ["COUNT_OF_LOAN", "AVG_LOAN_AMOUNT"],
        ),
    ]

    for frame in pieces:
        combined = combined.merge(
            frame,
            on=ID,
            how="left",
            validate="one_to_one",
        )

    require_columns(
        combined,
        [ID]
        + ACTIVITY_FEATURES
        + TRANSACTION_FEATURES
        + DEPOSIT_FEATURES
        + LENDING_FEATURES,
        "B-sym dynamic",
    )

    return combined

def make_bsym_cutoffs(timeline, train_idx, seed, label):
    t = timeline.copy().reset_index(drop=True)

    y = t[TARGET].to_numpy(dtype=np.int8)

    train_idx = np.asarray(train_idx, dtype=int)
    train_positive_idx = train_idx[y[train_idx] == 1]

    durations = (
        t.loc[train_positive_idx, "FIRST_CARD_DATE"].reset_index(drop=True)
        - t.loc[train_positive_idx, "RELATIONSHIP_DATE"].reset_index(drop=True)
    ).dt.days.to_numpy()

    durations = durations[np.isfinite(durations)].astype(int)

    if len(durations) == 0:
        raise ValueError(f"{label}: không có positive duration trong training.")

    if np.min(durations) <= LANDMARK_DAYS:
        raise ValueError(f"{label}: có positive duration <= 60.")

    rng = np.random.default_rng(20261012 + int(seed))

    positive = y == 1
    negative = ~positive

    sampled = np.full(len(t), np.nan)

    positive_duration = (
        t.loc[positive, "FIRST_CARD_DATE"]
        - t.loc[positive, "RELATIONSHIP_DATE"]
    ).dt.days.to_numpy()

    sampled[positive] = positive_duration

    neg_idx = np.where(negative)[0]
    sampled[neg_idx] = rng.choice(
        durations,
        size=len(neg_idx),
        replace=True,
    )

    proposed = (
        pd.to_datetime(t["RELATIONSHIP_DATE"])
        + pd.to_timedelta(sampled, unit="D")
    )

    cutoff = proposed.copy()

    # Positive: actual event cutoff.
    cutoff.loc[positive] = pd.to_datetime(
        t.loc[positive, "FIRST_CARD_DATE"]
    )

    # Negative: pseudo duration, capped at T_end.
    t_end = pd.to_datetime(t["T_END"])
    cutoff.loc[negative] = np.minimum(
        proposed.loc[negative].to_numpy(dtype="datetime64[ns]"),
        t_end.loc[negative].to_numpy(dtype="datetime64[ns]"),
    )

    landmark = pd.to_datetime(t["LANDMARK_DATE"])

    if (positive & ~(cutoff > landmark)).any():
        raise ValueError(f"{label}: positive B-sym cutoff not after landmark.")

    if (negative & (cutoff < landmark)).any():
        raise ValueError(f"{label}: negative B-sym cutoff before landmark.")

    out = t[[ID, TARGET, "RELATIONSHIP_DATE", "LANDMARK_DATE", "FIRST_CARD_DATE", "T_END"]].copy()
    out["SAMPLED_DURATION_DAYS"] = sampled.astype(int)
    out["PROPOSED_CUTOFF"] = proposed
    out["BSYM_CUTOFF"] = pd.to_datetime(cutoff)
    out["CAPPED_AT_TEND"] = (
        negative & (out["BSYM_CUTOFF"] < out["PROPOSED_CUTOFF"])
    )

    return out

def build_or_load_bsym(
    base_a,
    timeline,
    train_idx,
    seed,
    cohort_name,
    force=False,
):
    cache_dir = CACHE_ROOT / cohort_name
    cache_dir.mkdir(parents=True, exist_ok=True)

    matrix_file = cache_dir / f"Bsym_seed_{int(seed):03d}.csv"
    cutoff_file = cache_dir / f"Bsym_cutoff_seed_{int(seed):03d}.csv"

    if matrix_file.is_file() and cutoff_file.is_file() and not force:
        matrix = read_processed(matrix_file)
        cutoffs = read_processed(cutoff_file)

        matrix[TARGET] = pd.to_numeric(
            matrix[TARGET], errors="raise"
        ).astype(np.int8)

        if len(matrix) != len(base_a):
            raise ValueError(f"{cohort_name}: cached B-sym wrong N.")

        if not np.array_equal(
            matrix[ID].astype(str).to_numpy(),
            base_a[ID].astype(str).to_numpy(),
        ):
            raise ValueError(f"{cohort_name}: cached B-sym ID/order mismatch.")

        if not np.array_equal(
            matrix[TARGET].to_numpy(),
            base_a[TARGET].to_numpy(),
        ):
            raise ValueError(f"{cohort_name}: cached B-sym target mismatch.")

        print(f"Reuse cached {cohort_name} B-sym seed {seed}.")
        return matrix, cutoffs

    print(f"\nBUILD {cohort_name} B-sym seed {seed} ...")

    cutoffs = make_bsym_cutoffs(
        timeline=timeline,
        train_idx=train_idx,
        seed=seed,
        label=cohort_name,
    )

    master = cutoffs[
        [ID, "RELATIONSHIP_DATE", "BSYM_CUTOFF"]
    ].copy()

    dynamic = rebuild_bsym_dynamic(master)

    matrix = (
        base_a[[ID, TARGET] + STATIC_FEATURES]
        .copy()
        .merge(
            dynamic,
            on=ID,
            how="left",
            validate="one_to_one",
            sort=False,
        )
    )

    matrix = matrix[[ID, TARGET] + FEATURES].reset_index(drop=True)

    if not np.array_equal(
        matrix[ID].astype(str).to_numpy(),
        base_a[ID].astype(str).to_numpy(),
    ):
        raise AssertionError(f"{cohort_name}: B-sym ID/order differs from A.")

    if not np.array_equal(
        matrix[TARGET].to_numpy(dtype=np.int8),
        base_a[TARGET].to_numpy(dtype=np.int8),
    ):
        raise AssertionError(f"{cohort_name}: B-sym target differs from A.")

    matrix.to_csv(matrix_file, index=False)
    cutoffs.to_csv(cutoff_file, index=False)

    print("Saved:", matrix_file)

    return matrix, cutoffs

# ============================================================
# 6. MODELS
# ============================================================

def make_preprocessor(X_train, learner):
    numeric_cols = (
        X_train.select_dtypes(include=np.number).columns.tolist()
    )
    categorical_cols = [
        c for c in X_train.columns if c not in numeric_cols
    ]

    if learner == "LR":
        numeric_pipe = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
            ]
        )
    else:
        numeric_pipe = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ]
        )

    cat_kwargs = {"handle_unknown": "ignore"}
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        cat_kwargs["sparse_output"] = False
    else:
        cat_kwargs["sparse"] = False

    categorical_pipe = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="MISSING"),
            ),
            ("onehot", OneHotEncoder(**cat_kwargs)),
        ]
    )

    transforms = []

    if numeric_cols:
        transforms.append(("num", numeric_pipe, numeric_cols))

    if categorical_cols:
        transforms.append(("cat", categorical_pipe, categorical_cols))

    return ColumnTransformer(
        transforms,
        remainder="drop",
    )

def make_model(learner, seed):
    if learner == "LR":
        return LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
            random_state=seed,
        )

    if learner == "RF":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=seed,
            n_jobs=2,
        )

    if learner == "LightGBM":
        return LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=seed,
            n_jobs=2,
            verbosity=-1,
        )

    if learner == "XGBoost":
        return XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            max_bin=256,
            random_state=seed,
            n_jobs=2,
        )

    raise ValueError(learner)

def fit_branch(matrix, train_idx, test_idx, learner, seed):
    X_train = matrix.loc[train_idx, FEATURES].copy()
    X_test = matrix.loc[test_idx, FEATURES].copy()

    y_train = matrix.loc[train_idx, TARGET].to_numpy(dtype=np.int8)
    y_test = matrix.loc[test_idx, TARGET].to_numpy(dtype=np.int8)

    preprocess = make_preprocessor(X_train, learner)

    Xtr = preprocess.fit_transform(X_train)
    Xte = preprocess.transform(X_test)

    model = make_model(learner, seed)
    model.fit(Xtr, y_train)

    prob = model.predict_proba(Xte)[:, 1]

    auc = float(roc_auc_score(y_test, prob))

    fpr, tpr, _ = roc_curve(y_test, prob)
    ks = float(np.max(tpr - fpr))

    brier = float(brier_score_loss(y_test, prob))

    prevalence = float(y_test.mean())
    ref = prevalence * (1 - prevalence)
    bss = float(1 - brier / ref) if ref > 0 else np.nan

    del model, preprocess, Xtr, Xte
    gc.collect()

    return {
        "AUC": auc,
        "KS": ks,
        "Brier": brier,
        "BSS": bss,
    }, prob

def paired_bootstrap(y, probabilities, seed=42, n_bootstrap=1000):
    y = np.asarray(y, dtype=np.int8)
    rng = np.random.default_rng(seed)

    contrasts = {
        "B-A": [],
        "Bsym-A": [],
        "B-Bsym": [],
    }

    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y), size=len(y))
        yb = y[idx]

        if yb.min() == yb.max():
            continue

        auc = {
            k: roc_auc_score(yb, np.asarray(v)[idx])
            for k, v in probabilities.items()
        }

        contrasts["B-A"].append(auc["B"] - auc["A"])
        contrasts["Bsym-A"].append(auc["Bsym"] - auc["A"])
        contrasts["B-Bsym"].append(auc["B"] - auc["Bsym"])

    rows = []

    for contrast, values in contrasts.items():
        values = np.asarray(values, dtype=float)

        left, right = contrast.split("-")

        observed = (
            roc_auc_score(y, probabilities[left])
            - roc_auc_score(y, probabilities[right])
        )

        rows.append(
            {
                "contrast": contrast,
                "observed": float(observed),
                "q025": float(np.quantile(values, 0.025)),
                "q975": float(np.quantile(values, 0.975)),
                "bootstrap_valid": int(len(values)),
                "n_gt_0": int((values > 0).sum()),
            }
        )

    return pd.DataFrame(rows)

def evaluate_four_learners(
    a,
    bsym,
    b,
    train_idx,
    test_idx,
    seed,
    label,
):
    # Exact shared cohort/order/label check.
    for name, frame in [("Bsym", bsym), ("B", b)]:
        if not np.array_equal(
            a[ID].astype(str).to_numpy(),
            frame[ID].astype(str).to_numpy(),
        ):
            raise ValueError(f"{label}: {name} ID/order mismatch.")

        if not np.array_equal(
            a[TARGET].to_numpy(dtype=np.int8),
            frame[TARGET].to_numpy(dtype=np.int8),
        ):
            raise ValueError(f"{label}: {name} target mismatch.")

    learners = ["LR", "RF", "LightGBM", "XGBoost"]
    branches = {"A": a, "Bsym": bsym, "B": b}

    rows = []
    probs_by_learner = {}

    for learner in learners:
        probs = {}
        aucs = {}

        print(f"\n{label} | {learner}")

        for branch_name in ["A", "Bsym", "B"]:
            metrics, prob = fit_branch(
                branches[branch_name],
                train_idx,
                test_idx,
                learner,
                seed,
            )

            probs[branch_name] = prob
            aucs[branch_name] = metrics["AUC"]

            rows.append(
                {
                    "label": label,
                    "seed": int(seed),
                    "learner": learner,
                    "branch": branch_name,
                    **metrics,
                }
            )

            print(
                f"  {branch_name:4s}: "
                f"AUC={metrics['AUC']:.6f} | "
                f"KS={metrics['KS']:.6f} | "
                f"BSS={metrics['BSS']:.6f}"
            )

        tau_ab = float(
            kendalltau(probs["A"], probs["B"], nan_policy="omit").statistic
        )
        tau_absym = float(
            kendalltau(probs["A"], probs["Bsym"], nan_policy="omit").statistic
        )
        tau_bsymb = float(
            kendalltau(probs["Bsym"], probs["B"], nan_policy="omit").statistic
        )

        rows.append(
            {
                "label": label,
                "seed": int(seed),
                "learner": learner,
                "branch": "CONTRASTS",
                "AUC": np.nan,
                "KS": np.nan,
                "Brier": np.nan,
                "BSS": np.nan,
                "D_total_B_minus_A": aucs["B"] - aucs["A"],
                "D_matched_Bsym_minus_A": aucs["Bsym"] - aucs["A"],
                "D_endpoint_B_minus_Bsym": aucs["B"] - aucs["Bsym"],
                "tau_A_B": tau_ab,
                "tau_A_Bsym": tau_absym,
                "tau_Bsym_B": tau_bsymb,
            }
        )

        probs_by_learner[learner] = probs

    return pd.DataFrame(rows), probs_by_learner

# ============================================================
# 7. PRIMARY R50
# ============================================================

def run_primary_r50(a, b, timeline, out):
    rows = []
    checkpoint = out / "PRIMARY_R50_four_learners.csv"

    all_idx = np.arange(len(a))
    y = a[TARGET].to_numpy(dtype=np.int8)

    for r, seed in enumerate(R50_SEEDS, start=1):
        print("\n" + "=" * 110)
        print(f"PRIMARY R50 {r}/50 | seed={seed}")
        print("=" * 110)

        train_idx, test_idx = train_test_split(
            all_idx,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed,
        )

        bsym, _ = build_or_load_bsym(
            base_a=a,
            timeline=timeline,
            train_idx=train_idx,
            seed=seed,
            cohort_name="PRIMARY_R50",
        )

        result, _ = evaluate_four_learners(
            a=a,
            bsym=bsym,
            b=b,
            train_idx=train_idx,
            test_idx=test_idx,
            seed=seed,
            label=f"PRIMARY_R50_seed_{seed}",
        )

        contrast_rows = result.loc[
            result["branch"].eq("CONTRASTS")
        ].copy()

        rows.append(contrast_rows)

        pd.concat(rows, ignore_index=True).to_csv(
            checkpoint,
            index=False,
        )

        del bsym
        gc.collect()

    r50 = pd.concat(rows, ignore_index=True)

    summary_rows = []

    for learner, g in r50.groupby("learner"):
        for metric in [
            "D_total_B_minus_A",
            "D_matched_Bsym_minus_A",
            "D_endpoint_B_minus_Bsym",
            "tau_A_B",
            "tau_A_Bsym",
            "tau_Bsym_B",
        ]:
            values = pd.to_numeric(g[metric], errors="coerce").dropna().to_numpy()
            summary_rows.append(
                {
                    "learner": learner,
                    "metric": metric,
                    "R": len(values),
                    "mean": float(np.mean(values)),
                    "sd": float(np.std(values, ddof=1)),
                    "median": float(np.median(values)),
                    "q025": float(np.quantile(values, 0.025)),
                    "q975": float(np.quantile(values, 0.975)),
                    "n_gt_0": int((values > 0).sum())
                    if metric.startswith("D_")
                    else np.nan,
                }
            )

    summary = pd.DataFrame(summary_rows)

    r50.to_csv(out / "PRIMARY_R50_four_learners.csv", index=False)
    summary.to_csv(out / "PRIMARY_R50_summary.csv", index=False)

    return r50, summary

# ============================================================
# 8. MAIN
# ============================================================

def run_all():
    mount_drive()

    for p in [
        PRIMARY_A_FILE,
        PRIMARY_B_FILE,
        PRIMARY_C_FILE,
        E2_SPLIT_FILE,
        *RAW_FILES.values(),
    ]:
        if not Path(p).is_file():
            raise FileNotFoundError(p)

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    out = OUTPUT_PARENT / f"run_{stamp}"
    out.mkdir(parents=True, exist_ok=False)

    CACHE_ROOT.mkdir(parents=True, exist_ok=True)

    print("=" * 120)
    print("VIB — REUSE PROCESSED A/B/C/H180; BUILD ONLY B-SYM; THEN 4 LEARNERS")
    print("=" * 120)
    print("Output:", out)

    # --------------------------------------------------------
    # PRIMARY
    # --------------------------------------------------------
    print("\n" + "=" * 120)
    print("PART I — PRIMARY VIB60ABC")
    print("=" * 120)

    a, b, timeline, train_e2, test_e2 = load_primary_processed()

    bsym_e2, cutoff_e2 = build_or_load_bsym(
        base_a=a,
        timeline=timeline,
        train_idx=train_e2,
        seed=42,
        cohort_name="PRIMARY_EXACT_E2",
    )

    primary_results, primary_probs = evaluate_four_learners(
        a=a,
        bsym=bsym_e2,
        b=b,
        train_idx=train_e2,
        test_idx=test_e2,
        seed=42,
        label="PRIMARY_EXACT_E2",
    )

    primary_results.to_csv(
        out / "PRIMARY_exactE2_four_learners.csv",
        index=False,
    )

    y_test = a.loc[test_e2, TARGET].to_numpy(dtype=np.int8)

    primary_boot = paired_bootstrap(
        y=y_test,
        probabilities=primary_probs["XGBoost"],
        seed=42,
        n_bootstrap=N_BOOTSTRAP,
    )

    primary_boot.to_csv(
        out / "PRIMARY_XGB_paired_bootstrap.csv",
        index=False,
    )

    if RUN_R50:
        r50, r50_summary = run_primary_r50(
            a=a,
            b=b,
            timeline=timeline,
            out=out,
        )
    else:
        r50 = None
        r50_summary = None

    # --------------------------------------------------------
    # H180
    # --------------------------------------------------------
    print("\n" + "=" * 120)
    print("PART II — H180 FROM PROCESSED FILES")
    print("=" * 120)

    h_a, h_b, h_timeline, h_train, h_test, h_run = load_h180_processed()

    h_bsym, h_cutoff = build_or_load_bsym(
        base_a=h_a,
        timeline=h_timeline,
        train_idx=h_train,
        seed=42,
        cohort_name="H180_SEED42",
    )

    h_results, h_probs = evaluate_four_learners(
        a=h_a,
        bsym=h_bsym,
        b=h_b,
        train_idx=h_train,
        test_idx=h_test,
        seed=42,
        label="H180_SEED42",
    )

    h_results.to_csv(
        out / "H180_four_learners.csv",
        index=False,
    )

    h_y_test = h_a.loc[h_test, TARGET].to_numpy(dtype=np.int8)

    h_boot = paired_bootstrap(
        y=h_y_test,
        probabilities=h_probs["XGBoost"],
        seed=42,
        n_bootstrap=N_BOOTSTRAP,
    )

    h_boot.to_csv(
        out / "H180_XGB_paired_bootstrap.csv",
        index=False,
    )

    # --------------------------------------------------------
    # FINAL PRINT
    # --------------------------------------------------------
    print("\n" + "=" * 120)
    print("PRIMARY — 4 LEARNERS")
    print("=" * 120)
    print(
        primary_results.loc[
            primary_results["branch"].eq("CONTRASTS"),
            [
                "learner",
                "D_total_B_minus_A",
                "D_matched_Bsym_minus_A",
                "D_endpoint_B_minus_Bsym",
                "tau_A_B",
                "tau_A_Bsym",
                "tau_Bsym_B",
            ],
        ].round(6).to_string(index=False)
    )

    print("\nPRIMARY XGB PAIRED BOOTSTRAP")
    print(primary_boot.round(6).to_string(index=False))

    if r50_summary is not None:
        print("\n" + "=" * 120)
        print("PRIMARY R50 SUMMARY")
        print("=" * 120)
        print(r50_summary.round(6).to_string(index=False))

    print("\n" + "=" * 120)
    print("H180 — 4 LEARNERS")
    print("=" * 120)
    print(
        h_results.loc[
            h_results["branch"].eq("CONTRASTS"),
            [
                "learner",
                "D_total_B_minus_A",
                "D_matched_Bsym_minus_A",
                "D_endpoint_B_minus_Bsym",
                "tau_A_B",
                "tau_A_Bsym",
                "tau_Bsym_B",
            ],
        ].round(6).to_string(index=False)
    )

    print("\nH180 XGB PAIRED BOOTSTRAP")
    print(h_boot.round(6).to_string(index=False))

    print("\nSAVED:", out)

    return {
        "output": out,
        "primary_results": primary_results,
        "primary_bootstrap": primary_boot,
        "r50_summary": r50_summary,
        "h180_results": h_results,
        "h180_bootstrap": h_boot,
        "h180_source_run": h_run,
    }


In [ ]:

RESULTS_VIB_REUSE_PROCESSED = run_all()


VIB — REUSE PROCESSED A/B/C/H180; BUILD ONLY B-SYM; THEN 4 LEARNERS
Output: /content/drive/MyDrive/Fintect/VIB_REUSE_PROCESSED_ABCH180_BSYM_4LEARNERS/run_20261121T025814486520Z

PART I — PRIMARY VIB60ABC
PRIMARY processed files OK: N=94,453; positives=4,374; train/test=75,562/18,891
Reuse cached PRIMARY_EXACT_E2 B-sym seed 42.

PRIMARY_EXACT_E2 | LR
  A   : AUC=0.883482 | KS=0.612554 | BSS=-2.213862
  Bsym: AUC=0.899876 | KS=0.645569 | BSS=-2.006884
  B   : AUC=0.896169 | KS=0.639355 | BSS=-2.065128

PRIMARY_EXACT_E2 | RF
  A   : AUC=0.879328 | KS=0.622858 | BSS=0.180107
  Bsym: AUC=0.900065 | KS=0.656401 | BSS=0.210155
  B   : AUC=0.903174 | KS=0.650412 | BSS=0.218966

PRIMARY_EXACT_E2 | LightGBM
  A   : AUC=0.907719 | KS=0.660362 | BSS=0.211041
  Bsym: AUC=0.919474 | KS=0.693974 | BSS=0.237533
  B   : AUC=0.919502 | KS=0.686424 | BSS=0.255168

PRIMARY_EXACT_E2 | XGBoost
  A   : AUC=0.909824 | KS=0.665665 | BSS=0.216693
  Bsym: AUC=0.922121 | KS=0.701718 | BSS=0.248138
  B   : AUC=0.9

FileNotFoundError: Không tìm thấy run H180 đã xử lý có đủ:
MATRICES/A_fixed_horizon_matrix.csv
MATRICES/B_fixed_horizon_matrix.csv
cohort_fixed_horizon_master.csv
shared_H180_train_test_split.csv

### Step 4 — Table 5 (seed-42 split, XGBoost, 1,000 paired bootstrap)

In [ ]:
# HƯỚNG 2 — BƯỚC 3: TABLE 5 (chạy SAU ô R50 "VIB60ABC — A / B / B-sym + R50 + H=180" và run_all)
# seed-42 split, XGBoost, 1,000 paired bootstrap (seed 42). Kết quả 21/11/2026:
# A 0.910242 [0.9018,0.9188]; B-sym 0.923461 [0.9159,0.9307]; B 0.922914 [0.9153,0.9303]
# B-A 0.012672 [0.0078,0.0178]; Bsym-A 0.013219 [0.0083,0.0181]; B-Bsym -0.000547 [-0.0025,0.0014]
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
_a, _b, _tl, _tr_e2, _te_e2 = load_primary_processed()
_y = _a[TARGET].to_numpy(dtype=np.int8)
_tr, _te = train_test_split(np.arange(len(_a)), test_size=TEST_SIZE, stratify=_y, random_state=42)
_bs, _ = build_or_load_bsym(base_a=_a, timeline=_tl, train_idx=_tr, seed=42, cohort_name='PRIMARY_R50')
_P, _M = {}, {}
for _n, _f in [('A', _a), ('Bsym', _bs), ('B', _b)]:
    _M[_n], _P[_n] = fit_branch(_f, _tr, _te, 'XGBoost', 42)
    print(_n, {k: round(v, 6) for k, v in _M[_n].items()})
_yt = _y[_te]
_rng = np.random.default_rng(42)
_rows = {k: [] for k in ['A', 'Bsym', 'B', 'B-A', 'Bsym-A', 'B-Bsym']}
for _ in range(1000):
    _i = _rng.integers(0, len(_yt), len(_yt))
    if _yt[_i].min() == _yt[_i].max():
        continue
    _au = {k: roc_auc_score(_yt[_i], np.asarray(v)[_i]) for k, v in _P.items()}
    for k in ['A', 'Bsym', 'B']:
        _rows[k].append(_au[k])
    _rows['B-A'].append(_au['B'] - _au['A']); _rows['Bsym-A'].append(_au['Bsym'] - _au['A']); _rows['B-Bsym'].append(_au['B'] - _au['Bsym'])
_obs = {k: _M[k]['AUC'] for k in ['A', 'Bsym', 'B']}
_obs.update({'B-A': _obs['B'] - _obs['A'], 'Bsym-A': _obs['Bsym'] - _obs['A'], 'B-Bsym': _obs['B'] - _obs['Bsym']})
_T5 = pd.DataFrame([{'quantity': k, 'observed': _obs[k], 'q025': np.quantile(v, .025), 'q975': np.quantile(v, .975), 'n_valid': len(v), 'n_gt_0': int((np.asarray(v) > 0).sum())} for k, v in _rows.items()])
print(_T5.round(6).to_string(index=False))


## Section B — H180 cohort (Table 6, S19)

VIB H=180 — A / B / B-sym — XGBoost — 1,000 paired bootstrap
Notebook này được xây trực tiếp trên code VIB H180 V3 đã dùng trước đây.
Giữ nguyên H180 V3:
- cohort đủ theo dõi đến ngày 240;
- target: mở thẻ trong (r+60, r+240];
- A dùng 18 biến A60 gốc;
- B: positive cutoff = event, negative cutoff = r+240;
- shared stratified 80/20 split seed 42;
- XGBoost H180 V3, không resampling.
Bổ sung duy nhất B-sym:
- positive giữ event cutoff;
- negative rút L*=e-r từ positive TRAIN customers only, with replacement;
- cutoff tau*=min(r+L*, r+240);
- cùng ID / label / split A-B-Bsym;
- 1,000 paired customer bootstrap cho B-A, Bsym-A, B-Bsym.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "joblib": "joblib",
    "openpyxl": "openpyxl",
}

missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Libraries ready.")


Libraries ready.


In [ ]:
# -*- coding: utf-8 -*-
# Khi chạy trong một ô Colab, chuẩn bị các gói còn thiếu trước imports.
if __name__ == '__main__':
    import importlib.util
    import subprocess
    import sys
    _packages = {'numpy': 'numpy', 'pandas': 'pandas>=2.0', 'sklearn': 'scikit-learn',
                 'xgboost': 'xgboost', 'joblib': 'joblib', 'openpyxl': 'openpyxl'}
    _missing = [p for m, p in _packages.items() if importlib.util.find_spec(m) is None]
    if _missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

# -*- coding: utf-8 -*-
"""VIB: landmark 60 ngày, theo dõi H=180 ngày, A/B cùng cohort.

Chạy file này trên Colab hoặc chạy notebook cùng tên.
Kết quả nghiên cứu chỉ được tạo khi chạy trên dữ liệu VIB của tác giả.
"""

# %% 1. IMPORTS VÀ CẤU HÌNH
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path('/content/drive/MyDrive/Fintect')
LANDMARK_DAYS = 60
HORIZON_DAYS = 180
# Bao gồm toàn bộ ngày 31/12/2019; không chuyển MONTH về đầu tháng.
OBSERVATION_END = pd.Timestamp('2019-12-31 23:59:59.999999999')
SPLIT_SEED = 42
MODEL_SEED = 42
BOOTSTRAP_SEED = 42
TEST_SIZE = 0.20
N_BOOTSTRAP = 1000
CSV_CHUNKSIZE = 250_000
MIN_EVENTS_FOR_REVIEW = 300  # Chỉ nhắc kiểm tra; không tự đổi H.
EXPECTED_BASE_CUSTOMERS = 94_453
EXPECTED_BASE_POSITIVES = 4_374

ID = 'CUSTOMER_NUMBER'
Y_OLD = 'TARGET_60D'
Y = 'TARGET_H'
START = 'RELATIONSHIP_START_DATE'
EVENT = 'FIRST_CARD_MONTH'
CUTOFF = 'FEATURE_CUTOFF_DATE'

STATIC_FEATURES = ['CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'AGE']
TRANSACTION_FEATURES = ['TRANS_LV1_MODE', 'TRANS_LV2_MODE',
                        'TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']
ACTIVITY_FEATURES = ['LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO']
DEPOSIT_FEATURES = ['COUNT_CA_ACCT', 'AVG_CA_BALANCE', 'COUNT_TD_ACCT', 'AVG_TD_BALANCE']
LENDING_FEATURES = ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT', 'TOTAL_LOAN_AMOUNT']
# Đúng thứ tự 18 biến trong mã A/B hiện tại của tác giả.
FEATURES = STATIC_FEATURES + ACTIVITY_FEATURES + TRANSACTION_FEATURES + DEPOSIT_FEATURES + LENDING_FEATURES

FILES = {
    'A_60D': ROOT / 'final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv',
    'customer': ROOT / '1.Data_Customer.csv',
    'card': ROOT / '6.Data_Card.xlsx',
    'transaction': ROOT / '2.Data_MyVIB_Transaction.csv',
    'activity': ROOT / '3.Data_MyVIB_Activity.csv',
    'deposit': ROOT / '4.Data_Deposit.csv',
    'lending': ROOT / '5.Data_Lending.xlsx',
}
OUTPUT_PARENT = ROOT / f'VIB_FIXED_HORIZON_H{HORIZON_DAYS}_A{LANDMARK_DAYS}_B{LANDMARK_DAYS+HORIZON_DAYS}_V3'
# Không đoán dd/mm và mm/dd. ISO yyyy-mm-dd/Excel datetime không cần thay biến này.
DATE_DAYFIRST = None
# Đã kiểm tra từ log chẩn đoán của tác giả ngày 06/10/2026:
# activity có 1/22/2019; deposit có 1/31/2019 (tháng/ngày/năm).
# Nguồn khác tiếp tục dừng khi gặp định dạng mơ hồ chưa xác định.
SOURCE_DATE_DAYFIRST = {'activity': False, 'deposit': False}
CODE_REVISION = 'V3_ID_PADDING_MMDD_20261006'
EXTRA_DATE_SOURCE_FILES = []  # Thêm đường dẫn master 60D nếu lưu ngoài ROOT.
A_AUDIT_RTOL = 1e-12
A_AUDIT_ATOL = 1e-8
DYNAMIC_FEATURES = ACTIVITY_FEATURES + TRANSACTION_FEATURES + DEPOSIT_FEATURES + LENDING_FEATURES
XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'eval_metric': 'logloss', 'objective': 'binary:logistic',
    'random_state': MODEL_SEED, 'n_jobs': -1,
}


# %% 2. HÀM ĐỌC, KIỂM TRA VÀ LƯU
def show_table(df, title):
    print('\n' + title)
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))


def require_columns(df, columns, name):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f'{name}: thiếu cột {missing}; hiện có {list(df.columns)}')


def clean_id(series):
    result = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    result = result.mask(result.str.upper().isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)
    # Đã xác minh: bốn CSV gốc dùng ID số có padding (00639362),
    # A/master dùng 639362, và chẩn đoán không thấy hai ID bị gộp.
    # Chuẩn hóa chính xác trên CHUỖI; không qua float hoặc ký hiệu khoa học.
    numeric = result.str.fullmatch(r'[0-9]+', na=False)
    result.loc[numeric] = result.loc[numeric].str.lstrip('0').replace('', '0')
    return result


def clean_number(series):
    return pd.to_numeric(series.astype('string').str.replace(',', '', regex=False).str.strip(), errors='coerce')


def clean_category(series):
    result = series.astype('string').str.strip().str.upper()
    return result.mask(result.isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)


def normalize_frame(df, name):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.upper()
    if df.columns.duplicated().any():
        raise ValueError(f'{name}: tên cột trùng sau chuẩn hóa.')
    require_columns(df, [ID], name)
    df[ID] = clean_id(df[ID])
    return df.loc[df[ID].notna()].copy()


def read_table(path, columns=None):
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        header = pd.read_excel(path, nrows=0)
        names = {str(c).strip().upper(): c for c in header.columns}
        if ID not in names:
            raise ValueError(f'{path.name}: thiếu {ID}.')
        df = pd.read_excel(path, dtype={names[ID]: str})
    else:
        header = pd.read_csv(path, nrows=0)
        names = {str(c).strip().upper(): c for c in header.columns}
        if ID not in names:
            raise ValueError(f'{path.name}: thiếu {ID}.')
        df = pd.read_csv(path, dtype={names[ID]: str}, low_memory=False)
    df = normalize_frame(df, path.name)
    if columns is not None:
        require_columns(df, columns, path.name)
        df = df[columns].copy()
    return df


def iter_source(path, columns):
    """CSV đọc theo chunk để tránh ghép toàn bộ lịch sử vào RAM."""
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        yield read_table(path, columns)
        return
    header = pd.read_csv(path, nrows=0)
    raw_names = {str(c).strip().upper(): c for c in header.columns}
    missing = [c for c in columns if c not in raw_names]
    if missing:
        raise ValueError(f'{path.name}: thiếu {missing}')
    selected = [raw_names[c] for c in columns]
    for chunk in pd.read_csv(path, usecols=selected, dtype={raw_names[ID]: str},
                             chunksize=CSV_CHUNKSIZE, low_memory=False):
        yield normalize_frame(chunk, path.name)[columns].copy()


def fail_with_rows(df, out, filename, message):
    path = out / filename
    df.to_csv(path, index=False)
    raise ValueError(f'{message}\nĐã lưu danh sách kiểm tra: {path}')


def save_json(data, path):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


def file_signature(path, with_hash=False):
    stat = Path(path).stat()
    info = {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
    if with_hash:
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
                digest.update(block)
        info['sha256'] = digest.hexdigest()
    return info


def validate_binary(series, name):
    values = pd.to_numeric(series, errors='coerce')
    if values.isna().any() or not values.isin([0, 1]).all():
        raise ValueError(f'{name}: nhãn phải là 0 hoặc 1, không thiếu.')
    return values.astype(np.int8)



def parse_dates(df, column, out, source):
    raw = df[column]
    text = clean_category(raw)
    parts = text.str.extract(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)')
    first = pd.to_numeric(parts[0], errors='coerce')
    second = pd.to_numeric(parts[1], errors='coerce')
    ambiguous = (first.between(1, 12) & second.between(1, 12) & first.ne(second)).fillna(False)
    dayfirst = SOURCE_DATE_DAYFIRST.get(source, DATE_DAYFIRST)
    if dayfirst is None and ambiguous.any():
        fail_with_rows(df.loc[ambiguous, [ID, column]], out,
                       f'CHECK_{source}_{column}_ambiguous_dates.csv',
                       'Có ngày dd/mm/yyyy hoặc mm/dd/yyyy chưa xác định định dạng. '
                       'Kiểm tra nguồn trước khi đặt DATE_DAYFIRST=True/False.')
    if source in SOURCE_DATE_DAYFIRST:
        # format='mixed' có thể tự đổi sang dd/mm nếu gặp 22/1/2019.
        # Dừng khi nguồn có dấu hiệu trái quy ước đã xác nhận, không tự đảo ngày.
        short_layout = parts[0].notna()
        left_max, right_max = (31, 12) if dayfirst else (12, 31)
        inconsistent = short_layout & (
            ~first.between(1, left_max) | ~second.between(1, right_max))
        if inconsistent.any():
            fail_with_rows(df.loc[inconsistent, [ID, column]], out,
                           f'CHECK_{source}_{column}_date_layout_conflict.csv',
                           'Nguồn có ngày trái định dạng đã xác nhận. '
                           'Cần kiểm tra nguồn; không tự đảo tháng và ngày.')
    # Giữ nguyên thời điểm; activity/deposit đọc tháng/ngày theo nguồn đã xác nhận.
    parsed = pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=bool(dayfirst))
    invalid = text.notna() & parsed.isna()
    if invalid.any():
        fail_with_rows(df.loc[invalid, [ID, column]], out,
                       f'CHECK_{source}_{column}_invalid_dates.csv', 'Có giá trị ngày chưa đọc được.')
    if not pd.api.types.is_datetime64_any_dtype(parsed) or parsed.dt.tz is not None:
        raise ValueError(f'{source}/{column}: cần thống nhất kiểu ngày và timezone với dữ liệu nguồn.')
    return parsed


def parse_numbers(df, column, out, source):
    text = clean_category(df[column])
    numbers = pd.to_numeric(text.str.replace(',', '', regex=False), errors='coerce')
    invalid = text.notna() & numbers.isna()
    if invalid.any():
        fail_with_rows(df.loc[invalid, [ID, column]], out,
                       f'CHECK_{source}_{column}_invalid_numbers.csv', 'Có giá trị chưa đọc được thành số.')
    nonfinite = numbers.notna() & ~np.isfinite(numbers.to_numpy(dtype=float, na_value=np.nan))
    if nonfinite.any():
        fail_with_rows(df.loc[nonfinite, [ID, column]], out,
                       f'CHECK_{source}_{column}_nonfinite.csv', 'Có giá trị số vô hạn.')
    return numbers


def self_check_time_rules():
    r = pd.Timestamp('2019-01-01')
    mark, end = r + pd.Timedelta(days=60), r + pd.Timedelta(days=240)
    events = pd.Series([mark, mark + pd.Timedelta(nanoseconds=1), end,
                        end + pd.Timedelta(nanoseconds=1), pd.NaT])
    y = (events.notna() & events.gt(mark) & events.le(end)).astype(int)
    assert y.tolist() == [0, 1, 1, 0, 0]
    print('PASS: nhãn mở bên ngày 60, đóng bên ngày 240.')


def discover_date_sources():
    candidates = set()
    for pattern in ['target_landmark_60d*.csv', '*master*60d*.csv']:
        candidates.update(p for p in ROOT.rglob(pattern) if p.is_file())
    candidates.update(Path(p) for p in EXTRA_DATE_SOURCE_FILES if Path(p).is_file())
    candidates.discard(FILES['A_60D'])
    return sorted(candidates, key=lambda p: str(p))


def rebuild_relationship_master(a, customer, out):
    """Ghép ngày đã lưu; không lấy ngày giao dịch/snapshot đầu làm ngày tạo.

    Nguồn bổ sung chỉ được dùng sau khi đối chiếu nhãn A60 và r+60.
    Không suy ngày tạo bằng cách trừ cutoff, không sửa ID theo phỏng đoán.
    """
    require_columns(customer, [ID, 'CLIENT_CREATE_DATE'], 'customer')
    customer = customer.loc[customer[ID].isin(a[ID])].copy()
    customer['CLIENT_CREATE_DATE'] = parse_dates(customer, 'CLIENT_CREATE_DATE', out, 'customer')
    raw_stats = customer.groupby(ID, as_index=False).agg(
        RAW_CUSTOMER_ROWS=(ID, 'size'), RAW_VALID_DATES=('CLIENT_CREATE_DATE', 'count'),
        RAW_DISTINCT_DATES=('CLIENT_CREATE_DATE', 'nunique'),
        RAW_START=('CLIENT_CREATE_DATE', 'min'))
    raw_stats.to_csv(out / 'raw_customer_date_audit.csv', index=False)
    # Cùng quy tắc mã gốc: ngày tạo sớm nhất nếu customer có nhiều dòng.
    evidence = raw_stats.loc[raw_stats['RAW_START'].notna(), [ID, 'RAW_START']].rename(
        columns={'RAW_START': START})
    evidence['DATE_SOURCE'] = str(FILES['customer'])
    evidence['SOURCE_TYPE'] = 'RAW_CUSTOMER_EARLIEST_CREATE_DATE'
    evidence_parts = [evidence]
    source_audits = []
    source_signatures = []

    def register_saved(frame, name, direct_a=False):
        frame = frame.loc[frame[ID].isin(a[ID])].copy()
        date_column = START if START in frame.columns else (
            'CLIENT_CREATE_DATE' if 'CLIENT_CREATE_DATE' in frame.columns else None)
        if date_column is None:
            source_audits.append({'source': name, 'status': 'NO_DIRECT_CREATE_DATE', 'rows': len(frame)})
            return
        if frame.empty:
            source_audits.append({'source': name, 'status': 'NO_MATCHING_A_IDS', 'rows': 0})
            return
        frame[date_column] = parse_dates(frame, date_column, out, 'saved_master')
        relevant = frame.loc[frame[date_column].notna()].copy()
        # Mỗi nguồn đã xử lý phải có cùng một ngày tạo cho một ID.
        conflicts = relevant.groupby(ID)[date_column].nunique()
        conflicts = conflicts[conflicts > 1].index
        if len(conflicts):
            fail_with_rows(relevant.loc[relevant[ID].isin(conflicts)], out,
                           'CHECK_saved_date_source_duplicate_conflict.csv',
                           f'Nguồn {name}: một ID có các ngày tạo khác nhau.')
        cutoff_col = CUTOFF if CUTOFF in frame.columns else (
            'LANDMARK_DATE' if 'LANDMARK_DATE' in frame.columns else None)
        if not direct_a and (Y_OLD not in frame.columns or cutoff_col is None):
            source_audits.append({'source': name, 'status': 'UNVERIFIED_LABEL_OR_60D_CUTOFF',
                                  'rows': len(relevant)})
            return
        if cutoff_col is not None:
            relevant[cutoff_col] = parse_dates(relevant, cutoff_col, out, 'saved_master')
            right_cutoff = relevant[cutoff_col].eq(
                relevant[date_column] + pd.Timedelta(days=LANDMARK_DAYS))
        else:
            right_cutoff = pd.Series(True, index=relevant.index)
        if Y_OLD in relevant.columns:
            relevant['_SOURCE_Y'] = pd.to_numeric(relevant[Y_OLD], errors='coerce')
            original = a.set_index(ID)[Y_OLD]
            right_label = relevant['_SOURCE_Y'].eq(relevant[ID].map(original))
        else:
            right_label = pd.Series(False, index=relevant.index)
        verified = right_cutoff & right_label
        if not verified.all():
            # Không tự dùng nguồn thuộc một cohort/nhãn/cutoff khác.
            source_audits.append({'source': name, 'status': 'UNVERIFIED_60D_CONTEXT',
                                  'rows': len(relevant), 'invalid_context_rows': int((~verified).sum())})
            relevant.loc[~verified].to_csv(
                out / f'CHECK_unverified_date_source_{len(source_audits):02d}.csv', index=False)
            return
        accepted = relevant[[ID, date_column]].drop_duplicates().rename(columns={date_column: START})
        accepted['DATE_SOURCE'] = name
        accepted['SOURCE_TYPE'] = 'VERIFIED_SAVED_60D_DATE'
        evidence_parts.append(accepted)
        source_audits.append({'source': name, 'status': 'VERIFIED', 'rows': len(accepted)})

    # Nếu A đã lưu trực tiếp ngày tạo, dùng đúng giá trị đã lưu, không suy từ cutoff.
    register_saved(a, str(FILES['A_60D']), direct_a=True)
    for path in discover_date_sources():
        print('ĐỌC LẠI NGUỒN NGÀY 60D:', path)
        source_signatures.append(file_signature(path, with_hash=True))
        saved = read_table(path)
        register_saved(saved, str(path))
    pd.DataFrame(source_audits).to_csv(out / 'saved_date_source_validation.csv', index=False)
    save_json(source_signatures, out / 'saved_date_source_signatures.json')
    all_evidence = pd.concat(evidence_parts, ignore_index=True)
    all_evidence.to_csv(out / 'relationship_date_evidence.csv', index=False)
    conflicts = all_evidence.groupby(ID)[START].nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        fail_with_rows(all_evidence.loc[all_evidence[ID].isin(conflicts)], out,
                       'CHECK_relationship_date_source_conflict.csv',
                       'Ngày tạo khác nhau giữa nguồn customer và các nguồn 60D đã xác minh. '
                       'Cần chọn đúng nguồn của lần chạy; mã không tự chọn một ngày.')
    resolved = all_evidence.groupby(ID, as_index=False).agg(
        RELATIONSHIP_START_DATE=(START, 'first'),
        DATE_SOURCES=('DATE_SOURCE', lambda x: ' | '.join(sorted(set(x)))),
        DATE_SOURCE_TYPES=('SOURCE_TYPE', lambda x: ' | '.join(sorted(set(x)))))
    master = a[[ID, Y_OLD]].merge(resolved, on=ID, how='left', validate='one_to_one').merge(
        raw_stats, on=ID, how='left', validate='one_to_one')
    master['DATE_FOUND_IN_RAW_CUSTOMER'] = master['RAW_START'].notna()
    master['DATE_READ_FROM_VERIFIED_SAVED_SOURCE'] = master[START].notna() & master['RAW_START'].isna()
    summary = pd.DataFrame([{
        'n_original_FLDC': len(a),
        'n_date_in_raw_customer': int(master['DATE_FOUND_IN_RAW_CUSTOMER'].sum()),
        'n_date_rejoined_from_verified_saved_source': int(master['DATE_READ_FROM_VERIFIED_SAVED_SOURCE'].sum()),
        'n_still_missing_date': int(master[START].isna().sum())}])
    master.to_csv(out / 'relationship_master_rejoined.csv', index=False)
    summary.to_csv(out / 'relationship_master_join_summary.csv', index=False)
    show_table(summary, 'GHÉP LẠI NGÀY TẠO TỪ CÁC NGUỒN ĐÃ ĐỌC')
    if master[START].isna().any():
        missing = master.loc[master[START].isna()].copy()
        missing['WHY'] = np.where(missing['RAW_CUSTOMER_ROWS'].isna(),
                                  'ID_NOT_FOUND_IN_RAW_CUSTOMER', 'RAW_CREATE_DATE_EMPTY')
        fail_with_rows(missing, out, 'CHECK_missing_relationship_date_after_all_joins.csv',
                       'Đã đọc lại customer, A và các master 60D tìm được nhưng vẫn thiếu ngày tạo. '
                       'Các nguồn không cung cấp đủ ngày thật cho những ID này. '
                       'Không suy từ giao dịch/thẻ, không tự loại khách.')
    return master


def rebuild_dynamic(master, out, branch):
    out.mkdir(parents=True, exist_ok=True)
    combined, audits = master[[ID]].copy(), []
    builders = [
        ('transaction', lambda: build_transactions(FILES['transaction'], master, out)),
        ('activity', lambda: build_activity(FILES['activity'], master, out)),
        ('deposit', lambda: build_snapshot(FILES['deposit'], master, out, 'deposit', DEPOSIT_FEATURES)),
        ('lending', lambda: build_snapshot(FILES['lending'], master, out, 'lending',
                                         ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT']))]
    for name, builder in builders:
        print(f'\nĐỌC LẠI + TẠO ĐẶC TRƯNG {branch}: {name}')
        frame, audit = builder()
        combined = combined.merge(frame, on=ID, how='left', validate='one_to_one')
        audit['branch'] = branch
        audits.append(audit)
        print(f"Dùng {audit['rows_in_window']:,} dòng; vi phạm timestamp={audit['timestamp_violations']}.")
    require_columns(combined, [ID] + DYNAMIC_FEATURES, branch)
    combined.to_csv(out / 'all_dynamic_features.csv', index=False)
    return combined, pd.DataFrame(audits)


def verify_reconstructed_A(saved, rebuilt, numeric_columns, out):
    rebuilt = rebuilt.set_index(ID).loc[saved[ID].tolist()].reset_index()
    detail, summary = [], []
    for col in DYNAMIC_FEATURES:
        if col in numeric_columns:
            original = pd.to_numeric(saved[col], errors='coerce').to_numpy(dtype=float)
            recomputed = pd.to_numeric(rebuilt[col], errors='coerce').to_numpy(dtype=float)
            matched = np.isclose(original, recomputed, rtol=A_AUDIT_RTOL, atol=A_AUDIT_ATOL, equal_nan=True)
        else:
            matched = saved[col].astype('string').fillna('<MISSING>').eq(
                rebuilt[col].astype('string').fillna('<MISSING>')).to_numpy(dtype=bool)
        summary.append({'feature': col, 'n_checked': len(saved), 'n_mismatch': int((~matched).sum())})
        if not matched.all():
            detail.append(pd.DataFrame({ID: saved.loc[~matched, ID].to_numpy(), 'feature': col,
                                       'A_saved': saved.loc[~matched, col].to_numpy(),
                                       'A_rebuilt': rebuilt.loc[~matched, col].to_numpy()}))
    table = pd.DataFrame(summary)
    table.to_csv(out / 'A60_reconstruction_comparison.csv', index=False)
    show_table(table, 'ĐỐI CHIẾU A60 ĐÃ LƯU VỚI 13 BIẾN TÁI TẠO TỪ NGUỒN')
    if detail:
        fail_with_rows(pd.concat(detail, ignore_index=True), out, 'CHECK_A60_reconstruction_mismatch.csv',
                       'A60 tái tạo chưa khớp A đã lưu. Kiểm tra phiên bản nguồn/công thức. '
                       'Chưa dùng chênh lệch AUC để kết luận về cửa sổ; không thay A gốc.')


# %% 3. COHORT VÀ NHÃN H=180
def build_cohort(a, customer, card, out):
    """Giữ FLDC gốc; xét (r+60, r+60+H], không loại sự kiện ngày 61..180."""
    require_columns(a, [ID, Y_OLD] + FEATURES, 'A 60 ngày')
    if a[ID].duplicated().any():
        fail_with_rows(a[a[ID].duplicated(False)], out, 'CHECK_duplicate_A_ID.csv', 'A có ID trùng.')
    a = a.copy()
    a[Y_OLD] = validate_binary(a[Y_OLD], Y_OLD)
    if len(a) != EXPECTED_BASE_CUSTOMERS or int(a[Y_OLD].sum()) != EXPECTED_BASE_POSITIVES:
        raise ValueError(f'Bộ A đang đọc có n={len(a):,}, positives={int(a[Y_OLD].sum()):,}; '
                         f'khác bộ FLDC đã xác nhận ({EXPECTED_BASE_CUSTOMERS:,}; '
                         f'{EXPECTED_BASE_POSITIVES:,}). Kiểm tra đúng phiên bản tệp trước khi chạy tiếp.')

    master = rebuild_relationship_master(a, customer, out)

    require_columns(card, [ID, 'MONTH', 'COUNT_CREDITCARD'], 'card')
    card = card[card[ID].isin(a[ID])][[ID, 'MONTH', 'COUNT_CREDITCARD']].copy()
    card['MONTH'] = parse_dates(card, 'MONTH', out, 'card')
    card['COUNT_CREDITCARD'] = parse_numbers(card, 'COUNT_CREDITCARD', out, 'card')
    invalid = card['MONTH'].isna() | card['COUNT_CREDITCARD'].isna() | (card['COUNT_CREDITCARD'] < 0)
    if invalid.any():
        fail_with_rows(card[invalid], out, 'CHECK_invalid_card_rows.csv',
                       'Có dòng dữ liệu thẻ chưa xác định được ngày hoặc trạng thái. Cần kiểm tra; không tự gán 0.')
    # Giữ nguyên ngày MONTH được lưu trong tệp, theo quy ước của bộ 60 ngày.
    card = card[card['MONTH'] <= OBSERVATION_END].copy()
    observed = card.groupby(ID, as_index=False).agg(
        FIRST_OBS_MONTH=('MONTH', 'min'), LAST_OBS_MONTH=('MONTH', 'max'))
    first_card = card.loc[card['COUNT_CREDITCARD'] > 0].groupby(ID, as_index=False)['MONTH'].min().rename(
        columns={'MONTH': EVENT})
    master = master.merge(observed, on=ID, how='left', validate='one_to_one').merge(
        first_card, on=ID, how='left', validate='one_to_one')
    master['LANDMARK_DATE'] = master[START] + pd.to_timedelta(LANDMARK_DAYS, unit='D')
    master['HORIZON_END_DATE'] = master['LANDMARK_DATE'] + pd.to_timedelta(HORIZON_DAYS, unit='D')
    master['TARGET_60D_FROM_CARD'] = (
        master[EVENT].notna() & (master[EVENT] > master['LANDMARK_DATE'])).astype(np.int8)
    invalid_base = (
        master['LAST_OBS_MONTH'].isna()
        | (master['LAST_OBS_MONTH'] < master['LANDMARK_DATE'])
        | (master[EVENT].notna() & (master[EVENT] <= master['LANDMARK_DATE']))
        | (master[Y_OLD] != master['TARGET_60D_FROM_CARD'])
    )
    if invalid_base.any():
        fail_with_rows(master[invalid_base], out, 'CHECK_original_cohort_or_label_mismatch.csv',
                       'Ngày/sự kiện thẻ không khớp cohort hoặc nhãn 60 ngày gốc. '
                       'Kiểm tra phiên bản dữ liệu và quy ước MONTH; không tự sửa nhãn.')

    master['ADMIN_ELIGIBLE_H'] = master['HORIZON_END_DATE'] <= OBSERVATION_END
    master[Y] = (master[EVENT].notna() & (master[EVENT] > master['LANDMARK_DATE'])
                 & (master[EVENT] <= master['HORIZON_END_DATE'])).astype(np.int8)
    master['HAS_SNAPSHOT_AT_OR_AFTER_H'] = master['LAST_OBS_MONTH'] >= master['HORIZON_END_DATE']
    master['EVENT_AFTER_H'] = master[EVENT].notna() & (master[EVENT] > master['HORIZON_END_DATE'])
    master.to_csv(out / 'original_FLDC_horizon_eligibility_audit.csv', index=False)
    cohort = master.loc[master['ADMIN_ELIGIBLE_H']].sort_values(ID, kind='stable').reset_index(drop=True)
    if len(cohort) == 0 or cohort[Y].nunique() != 2:
        raise ValueError('Cohort H hiện tại rỗng hoặc chỉ có một lớp; cần rà soát H và dữ liệu.')
    cohort['CUTOFF_A'] = cohort['LANDMARK_DATE']
    cohort['CUTOFF_B'] = cohort['HORIZON_END_DATE'].where(cohort[Y].eq(0), cohort[EVENT])
    cohort['B_WINDOW_DAYS'] = (cohort['CUTOFF_B'] - cohort[START]).dt.total_seconds() / 86400
    if not (cohort['CUTOFF_B'] > cohort['CUTOFF_A']).all():
        raise AssertionError('Cutoff B phải sau ngày 60 trong cohort này.')
    cohort.to_csv(out / 'cohort_fixed_horizon_master.csv', index=False)
    negatives = cohort[Y].eq(0)
    missing_followup = negatives & ~cohort['HAS_SNAPSHOT_AT_OR_AFTER_H'] & ~cohort['EVENT_AFTER_H']
    # Đây là audit, không tự bổ sung tiêu chí loại khách khác với kế hoạch đã nêu.
    summary = pd.DataFrame([{
        'H_days': HORIZON_DAYS, 'landmark_days': LANDMARK_DAYS,
        'base_n': len(a), 'base_positives': int(a[Y_OLD].sum()),
        'n': len(cohort), 'events': int(cohort[Y].sum()),
        'prevalence': float(cohort[Y].mean()),
        'excluded_administrative_end': int((~master['ADMIN_ELIGIBLE_H']).sum()),
        'event_after_H_now_label_0': int(cohort['EVENT_AFTER_H'].sum()),
        'negatives_without_snapshot_at_or_after_H': int((negatives & ~cohort['HAS_SNAPSHOT_AT_OR_AFTER_H']).sum()),
        'negatives_without_endpoint_snapshot_or_later_event': int(missing_followup.sum()),
    }])
    summary.to_csv(out / 'cohort_summary.csv', index=False)
    show_table(summary, 'COHORT VÀ NHÃN THEO HORIZON CỐ ĐỊNH')
    if int(cohort[Y].sum()) < MIN_EVENTS_FOR_REVIEW:
        print(f'Có dưới {MIN_EVENTS_FOR_REVIEW} sự kiện. H vẫn là {HORIZON_DAYS}; '
              'chỉ cân nhắc H=120/90 sau khi xem số sự kiện, không chọn theo AUC.')
    if missing_followup.any():
        fail_with_rows(cohort.loc[missing_followup], out, 'CHECK_negative_followup_H.csv',
                       'Có khách nhãn 0 chưa có bằng chứng quan sát tới cuối H. '
                       'Cần xác minh nguồn theo dõi trước khi báo kết quả fixed follow-up; '
                       'mã không tự loại khách hoặc coi nhãn đó đã được xác nhận.')
    return cohort, summary


# %% 4. TẠO LẠI ĐẶC TRƯNG B TỪ DỮ LIỆU GỐC
def rows_in_window(raw, master, date_col, stats):
    raw = raw.loc[raw[ID].isin(master[ID])].copy()
    stats['rows_for_cohort'] += len(raw)
    raw[date_col] = parse_dates(raw, date_col, Path(stats['_out']), stats['source'])
    stats['invalid_date_rows'] += int(raw[date_col].isna().sum())
    joined = raw.merge(master[[ID, START, CUTOFF]], on=ID, how='inner', validate='many_to_one')
    before = joined[date_col].notna() & (joined[date_col] < joined[START])
    after = joined[date_col].notna() & (joined[date_col] >= joined[CUTOFF])
    stats['rows_before_relationship'] += int(before.sum())
    stats['rows_at_or_after_cutoff'] += int(after.sum())
    valid = joined[date_col].notna() & ~before & ~after
    selected = joined.loc[valid, raw.columns].copy()
    stats['rows_in_window'] += len(selected)
    if not selected.empty:
        bounds = selected.groupby(ID)[date_col].agg(['min', 'max', 'size'])
        for customer, row in bounds.iterrows():
            old = stats['_bounds'].get(customer)
            stats['_bounds'][customer] = (
                row['min'] if old is None else min(old[0], row['min']),
                row['max'] if old is None else max(old[1], row['max']),
                int(row['size']) + (0 if old is None else old[2]),
            )
    return selected


def new_stats(source):
    return {'source': source, 'rows_for_cohort': 0, 'invalid_date_rows': 0,
            'rows_before_relationship': 0, 'rows_at_or_after_cutoff': 0,
            'rows_in_window': 0, '_bounds': {}}


def finish_audit(stats, master, out):
    bounds = stats.pop('_bounds')
    stats.pop('_out', None)
    b = pd.DataFrame([(k, *v) for k, v in bounds.items()],
                     columns=[ID, 'MIN_SOURCE_TIME', 'MAX_SOURCE_TIME', 'N_SOURCE_ROWS'])
    audit = master[[ID, START, CUTOFF]].merge(b, on=ID, how='left', validate='one_to_one')
    for col in ['MIN_SOURCE_TIME', 'MAX_SOURCE_TIME']:
        audit[col] = pd.to_datetime(audit[col])
    audit['N_SOURCE_ROWS'] = pd.to_numeric(audit['N_SOURCE_ROWS'], errors='raise').fillna(0).astype(int)
    audit['VIOLATION'] = ((audit['MIN_SOURCE_TIME'] < audit[START])
                          | (audit['MAX_SOURCE_TIME'] >= audit[CUTOFF]))
    audit.to_csv(out / f"{stats['source']}_timestamp_audit.csv", index=False)
    if audit['VIOLATION'].any():
        raise AssertionError(f"{stats['source']}: còn dữ liệu ngoài cutoff.")
    stats['customers_with_source_rows'] = len(bounds)
    stats['customers_without_source_rows'] = len(master) - len(bounds)
    stats['timestamp_violations'] = int(audit['VIOLATION'].sum())
    return stats


def reduce_mode(parts, name):
    if not parts:
        return pd.Series(dtype=object, name=name)
    counts = pd.concat(parts).groupby([ID, 'CATEGORY'], as_index=False)['N'].sum()
    # Đồng hạng: thứ tự chữ cái như get_mode_value trong mã 180 ngày của tác giả.
    winner = counts.sort_values([ID, 'N', 'CATEGORY'], ascending=[True, False, True],
                                kind='stable').drop_duplicates(ID)
    return winner.set_index(ID)['CATEGORY'].rename(name)


def build_transactions(path, master, out):
    stats = new_stats('transaction')
    stats['_out'] = str(out)
    nums, modes1, modes2 = [], [], []
    columns = [ID, 'TRANS_DATE', 'TRANS_LV1', 'TRANS_LV2', 'TRANS_AMOUNT']
    for chunk in iter_source(path, columns):
        d = rows_in_window(chunk, master, 'TRANS_DATE', stats)
        if d.empty:
            continue
        d['TRANS_AMOUNT'] = parse_numbers(d, 'TRANS_AMOUNT', out, 'transaction')
        n = d.groupby(ID)['TRANS_AMOUNT'].agg(['sum', 'count', 'max']).rename(
            columns={'sum': 'AMOUNT_SUM', 'count': 'AMOUNT_N', 'max': 'TRANS_AMOUNT_MAX'})
        nums.append(n)
        for column, container in [('TRANS_LV1', modes1), ('TRANS_LV2', modes2)]:
            cat = clean_category(d[column])
            pairs = pd.DataFrame({ID: d[ID], 'CATEGORY': cat}).dropna(subset=['CATEGORY'])
            container.append(pairs.groupby([ID, 'CATEGORY']).size().rename('N').reset_index())
    base = master[[ID]].set_index(ID)
    if nums:
        n = pd.concat(nums).groupby(level=0).agg(
            AMOUNT_SUM=('AMOUNT_SUM', 'sum'), AMOUNT_N=('AMOUNT_N', 'sum'),
            TRANS_AMOUNT_MAX=('TRANS_AMOUNT_MAX', 'max'))
        n['TRANS_AMOUNT_MEAN'] = n['AMOUNT_SUM'] / n['AMOUNT_N'].replace(0, np.nan)
        base = base.join(n[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']])
    else:
        base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']] = np.nan
    base = base.join(reduce_mode(modes1, 'TRANS_LV1_MODE')).join(reduce_mode(modes2, 'TRANS_LV2_MODE'))
    base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']] = base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']].fillna('NO_TRANSACTION')
    base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']] = base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']].fillna(0)
    feature = base.reset_index()[[ID] + TRANSACTION_FEATURES]
    feature.to_csv(out / 'transaction_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_activity(path, master, out):
    stats, counts, days = new_stats('activity'), [], []
    stats['_out'] = str(out)
    for chunk in iter_source(path, [ID, 'ACTIVITY_DATE', 'ACTIVITY_NAME']):
        d = rows_in_window(chunk, master, 'ACTIVITY_DATE', stats)
        if d.empty:
            continue
        category = clean_category(d['ACTIVITY_NAME'])
        d['LOGIN_N'] = category.str.contains('LOGIN', na=False, regex=False).astype(int)
        d['INTEREST_N'] = category.str.contains('INTEREST_RATE', na=False, regex=False).astype(int)
        d['ACTIVITY_N'] = 1
        counts.append(d.groupby(ID)[['LOGIN_N', 'INTEREST_N', 'ACTIVITY_N']].sum())
        days.append(pd.DataFrame({ID: d[ID], 'DAY': d['ACTIVITY_DATE'].dt.normalize()}).drop_duplicates())
    base = master[[ID]].set_index(ID)
    if counts:
        c = pd.concat(counts).groupby(level=0).sum()
        active = pd.concat(days).drop_duplicates().groupby(ID).size()
        c['LOGIN_PER_ACTIVE_DAY'] = c['LOGIN_N'] / active
        c['INTEREST_RATE_RATIO'] = c['INTEREST_N'] / c['ACTIVITY_N']
        base = base.join(c[ACTIVITY_FEATURES])
    else:
        base[ACTIVITY_FEATURES] = np.nan
    feature = base.fillna(0).reset_index()[[ID] + ACTIVITY_FEATURES]
    feature.to_csv(out / 'activity_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_snapshot(path, master, out, source, numeric_columns):
    stats, parts = new_stats(source), []
    stats['_out'] = str(out)
    for chunk in iter_source(path, [ID, 'MONTH'] + numeric_columns):
        d = rows_in_window(chunk, master, 'MONTH', stats)
        if d.empty:
            continue
        for col in numeric_columns:
            d[col] = parse_numbers(d, col, out, source)
        parts.append(d[[ID, 'MONTH'] + numeric_columns])
    if parts:
        all_rows = pd.concat(parts, ignore_index=True)
        maximum = all_rows.groupby(ID)['MONTH'].transform('max')
        latest = all_rows.loc[all_rows['MONTH'].eq(maximum)].drop_duplicates()
        conflicts = latest[ID].duplicated(False)
        if conflicts.any():
            fail_with_rows(latest.loc[conflicts], out, f'CHECK_{source}_latest_snapshot_conflict.csv',
                           'Cùng ID/tháng cuối có các snapshot khác nhau; cần kiểm tra nguồn. '
                           'Mã không chọn một dòng theo thứ tự đọc.')
        latest = latest[[ID] + numeric_columns]
    else:
        latest = pd.DataFrame(columns=[ID] + numeric_columns)
    feature = master[[ID]].merge(latest, on=ID, how='left', validate='one_to_one')
    feature[numeric_columns] = feature[numeric_columns].astype(float).fillna(0)
    count_cols = ['COUNT_CA_ACCT', 'COUNT_TD_ACCT'] if source == 'deposit' else ['COUNT_OF_LOAN']
    invalid_count = feature[count_cols].lt(0).any(axis=1)
    if invalid_count.any():
        fail_with_rows(feature.loc[invalid_count], out, f'CHECK_{source}_negative_counts.csv',
                       'Số tài khoản/khoản vay âm; cần kiểm tra dữ liệu.')
    for col in count_cols:
        feature[col] = feature[col].round().astype(int)
    if source == 'lending':
        feature['TOTAL_LOAN_AMOUNT'] = feature['COUNT_OF_LOAN'] * feature['AVG_LOAN_AMOUNT']
    selected = DEPOSIT_FEATURES if source == 'deposit' else LENDING_FEATURES
    nonnegative = count_cols if source == 'deposit' else selected
    invalid = feature[nonnegative].lt(0).any(axis=1) | ~np.isfinite(
        feature[selected].to_numpy(dtype=float)).all(axis=1)
    if invalid.any():
        fail_with_rows(feature.loc[invalid], out, f'CHECK_{source}_numeric_values.csv',
                       'Có giá trị đếm/cho vay âm hoặc số không hữu hạn.')
    feature = feature[[ID] + selected]
    feature.to_csv(out / f'{source}_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_matrices(a, cohort, out):
    numeric_columns = a[FEATURES].select_dtypes(include=np.number).columns.tolist()
    saved_a = a.set_index(ID).loc[cohort[ID].tolist(), FEATURES].reset_index()
    master_a = cohort[[ID, START, 'CUTOFF_A']].rename(columns={'CUTOFF_A': CUTOFF})
    rebuilt_a, audit_a = rebuild_dynamic(master_a, out / 'A60_REBUILD_CHECK', 'A60_check')
    verify_reconstructed_A(saved_a, rebuilt_a, numeric_columns, out)
    master_b = cohort[[ID, START, 'CUTOFF_B']].rename(columns={'CUTOFF_B': CUTOFF})
    rebuilt_b, audit_b = rebuild_dynamic(master_b, out / 'B_FIXED_HORIZON', 'B')
    matrix_a = saved_a.copy()
    matrix_a.insert(1, Y, cohort[Y].to_numpy())
    matrix_b = cohort[[ID, Y]].merge(saved_a[[ID] + STATIC_FEATURES], on=ID, how='left',
                                    validate='one_to_one').merge(rebuilt_b, on=ID, how='left',
                                                               validate='one_to_one')
    matrix_b = matrix_b[[ID, Y] + FEATURES].reset_index(drop=True)
    matrix_a = matrix_a[[ID, Y] + FEATURES].reset_index(drop=True)
    for col in FEATURES:
        if col in numeric_columns:
            converted = pd.to_numeric(matrix_b[col], errors='coerce')
            invalid = matrix_b[col].notna() & converted.isna()
            if invalid.any():
                fail_with_rows(matrix_b.loc[invalid, [ID, col]], out, f'CHECK_B_dtype_{col}.csv',
                               f'{col}: kiểu B không khớp kiểu số của A.')
            matrix_b[col] = converted
        else:
            for matrix in [matrix_a, matrix_b]:
                s = matrix[col]
                matrix[col] = s.astype(str).where(s.notna(), np.nan)
    if not matrix_a[[ID, Y]].equals(matrix_b[[ID, Y]]):
        raise AssertionError('A/B khác ID, thứ tự hoặc nhãn.')
    for col in STATIC_FEATURES:
        if not matrix_a[col].equals(matrix_b[col]):
            raise AssertionError(f'Biến dùng lại {col} khác giữa A/B.')
    for branch, matrix in [('A', matrix_a), ('B', matrix_b)]:
        if np.isinf(matrix[numeric_columns].to_numpy(dtype=float)).any():
            raise ValueError(f'{branch}: có đặc trưng số vô hạn.')
        matrix.to_csv(out / f'{branch}_fixed_horizon_matrix.csv', index=False)
    all_audits = pd.concat([audit_a, audit_b], ignore_index=True)
    all_audits.to_csv(out / 'all_source_audits.csv', index=False)
    show_table(all_audits, 'AUDIT NGUỒN ĐẶC TRƯNG A/B ĐÃ GHÉP LẠI')
    return matrix_a, matrix_b, numeric_columns


# %% 5. MÔ HÌNH, METRICS VÀ BOOTSTRAP GHÉP CẶP
def make_preprocessor(numeric_columns):
    categorical = [c for c in FEATURES if c not in numeric_columns]
    onehot_args = {'handle_unknown': 'ignore'}
    if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
        onehot_args['sparse_output'] = False
    else:
        onehot_args['sparse'] = False
    transforms = []
    if numeric_columns:
        transforms.append(('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))]),
                           numeric_columns))
    if categorical:
        transforms.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
            ('onehot', OneHotEncoder(**onehot_args)),
        ]), categorical))
    return ColumnTransformer(transforms)


def metric_row(y, p, branch, n_train):
    auc = float(roc_auc_score(y, p))
    fpr, tpr, _ = roc_curve(y, p)
    brier = float(brier_score_loss(y, p))
    prevalence = float(np.mean(y))
    baseline = prevalence * (1 - prevalence)
    k = max(1, int(np.ceil(0.10 * len(y))))
    selected = np.argsort(-np.asarray(p), kind='stable')[:k]
    true_top = int(np.asarray(y)[selected].sum())
    return {'branch': branch, 'n_train': n_train, 'n_test': len(y), 'events_test': int(np.sum(y)),
            'AUC': auc, 'KS': float(np.max(tpr - fpr)), 'Brier': brier,
            'BSS': float(1 - brier / baseline), 'Precision_at_10': true_top / k,
            'Lift_at_10': (true_top / k) / prevalence, 'Capture_at_10': true_top / int(np.sum(y))}


def paired_bootstrap(y, p_a, p_b, n_bootstrap, seed):
    """Lấy lại khách holdout có hoàn lại; A/B dùng cùng index mỗi lần.

    Không fit lại mô hình trong bootstrap. Đây là CI holdout có điều kiện
    theo lần fit/split hiện tại, khác với thí nghiệm refit 20 split đã chạy.
    """
    y, p_a, p_b = map(np.asarray, [y, p_a, p_b])
    if len(y) != len(p_a) or len(y) != len(p_b) or np.unique(y).size != 2:
        raise ValueError('Bootstrap cần cùng holdout A/B và hai lớp.')
    rng, rows = np.random.default_rng(seed), []
    attempts, max_attempts = 0, max(10000, n_bootstrap * 20)
    while len(rows) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        index = rng.integers(0, len(y), size=len(y))
        yy = y[index]
        if np.unique(yy).size != 2:
            continue
        a = float(roc_auc_score(yy, p_a[index]))
        b = float(roc_auc_score(yy, p_b[index]))
        ba = float(brier_score_loss(yy, p_a[index]))
        bb = float(brier_score_loss(yy, p_b[index]))
        baseline = float(np.mean(yy) * (1 - np.mean(yy)))
        rows.append((len(rows) + 1, a, b, b - a, 1 - ba / baseline, 1 - bb / baseline))
        if len(rows) % 200 == 0 or len(rows) == n_bootstrap:
            print(f'Paired bootstrap: {len(rows)}/{n_bootstrap}')
    if len(rows) != n_bootstrap:
        raise RuntimeError(f'Chỉ có {len(rows)} bootstrap hợp lệ; chưa đủ {n_bootstrap}.')
    return pd.DataFrame(rows, columns=['replicate', 'AUC_A', 'AUC_B', 'Dwindow_H', 'BSS_A', 'BSS_B'])


def fit_and_report(a, b, numeric_columns, cohort_summary, out):
    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError('Chưa có xgboost. Trên Colab chạy: %pip install xgboost, rồi chạy lại.') from exc
    indexes = np.arange(len(a))
    y = a[Y].to_numpy(dtype=np.int8)
    train, test = train_test_split(indexes, test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED)
    split = a[[ID, Y]].copy()
    split['SPLIT'] = 'TRAIN'
    split.loc[test, 'SPLIT'] = 'TEST'
    split['ORDER_IN_SPLIT'] = -1
    split.loc[train, 'ORDER_IN_SPLIT'] = np.arange(len(train))
    split.loc[test, 'ORDER_IN_SPLIT'] = np.arange(len(test))
    split.to_csv(out / 'shared_H180_train_test_split.csv', index=False)
    print(f'\nTRAIN: {len(train):,}; TEST: {len(test):,}; '
          f'sự kiện train/test: {int(y[train].sum()):,}/{int(y[test].sum()):,}')
    probabilities, rows = {}, []
    for branch, matrix in [('A', a), ('B', b)]:
        print(f'\nFIT {branch}: 18 biến; XGBoost; không lấy mẫu lại')
        model = Pipeline([('preprocess', make_preprocessor(numeric_columns)),
                          ('xgb', XGBClassifier(**XGB_PARAMS))])
        model.fit(matrix.loc[train, FEATURES], y[train])
        p = model.predict_proba(matrix.loc[test, FEATURES])[:, 1]
        probabilities[branch] = p
        rows.append(metric_row(y[test], p, branch, len(train)))
        joblib.dump(model, out / f'model_{branch}_H180.joblib')
        print(f"{branch}: AUC={rows[-1]['AUC']:.6f}; BSS={rows[-1]['BSS']:.6f}; "
              f"KS={rows[-1]['KS']:.6f}")
    predictions = pd.DataFrame({ID: a.loc[test, ID].to_numpy(), Y: y[test],
                                'PROB_A': probabilities['A'], 'PROB_B': probabilities['B']})
    predictions.to_csv(out / 'paired_H180_holdout_predictions.csv', index=False)
    boot = paired_bootstrap(y[test], probabilities['A'], probabilities['B'],
                            n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED)
    boot.to_csv(out / 'paired_H180_bootstrap_values.csv', index=False)
    metrics = pd.DataFrame(rows)
    for index, branch in enumerate(['A', 'B']):
        ci = np.quantile(boot[f'AUC_{branch}'], [0.025, 0.975], method='linear')
        metrics.loc[index, ['AUC_ci_lower', 'AUC_ci_upper']] = ci
        ci_bss = np.quantile(boot[f'BSS_{branch}'], [0.025, 0.975], method='linear')
        metrics.loc[index, ['BSS_ci_lower', 'BSS_ci_upper']] = ci_bss
    metrics.to_csv(out / 'H180_holdout_metrics.csv', index=False)
    delta = float(rows[1]['AUC'] - rows[0]['AUC'])
    lo, hi = np.quantile(boot['Dwindow_H'], [0.025, 0.975], method='linear')
    paper = pd.DataFrame([{
        'H_days': HORIZON_DAYS, 'n': len(a), 'events': int(y.sum()),
        'n_train': len(train), 'n_test': len(test), 'events_test': int(y[test].sum()),
        'AUC_A': rows[0]['AUC'], 'BSS_A': rows[0]['BSS'], 'AUC_B': rows[1]['AUC'],
        'Dwindow_H': delta, 'Dwindow_H_ci_lower': float(lo), 'Dwindow_H_ci_upper': float(hi),
        'bootstrap_valid': len(boot), 'bootstrap_seed': BOOTSTRAP_SEED,
        'negatives_without_endpoint_snapshot_or_later_event': int(
            cohort_summary.iloc[0]['negatives_without_endpoint_snapshot_or_later_event']),
    }])
    paper.to_csv(out / 'Fixed_followup_horizon_paper_row.csv', index=False)
    # Bản văn chỉ được điền từ kết quả vừa tính trên dữ liệu thực của tác giả.
    paragraph = (
        f'With a {HORIZON_DAYS}-day follow-up horizon after the {LANDMARK_DAYS}-day landmark, '
        f'the administratively eligible cohort included {len(a):,} customers and {int(y.sum()):,} events. '
        f'Branch A used the original first-{LANDMARK_DAYS}-day features; branch B used pre-event features '
        'for adopters and features before the horizon endpoint for non-adopters. '
        'Both branches reused the same five customer attributes, including age, from branch A. '
        f'On a shared stratified 80:20 split (seed {SPLIT_SEED}), '
        f'AUC_A was {rows[0]["AUC"]:.4f}, BSS_A was {rows[0]["BSS"]:.4f}, '
        f'and AUC_B was {rows[1]["AUC"]:.4f}. '
        f'The paired difference D_window^H was {delta:.4f} '
        f'(95% paired customer-bootstrap interval {lo:.4f} to {hi:.4f}; {len(boot):,} replicates). '
        'The bootstrap interval is conditional on these fitted models and this split.'
    )
    missing = int(paper.iloc[0]['negatives_without_endpoint_snapshot_or_later_event'])
    if missing:
        paragraph += (f' For {missing:,} negative-labelled customers, endpoint follow-up was not '
                      'confirmed by a snapshot at or after the horizon or a later observed card event; '
                      'administrative eligibility alone does not resolve this limitation.')
    (out / 'Fixed_followup_horizon_paragraph_EN.txt').write_text(paragraph + '\n', encoding='utf-8')
    show_table(metrics.round(6), 'KẾT QUẢ A/B TRÊN CÙNG HOLDOUT')
    show_table(paper.round(6), 'DÒNG KẾT QUẢ ĐỂ ĐIỀN VÀO BÀI BÁO')
    print('\nĐOẠN TIẾNG ANH — kiểm tra trước khi đưa vào bài:\n' + paragraph)
    print('\nĐÃ LƯU KẾT QUẢ TẠI:\n' + str(out))
    return paper


# %% 6. CHẠY TOÀN BỘ; KẾT QUẢ HIỂN THỊ NGAY DƯỚI Ô NÀY
def run_experiment():
    """Không sửa các tệp nguồn, không ghi đè kết quả A/B/C hoặc refit 20 seed."""
    if str(ROOT).startswith('/content/'):
        try:
            from google.colab import drive
        except ImportError:
            pass
        else:
            if not (Path('/content/drive') / 'MyDrive').is_dir():
                drive.mount('/content/drive', force_remount=False)
    for name, path in FILES.items():
        if not path.is_file():
            raise FileNotFoundError(f'Thiếu nguồn {name}: {path}')
    self_check_time_rules()
    run_id = datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    out = OUTPUT_PARENT / run_id
    out.mkdir(parents=True, exist_ok=False)
    packages = {}
    for name in ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'joblib', 'openpyxl']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = 'not installed'
    manifest = {
        'status': 'STARTED', 'run_id': run_id, 'created_utc': datetime.now(timezone.utc).isoformat(),
        'code_revision': CODE_REVISION,
        'ID_format': 'digit-only ID: remove verified zero padding on string; never convert via float',
        'ID_format_evidence': 'author diagnostic: four CSV ID columns int64 under original reading; '
                              'leading-zero raw ID examples; no inference collisions in all four files',
        'python': platform.python_version(), 'packages': packages,
        'landmark_days': LANDMARK_DAYS, 'H_days': HORIZON_DAYS,
        'observation_end': str(OBSERVATION_END),
        'date_reading_dayfirst': DATE_DAYFIRST,
        'source_date_reading_dayfirst': SOURCE_DATE_DAYFIRST,
        'date_rejoin': 'raw customer + verified direct dates in A/saved 60D masters; stop on conflict',
        'A_dynamic_audit_tolerance': {'rtol': A_AUDIT_RTOL, 'atol': A_AUDIT_ATOL},
        'A_dynamic_audit': 'rebuild 13 first-60-day features with the same builders used for B; compare to saved A',
        'event_proxy': 'min original MONTH where COUNT_CREDITCARD > 0; MONTH is not rounded',
        'cohort': 'existing FLDC A IDs, then r+60+H <= administrative observation end',
        'label': '1 iff r+60 < FIRST_CARD_MONTH <= r+60+H',
        'feature_filter': 'RELATIONSHIP_START_DATE <= source time < FEATURE_CUTOFF_DATE',
        'A': 'reuse all original 18 A60 features; change label and cohort only',
        'B': 'reuse five A60 customer attributes; rebuild 13 behavioural features',
        'age': 'reuse AGE from A60 for both branches; do not recalculate at B cutoff',
        'negative_B_cutoff': 'r+60+H', 'positive_B_cutoff': 'FIRST_CARD_MONTH',
        'features': FEATURES, 'static_features_reused': STATIC_FEATURES,
        'split_seed': SPLIT_SEED, 'test_size': TEST_SIZE,
        'split': 'new shared stratified split using TARGET_H; not the original E2 split',
        'xgb_params': XGB_PARAMS, 'resampling': 'none', 'cv': 'not run in this horizon experiment',
        'preprocessing': 'train-only constant imputation and one-hot, as in current A/B code',
        'bootstrap': 'paired nonparametric customer bootstrap on holdout; percentile 95%; fixed fits',
        'bootstrap_seed': BOOTSTRAP_SEED, 'bootstrap_replicates': N_BOOTSTRAP,
        'BSS_definition': '1 - Brier / (p_test*(1-p_test)), matching the supplied A/B code',
        'sources': {name: file_signature(path, with_hash=name in ['A_60D', 'customer', 'card'])
                    for name, path in FILES.items()},
    }
    save_json(manifest, out / 'run_manifest.json')
    print(f'H={HORIZON_DAYS}; landmark={LANDMARK_DAYS}; mốc cuối={LANDMARK_DAYS + HORIZON_DAYS} ngày')
    print('PHIÊN BẢN CODE:', CODE_REVISION)
    print('ĐÃ SỬA: ID padding của CSV; activity/deposit dùng tháng/ngày/năm đã xác nhận.')
    print('DÙNG LẠI: năm biến khách hàng từ A60, bao gồm AGE. A giữ đủ 18 biến gốc.')
    print('THƯ MỤC CHẠY MỚI:\n' + str(out))
    started = time.monotonic()
    try:
        a = read_table(FILES['A_60D'])
        customer = read_table(FILES['customer'])
        card = read_table(FILES['card'], [ID, 'MONTH', 'COUNT_CREDITCARD'])
        cohort, summary = build_cohort(a, customer, card, out)
        matrix_a, matrix_b, numeric = build_matrices(a, cohort, out)
        result = fit_and_report(matrix_a, matrix_b, numeric, summary, out)
        manifest['status'] = 'COMPLETED'
        manifest['result'] = result.iloc[0].to_dict()
        return result
    except Exception as exc:
        manifest['status'] = 'FAILED'
        manifest['error'] = str(exc)
        print('\nCHƯA HOÀN TẤT. Kiểm tra lỗi và tệp CHECK nếu có. Không dùng kết quả dở dang để điền bài.')
        raise
    finally:
        manifest['elapsed_seconds'] = time.monotonic() - started
        save_json(manifest, out / 'run_manifest.json')




# ==================================================================================================
# NEW EXPERIMENT ADDED ON TOP OF THE VERIFIED VIB H180 V3 CODE
# A / B / B-SYM — XGBOOST CANONICAL ONLY — 1,000 PAIRED BOOTSTRAPS
# ==================================================================================================

BSYM_SEED = 42
OUTPUT_PARENT_BSYM = ROOT / 'VIB_FIXED_HORIZON_H180_A60_B240_BSYM'
CODE_REVISION_BSYM = 'VIB_H180_V3_PLUS_BSYM_20261013'

REFERENCE_H180 = {
    'AUC_A': 0.891002,
    'AUC_B': 0.906161,
    'Dwindow_H': 0.015158,
}


def make_shared_h180_split(matrix_a, out):
    """Exactly the same shared stratified 80:20 split definition used by the H180 V3 code."""
    indexes = np.arange(len(matrix_a))
    y = matrix_a[Y].to_numpy(dtype=np.int8)
    train, test = train_test_split(
        indexes,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=SPLIT_SEED,
    )
    split = matrix_a[[ID, Y]].copy()
    split['SPLIT'] = 'TRAIN'
    split.loc[test, 'SPLIT'] = 'TEST'
    split['ORDER_IN_SPLIT'] = -1
    split.loc[train, 'ORDER_IN_SPLIT'] = np.arange(len(train))
    split.loc[test, 'ORDER_IN_SPLIT'] = np.arange(len(test))
    split.to_csv(out / 'shared_H180_train_test_split.csv', index=False)
    print(
        f'\nSHARED H180 SPLIT — TRAIN={len(train):,}; TEST={len(test):,}; '
        f'events train/test={int(y[train].sum()):,}/{int(y[test].sum()):,}'
    )
    return train, test


def build_bsym_h180(matrix_a, cohort, train_idx, numeric_columns, out):
    """Construct H180 B-sym on the same cohort/labels/split.

    Positive customers:
        cutoff = observed FIRST_CARD_MONTH (same as Branch B).

    Negative customers:
        pseudo duration L* is sampled with replacement from
        (FIRST_CARD_MONTH - RELATIONSHIP_START_DATE) among positive TRAIN customers only.
        cutoff = min(RELATIONSHIP_START_DATE + L*, HORIZON_END_DATE).

    Feature rule is unchanged from H180 V3:
        RELATIONSHIP_START_DATE <= source time < cutoff.
    """
    bsym_out = out / 'BSYM_FIXED_HORIZON'
    bsym_out.mkdir(parents=True, exist_ok=True)

    c = cohort.copy().reset_index(drop=True)

    if len(c) != len(matrix_a):
        raise ValueError('H180 B-sym: cohort/matrix A length mismatch.')

    if not np.array_equal(
        c[ID].astype(str).to_numpy(),
        matrix_a[ID].astype(str).to_numpy()
    ):
        raise ValueError('H180 B-sym: cohort/matrix A ID order mismatch.')

    y = c[Y].to_numpy(dtype=np.int8)
    train_idx = np.asarray(train_idx, dtype=int)
    train_positive = train_idx[y[train_idx] == 1]

    if len(train_positive) == 0:
        raise ValueError('H180 B-sym: no positive customers in training.')

    positive_durations = (
        c.loc[train_positive, EVENT].reset_index(drop=True)
        - c.loc[train_positive, START].reset_index(drop=True)
    )

    if positive_durations.isna().any():
        raise ValueError('H180 B-sym: positive training duration contains missing values.')

    if not (positive_durations > pd.Timedelta(days=LANDMARK_DAYS)).all():
        raise ValueError('H180 B-sym: a positive training event is not strictly after day 60.')

    if not (positive_durations <= pd.Timedelta(days=LANDMARK_DAYS + HORIZON_DAYS)).all():
        raise ValueError('H180 B-sym: a positive training event is beyond day 240.')

    duration_values = positive_durations.to_numpy(dtype='timedelta64[ns]')
    rng = np.random.default_rng(BSYM_SEED)

    positive = c[Y].eq(1).to_numpy()
    negative = ~positive

    cutoff = pd.Series(pd.NaT, index=c.index, dtype='datetime64[ns]')
    proposed = pd.Series(pd.NaT, index=c.index, dtype='datetime64[ns]')
    sampled_duration = pd.Series(pd.NaT, index=c.index, dtype='timedelta64[ns]')

    # Positives: exact event cutoff.
    cutoff.loc[positive] = pd.to_datetime(c.loc[positive, EVENT]).to_numpy()
    proposed.loc[positive] = cutoff.loc[positive].to_numpy()
    sampled_duration.loc[positive] = (
        pd.to_datetime(c.loc[positive, EVENT]).to_numpy(dtype='datetime64[ns]')
        - pd.to_datetime(c.loc[positive, START]).to_numpy(dtype='datetime64[ns]')
    )

    # Negatives: sample train-positive duration, then cap at r+240.
    neg_idx = np.where(negative)[0]
    sampled_neg = rng.choice(duration_values, size=len(neg_idx), replace=True)
    sampled_duration.loc[negative] = sampled_neg

    proposed_neg = (
        pd.to_datetime(c.loc[negative, START]).to_numpy(dtype='datetime64[ns]')
        + sampled_neg
    )
    tend_neg = pd.to_datetime(
        c.loc[negative, 'HORIZON_END_DATE']
    ).to_numpy(dtype='datetime64[ns]')
    cutoff_neg = np.minimum(proposed_neg, tend_neg)

    proposed.loc[negative] = proposed_neg
    cutoff.loc[negative] = cutoff_neg

    landmark = pd.to_datetime(c['LANDMARK_DATE'])

    if (positive & ~(cutoff > landmark)).any():
        raise ValueError('H180 B-sym: positive cutoff is not strictly after landmark.')

    if (negative & ~(cutoff > landmark)).any():
        raise ValueError('H180 B-sym: negative pseudo-cutoff is not strictly after landmark.')

    if (cutoff > pd.to_datetime(c['HORIZON_END_DATE'])).any():
        raise ValueError('H180 B-sym: cutoff exceeds HORIZON_END_DATE.')

    cutoff_audit = c[
        [ID, Y, START, 'LANDMARK_DATE', EVENT, 'HORIZON_END_DATE']
    ].copy()
    cutoff_audit['SAMPLED_DURATION_DAYS'] = (
        sampled_duration.dt.total_seconds() / 86400.0
    )
    cutoff_audit['PROPOSED_CUTOFF'] = proposed
    cutoff_audit['BSYM_CUTOFF'] = cutoff
    cutoff_audit['CAPPED_AT_TEND'] = negative & (cutoff < proposed)
    cutoff_audit.to_csv(bsym_out / 'H180_Bsym_cutoff_audit.csv', index=False)

    neg_realized_days = (
        cutoff.loc[negative].reset_index(drop=True)
        - pd.to_datetime(c.loc[negative, START]).reset_index(drop=True)
    ).dt.total_seconds() / 86400.0
    pos_train_days = positive_durations.dt.total_seconds() / 86400.0

    construction_summary = pd.DataFrame([{
        'n_cohort': int(len(c)),
        'n_events': int(c[Y].sum()),
        'n_train_positive_durations': int(len(pos_train_days)),
        'positive_train_duration_mean_days': float(pos_train_days.mean()),
        'positive_train_duration_median_days': float(pos_train_days.median()),
        'positive_train_duration_q25_days': float(pos_train_days.quantile(0.25)),
        'positive_train_duration_q75_days': float(pos_train_days.quantile(0.75)),
        'positive_train_duration_min_days': float(pos_train_days.min()),
        'positive_train_duration_max_days': float(pos_train_days.max()),
        'negative_realized_duration_mean_days': float(neg_realized_days.mean()),
        'negative_realized_duration_median_days': float(neg_realized_days.median()),
        'negative_realized_duration_q25_days': float(neg_realized_days.quantile(0.25)),
        'negative_realized_duration_q75_days': float(neg_realized_days.quantile(0.75)),
        'negative_cutoffs_capped_at_Tend': int(cutoff_audit['CAPPED_AT_TEND'].sum()),
        'Bsym_seed': BSYM_SEED,
        'sampling_source': 'positive TRAIN customers only',
        'sampling': 'with replacement',
    }])
    construction_summary.to_csv(
        bsym_out / 'H180_Bsym_construction_summary.csv',
        index=False
    )

    # Reuse exact H180 V3 dynamic-feature builder.
    master_bsym = c[[ID, START]].copy()
    master_bsym[CUTOFF] = cutoff.to_numpy()
    rebuilt_bsym, audit_bsym = rebuild_dynamic(
        master_bsym,
        bsym_out / 'DYNAMIC_FEATURES',
        'Bsym'
    )

    # Same five static A60 attributes as Branch B.
    matrix_bsym = (
        c[[ID, Y]]
        .merge(
            matrix_a[[ID] + STATIC_FEATURES],
            on=ID,
            how='left',
            validate='one_to_one'
        )
        .merge(
            rebuilt_bsym,
            on=ID,
            how='left',
            validate='one_to_one'
        )
    )
    matrix_bsym = matrix_bsym[[ID, Y] + FEATURES].reset_index(drop=True)

    for col in FEATURES:
        if col in numeric_columns:
            converted = pd.to_numeric(matrix_bsym[col], errors='coerce')
            invalid = matrix_bsym[col].notna() & converted.isna()
            if invalid.any():
                fail_with_rows(
                    matrix_bsym.loc[invalid, [ID, col]],
                    bsym_out,
                    f'CHECK_Bsym_dtype_{col}.csv',
                    f'{col}: B-sym type does not match numeric A type.'
                )
            matrix_bsym[col] = converted
        else:
            s = matrix_bsym[col]
            matrix_bsym[col] = s.astype(str).where(s.notna(), np.nan)

    if not np.array_equal(
        matrix_bsym[ID].astype(str).to_numpy(),
        matrix_a[ID].astype(str).to_numpy()
    ):
        raise AssertionError('H180 B-sym ID/order differs from A.')

    if not np.array_equal(
        matrix_bsym[Y].to_numpy(dtype=np.int8),
        matrix_a[Y].to_numpy(dtype=np.int8)
    ):
        raise AssertionError('H180 B-sym labels differ from A.')

    for col in STATIC_FEATURES:
        left = matrix_a[col].astype('string').fillna('<MISSING>')
        right = matrix_bsym[col].astype('string').fillna('<MISSING>')
        if not left.equals(right):
            raise AssertionError(f'H180 B-sym reused static feature differs: {col}')

    if numeric_columns:
        numeric_array = matrix_bsym[numeric_columns].to_numpy(dtype=float)
        if np.isinf(numeric_array).any():
            raise ValueError('H180 B-sym contains infinite numeric values.')

    matrix_bsym.to_csv(
        bsym_out / 'Bsym_fixed_horizon_matrix.csv',
        index=False
    )
    audit_bsym.to_csv(
        bsym_out / 'Bsym_source_audit_summary.csv',
        index=False
    )

    show_table(construction_summary.round(6), 'H180 B-SYM — CONSTRUCTION SUMMARY')
    return matrix_bsym, cutoff_audit, construction_summary


def paired_bootstrap_three(y, p_a, p_bsym, p_b, n_bootstrap, seed):
    """Paired customer bootstrap on the same H180 holdout for A/B-sym/B."""
    y = np.asarray(y, dtype=np.int8)
    p_a = np.asarray(p_a, dtype=float)
    p_bsym = np.asarray(p_bsym, dtype=float)
    p_b = np.asarray(p_b, dtype=float)

    if not (len(y) == len(p_a) == len(p_bsym) == len(p_b)):
        raise ValueError('H180 bootstrap: prediction lengths differ.')
    if np.unique(y).size != 2:
        raise ValueError('H180 bootstrap: holdout must contain both classes.')

    rng = np.random.default_rng(seed)
    rows = []
    attempts = 0
    max_attempts = max(10000, n_bootstrap * 20)

    while len(rows) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        idx = rng.integers(0, len(y), size=len(y))
        yy = y[idx]
        if np.unique(yy).size != 2:
            continue

        auc_a = float(roc_auc_score(yy, p_a[idx]))
        auc_bsym = float(roc_auc_score(yy, p_bsym[idx]))
        auc_b = float(roc_auc_score(yy, p_b[idx]))

        rows.append({
            'replicate': len(rows) + 1,
            'AUC_A': auc_a,
            'AUC_Bsym': auc_bsym,
            'AUC_B': auc_b,
            'D_window_B_minus_A': auc_b - auc_a,
            'D_matched_Bsym_minus_A': auc_bsym - auc_a,
            'D_endpoint_B_minus_Bsym': auc_b - auc_bsym,
        })

        if len(rows) % 200 == 0 or len(rows) == n_bootstrap:
            print(f'Paired bootstrap A/B/B-sym: {len(rows)}/{n_bootstrap}')

    if len(rows) != n_bootstrap:
        raise RuntimeError(
            f'Only {len(rows)} valid H180 bootstrap replicates; expected {n_bootstrap}.'
        )

    return pd.DataFrame(rows)


def fit_and_report_three(
    matrix_a,
    matrix_bsym,
    matrix_b,
    numeric_columns,
    cohort_summary,
    train,
    test,
    out
):
    """Fit XGBoost A/B-sym/B on the same H180 split and report paired contrasts."""
    from xgboost import XGBClassifier

    y = matrix_a[Y].to_numpy(dtype=np.int8)

    for branch_name, matrix in [('Bsym', matrix_bsym), ('B', matrix_b)]:
        if not np.array_equal(
            matrix_a[ID].astype(str).to_numpy(),
            matrix[ID].astype(str).to_numpy()
        ):
            raise AssertionError(f'H180 {branch_name}: ID/order differs from A.')
        if not np.array_equal(
            matrix_a[Y].to_numpy(dtype=np.int8),
            matrix[Y].to_numpy(dtype=np.int8)
        ):
            raise AssertionError(f'H180 {branch_name}: labels differ from A.')

    probabilities = {}
    metric_rows = []

    for branch_name, matrix in [
        ('A', matrix_a),
        ('Bsym', matrix_bsym),
        ('B', matrix_b),
    ]:
        print(f'\nFIT H180 {branch_name}: 18 features; XGBoost; no resampling')

        model = Pipeline([
            ('preprocess', make_preprocessor(numeric_columns)),
            ('xgb', XGBClassifier(**XGB_PARAMS)),
        ])
        model.fit(matrix.loc[train, FEATURES], y[train])
        prob = model.predict_proba(matrix.loc[test, FEATURES])[:, 1]
        probabilities[branch_name] = prob

        row = metric_row(y[test], prob, branch_name, len(train))
        metric_rows.append(row)
        joblib.dump(model, out / f'model_{branch_name}_H180.joblib')

        print(
            f"{branch_name}: AUC={row['AUC']:.6f}; KS={row['KS']:.6f}; "
            f"BSS={row['BSS']:.6f}; Capture@10={row['Capture_at_10']:.6f}"
        )

    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(out / 'H180_AB_Bsym_holdout_metrics.csv', index=False)

    predictions = pd.DataFrame({
        ID: matrix_a.loc[test, ID].to_numpy(),
        Y: y[test],
        'PROB_A': probabilities['A'],
        'PROB_Bsym': probabilities['Bsym'],
        'PROB_B': probabilities['B'],
    })
    predictions.to_csv(out / 'H180_AB_Bsym_holdout_predictions.csv', index=False)

    boot = paired_bootstrap_three(
        y=y[test],
        p_a=probabilities['A'],
        p_bsym=probabilities['Bsym'],
        p_b=probabilities['B'],
        n_bootstrap=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED,
    )
    boot.to_csv(out / 'H180_AB_Bsym_paired_bootstrap_values.csv', index=False)

    auc_map = metrics.set_index('branch')['AUC'].to_dict()
    observed = {
        'D_window_B_minus_A': float(auc_map['B'] - auc_map['A']),
        'D_matched_Bsym_minus_A': float(auc_map['Bsym'] - auc_map['A']),
        'D_endpoint_B_minus_Bsym': float(auc_map['B'] - auc_map['Bsym']),
    }

    contrast_rows = []
    for name, obs in observed.items():
        values = boot[name].to_numpy(dtype=float)
        q025, q975 = np.quantile(values, [0.025, 0.975], method='linear')
        contrast_rows.append({
            'contrast': name,
            'observed': obs,
            'ci_lower': float(q025),
            'ci_upper': float(q975),
            'bootstrap_valid': int(len(values)),
            'n_gt_0': int((values > 0).sum()),
        })

    contrasts = pd.DataFrame(contrast_rows)
    contrasts.to_csv(out / 'H180_AB_Bsym_paired_contrasts.csv', index=False)

    ref_check = pd.DataFrame([{
        'current_AUC_A': auc_map['A'],
        'reference_AUC_A': REFERENCE_H180['AUC_A'],
        'abs_diff_AUC_A': abs(auc_map['A'] - REFERENCE_H180['AUC_A']),
        'current_AUC_B': auc_map['B'],
        'reference_AUC_B': REFERENCE_H180['AUC_B'],
        'abs_diff_AUC_B': abs(auc_map['B'] - REFERENCE_H180['AUC_B']),
        'current_Dwindow_H': observed['D_window_B_minus_A'],
        'reference_Dwindow_H': REFERENCE_H180['Dwindow_H'],
        'abs_diff_Dwindow_H': abs(
            observed['D_window_B_minus_A'] - REFERENCE_H180['Dwindow_H']
        ),
    }])
    ref_check.to_csv(out / 'H180_reference_reproduction_check.csv', index=False)

    paper = pd.DataFrame([{
        'H_days': HORIZON_DAYS,
        'landmark_days': LANDMARK_DAYS,
        'endpoint_days': LANDMARK_DAYS + HORIZON_DAYS,
        'n': int(len(matrix_a)),
        'events': int(y.sum()),
        'n_train': int(len(train)),
        'n_test': int(len(test)),
        'events_test': int(y[test].sum()),
        'AUC_A': float(auc_map['A']),
        'AUC_Bsym': float(auc_map['Bsym']),
        'AUC_B': float(auc_map['B']),
        **observed,
        'bootstrap_replicates': int(N_BOOTSTRAP),
        'split_seed': int(SPLIT_SEED),
        'Bsym_seed': int(BSYM_SEED),
        'negatives_without_endpoint_snapshot_or_later_event': int(
            cohort_summary.iloc[0]['negatives_without_endpoint_snapshot_or_later_event']
        ),
    }])
    paper.to_csv(out / 'H180_AB_Bsym_paper_row.csv', index=False)

    show_table(metrics.round(6), 'H180 A / B-SYM / B — HOLDOUT METRICS')
    show_table(contrasts.round(6), 'H180 A / B-SYM / B — 1,000 PAIRED BOOTSTRAP CONTRASTS')
    show_table(ref_check.round(6), 'H180 — CHECK AGAINST PREVIOUSLY CONFIRMED A/B RESULT')

    print('\nFINAL H180 PAPER ROW')
    print(paper.round(6).to_string(index=False))

    return {
        'metrics': metrics,
        'contrasts': contrasts,
        'bootstrap': boot,
        'predictions': predictions,
        'paper': paper,
        'reference_check': ref_check,
    }


def run_experiment_h180_bsym():
    """Run verified H180 V3 construction, then add B-sym on the same cohort/split."""
    if str(ROOT).startswith('/content/'):
        try:
            from google.colab import drive
        except ImportError:
            pass
        else:
            if not (Path('/content/drive') / 'MyDrive').is_dir():
                drive.mount('/content/drive', force_remount=False)

    for name, path in FILES.items():
        if not path.is_file():
            raise FileNotFoundError(f'Missing H180 source {name}: {path}')

    self_check_time_rules()

    run_id = datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    out = OUTPUT_PARENT_BSYM / run_id
    out.mkdir(parents=True, exist_ok=False)

    print('=' * 120)
    print('VIB H180 — A / B / B-SYM — XGBOOST — 1,000 PAIRED BOOTSTRAPS')
    print('=' * 120)
    print('Based directly on verified H180 V3 code.')
    print('Output:', out)

    manifest = {
        'status': 'STARTED',
        'run_id': run_id,
        'code_revision': CODE_REVISION_BSYM,
        'base_code': CODE_REVISION,
        'landmark_days': LANDMARK_DAYS,
        'H_days': HORIZON_DAYS,
        'endpoint_days': LANDMARK_DAYS + HORIZON_DAYS,
        'label': '1 iff r+60 < FIRST_CARD_MONTH <= r+240',
        'A': 'same H180 V3 A: original 18 A60 features, H180 cohort/label',
        'B': 'same H180 V3 B: event cutoff for positive; r+240 for negative',
        'Bsym_positive_cutoff': 'FIRST_CARD_MONTH',
        'Bsym_negative_cutoff': 'min(r + sampled positive-training event delay, r+240)',
        'Bsym_sampling_source': 'empirical e-r distribution from positive TRAIN customers only',
        'Bsym_sampling': 'with replacement',
        'same_ID_label_split': True,
        'split_seed': SPLIT_SEED,
        'Bsym_seed': BSYM_SEED,
        'xgb_params': XGB_PARAMS,
        'bootstrap_replicates': N_BOOTSTRAP,
        'bootstrap_seed': BOOTSTRAP_SEED,
        'resampling': 'none',
        'features': FEATURES,
    }
    save_json(manifest, out / 'run_manifest.json')

    started = time.monotonic()

    try:
        a_raw = read_table(FILES['A_60D'])
        customer = read_table(FILES['customer'])
        card = read_table(FILES['card'], [ID, 'MONTH', 'COUNT_CREDITCARD'])

        cohort, cohort_summary = build_cohort(a_raw, customer, card, out)

        matrix_a, matrix_b, numeric_columns = build_matrices(
            a_raw,
            cohort,
            out / 'MATRICES'
        )

        if len(matrix_a) != 31_149 or int(matrix_a[Y].sum()) != 1_454:
            raise ValueError(
                f'H180 expected N=31,149/events=1,454; '
                f'got N={len(matrix_a):,}/events={int(matrix_a[Y].sum()):,}.'
            )

        train, test = make_shared_h180_split(matrix_a, out)

        matrix_bsym, cutoff_audit, bsym_summary = build_bsym_h180(
            matrix_a=matrix_a,
            cohort=cohort,
            train_idx=train,
            numeric_columns=numeric_columns,
            out=out
        )

        results = fit_and_report_three(
            matrix_a=matrix_a,
            matrix_bsym=matrix_bsym,
            matrix_b=matrix_b,
            numeric_columns=numeric_columns,
            cohort_summary=cohort_summary,
            train=train,
            test=test,
            out=out
        )

        manifest['status'] = 'COMPLETED'
        manifest['result'] = results['paper'].iloc[0].to_dict()
        manifest['Bsym_construction'] = bsym_summary.iloc[0].to_dict()

        print('\n' + '=' * 120)
        print('COMPLETED H180 A / B / B-SYM')
        print('=' * 120)
        print('Saved:', out)

        return {
            'output': out,
            'cohort_summary': cohort_summary,
            'Bsym_summary': bsym_summary,
            **results,
        }

    except Exception as exc:
        manifest['status'] = 'FAILED'
        manifest['error'] = str(exc)
        print(
            '\nFAILED. Do not use partial output in the paper. '
            'Check the saved CHECK/audit files.'
        )
        raise

    finally:
        manifest['elapsed_seconds'] = time.monotonic() - started
        save_json(manifest, out / 'run_manifest.json')


In [ ]:
RESULTS_H180_AB_BSYM = run_experiment_h180_bsym()


PASS: nhãn mở bên ngày 60, đóng bên ngày 240.
VIB H180 — A / B / B-SYM — XGBOOST — 1,000 PAIRED BOOTSTRAPS
Based directly on verified H180 V3 code.
Output: /content/drive/MyDrive/Fintect/VIB_FIXED_HORIZON_H180_A60_B240_BSYM/run_20261013T021406455223Z
ĐỌC LẠI NGUỒN NGÀY 60D: /content/drive/MyDrive/Fintect/target_landmark_60d.csv

GHÉP LẠI NGÀY TẠO TỪ CÁC NGUỒN ĐÃ ĐỌC


,n_original_FLDC,n_date_in_raw_customer,n_date_rejoined_from_verified_saved_source,n_still_missing_date
0,94453,94453,0,0



COHORT VÀ NHÃN THEO HORIZON CỐ ĐỊNH


,H_days,landmark_days,base_n,base_positives,n,events,prevalence,excluded_administrative_end,event_after_H_now_label_0,negatives_without_snapshot_at_or_after_H,negatives_without_endpoint_snapshot_or_later_event
0,180,60,94453,4374,31149,1454,0.046679,63304,1093,0,0



ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: transaction
Dùng 57,742 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: activity
Dùng 927,440 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: deposit
Dùng 43,135 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: lending
Dùng 5,452 dòng; vi phạm timestamp=0.

ĐỐI CHIẾU A60 ĐÃ LƯU VỚI 13 BIẾN TÁI TẠO TỪ NGUỒN


,feature,n_checked,n_mismatch
0,LOGIN_PER_ACTIVE_DAY,31149,0
1,INTEREST_RATE_RATIO,31149,0
2,TRANS_LV1_MODE,31149,0
3,TRANS_LV2_MODE,31149,0
4,TRANS_AMOUNT_MAX,31149,0
5,TRANS_AMOUNT_MEAN,31149,0
6,COUNT_CA_ACCT,31149,0
7,AVG_CA_BALANCE,31149,0
8,COUNT_TD_ACCT,31149,0
9,AVG_TD_BALANCE,31149,0



ĐỌC LẠI + TẠO ĐẶC TRƯNG B: transaction
Dùng 385,886 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG B: activity


In [3]:
# =====================================================================
# ELAPS — REVISION RUNS (VIB): landmark-floored lag curve + H180 R50
# Run AFTER the cell "ELAPS — RUN_AUDIT CHO VIB" with RUN = () (it loads
# the data, the cohorts and the verified feature builder build_features).
#
# 1. Lag curve with positive cutoffs floored at the day-60 landmark:
#    t+ = max(e - k, r + 60 days). No lag branch has less history than A.
#    Reports, per cohort and k, the share of positives that hit the floor.
# 2. H180: four learners x 50 stratified 80:20 split/refit runs (seeds 42-91),
#    A / B-sym / B, B-sym redrawn per split from training positives (pooled),
#    same learner settings as the administrative-endpoint R50 grid.
# 3. Calibration-in-the-large (mean predicted probability) for every branch.
# Rows read from the raw files are cached in memory, so repeated feature builds
# give exactly the same features as the uncached builder, only faster.
# =====================================================================
import gc, json, time, inspect
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

REV_STAMP = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ')
REV_DIR = AUDIT_ROOT / f'REVISION_{REV_STAMP}'
REV_DIR.mkdir(parents=True, exist_ok=False)
print('Output:', REV_DIR)

# ---------------------------------------------------------------- cache of raw rows
_ROWS_CACHE = {}
_orig_rows_for_window = rows_for_window


def rows_for_window(path, cols, master, date_col):          # noqa: F811  (overrides the audit builder's reader)
    key = (str(path), tuple(cols), date_col, len(master), hash(tuple(master[ID].astype(str))))
    if key not in _ROWS_CACHE:
        full = master[[ID, START]].copy()
        full[CUTOFF] = pd.Timestamp('2262-01-01')            # read every row from r onwards once
        _ROWS_CACHE[key] = _orig_rows_for_window(path, cols, full, date_col).drop(columns=[CUTOFF])
    df = _ROWS_CACHE[key].merge(master[[ID, CUTOFF]], on=ID, how='inner', validate='many_to_one')
    df = df[df[date_col] < df[CUTOFF]]
    return df.reset_index(drop=True) if len(df) else pd.DataFrame(columns=cols + [START, CUTOFF])


# ---------------------------------------------------------------- 1. landmark-floored lag
LAG_SHARE = []


def lag_positive(start, event, k):                           # noqa: F811
    start = pd.to_datetime(pd.Series(start)).reset_index(drop=True)
    ev = pd.to_datetime(pd.Series(event)).reset_index(drop=True)
    t = ev - pd.Timedelta(days=k)
    floor = start + pd.Timedelta(days=DAY60)
    pos = ev.notna()
    LAG_SHARE.append(dict(k=k, n_pos=int(pos.sum()),
                          share_floored=float((t[pos] <= floor[pos]).mean())))
    return t.where(t > floor, floor)


_orig_finalize = finalize


def finalize(tag, y_te, scores, balance, info, t0, extra_auc=(), extra_bss=()):   # noqa: F811
    info = dict(info, lag_rule='positive cutoff = max(e - k, r + 60 days)',
                lag_share_floored=[dict(s, cohort=tag) for s in LAG_SHARE])
    calib = pd.DataFrame([dict(branch=k, mean_pred=float(np.mean(p)), prevalence=float(np.mean(y_te)))
                          for k, p in scores.items()])
    print('\nLAG FLOOR SHARE\n', pd.DataFrame(LAG_SHARE).round(4).to_string(index=False))
    print('\nCALIBRATION-IN-THE-LARGE\n', calib.round(5).to_string(index=False))
    calib.to_csv(REV_DIR / f'{tag}_calibration.csv', index=False)
    pd.DataFrame(LAG_SHARE).to_csv(REV_DIR / f'{tag}_lag_floor_share.csv', index=False)
    out = _orig_finalize(tag + '_LAGFLOOR', y_te, scores, balance, info, t0, extra_auc, extra_bss)
    LAG_SHARE.clear()
    return out


# ---------------------------------------------------------------- 2. H180 four-learner R50
R50_SEEDS = range(42, 92)
LEARNERS = ['LR', 'RF', 'LightGBM', 'XGBoost']


def _prep(X, learner):
    num = X.select_dtypes(include=np.number).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    kw = {'handle_unknown': 'ignore'}
    kw['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse'] = False
    nump = [('imputer', SimpleImputer(strategy='constant', fill_value=0))]
    if learner == 'LR':
        nump.append(('scaler', StandardScaler()))
    t = []
    if num:
        t.append(('num', Pipeline(nump), num))
    if cat:
        t.append(('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
                                   ('onehot', OneHotEncoder(**kw))]), cat))
    return ColumnTransformer(t, remainder='drop')


def _model(learner, seed):
    if learner == 'LR':
        return LogisticRegression(solver='lbfgs', max_iter=2000, class_weight='balanced', random_state=seed)
    if learner == 'RF':
        return RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=2)
    if learner == 'LightGBM':
        return LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=6, subsample=0.8,
                              colsample_bytree=0.8, random_state=seed, n_jobs=2, verbosity=-1)
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                         objective='binary:logistic', eval_metric='logloss', tree_method='hist', max_bin=256,
                         random_state=seed, n_jobs=2)


def _fit(mat, y, tr, te, learner, seed):
    Xtr, Xte = mat.loc[tr, FEATURES], mat.loc[te, FEATURES]
    p = _prep(Xtr, learner)
    m = _model(learner, seed).fit(p.fit_transform(Xtr), y[tr])
    return np.asarray(m.predict_proba(p.transform(Xte))[:, 1], float)


def run_h180_r50():
    t0 = time.monotonic()
    m = MASTER.copy()
    m['LANDMARK_DATE'] = m[START] + pd.Timedelta(days=DAY60)
    m['HORIZON_END_DATE'] = m['LANDMARK_DATE'] + pd.Timedelta(days=HORIZON)
    m['Y_H'] = (m[EVENT].notna() & (m[EVENT] > m['LANDMARK_DATE'])
                & (m[EVENT] <= m['HORIZON_END_DATE'])).astype(np.int8)
    c = m.loc[m['HORIZON_END_DATE'] <= OBSERVATION_END].sort_values(ID, kind='stable').reset_index(drop=True)
    y = c['Y_H'].to_numpy(np.int8)
    if len(c) != 31_149 or int(y.sum()) != 1_454:
        fail('H180 cohort differs')
    c[ID] = c[ID].astype(str)
    mat_a = A_ALL.set_index(ID).loc[c[ID].tolist(), FEATURES].reset_index()
    mat_a[ID] = mat_a[ID].astype(str)
    mat_a = h180_types(mat_a, NUM_H180)

    def matrix_at(cut, label):
        mm = c[[ID, START]].copy()
        mm[CUTOFF] = pd.to_datetime(cut).to_numpy()
        built = build_features(mm, A_ALL, label)
        built[ID] = built[ID].astype(str)
        mat = mat_a[[ID] + STATIC].merge(built[[ID] + DYNAMIC], on=ID, how='left', validate='one_to_one')
        mat = h180_types(mat[[ID] + FEATURES], NUM_H180)
        if mat[ID].tolist() != mat_a[ID].tolist() or mat[DYNAMIC].isna().any().any():
            fail(label + ': alignment')
        return mat

    start, event, T = c[START], c[EVENT], c['HORIZON_END_DATE']
    mat_b = matrix_at(event.where(c['Y_H'].eq(1), T), 'B_H180')
    # sanity: the A builder reproduces A on this cohort
    chk = matrix_at(start + pd.Timedelta(days=DAY60), 'CHECK_A_H180')
    bad = [col for col in DYNAMIC if not np.allclose(pd.to_numeric(chk[col], errors='coerce'),
                                                     pd.to_numeric(mat_a[col], errors='coerce'), equal_nan=True)
           if col in NUM_H180] + [col for col in DYNAMIC if col not in NUM_H180
                                  and not (chk[col].astype(str) == mat_a[col].astype(str)).all()]
    print('A rebuild on H180, mismatching columns:', bad)
    if bad:
        fail('builder does not reproduce A on H180')
    del chk
    delay_days = (event - start).dt.days.to_numpy()
    rows, ckpt = [], REV_DIR / 'H180_R50_four_learners.csv'
    for r, seed in enumerate(R50_SEEDS, start=1):
        tr, te = train_test_split(np.arange(len(c)), test_size=0.20, stratify=y, random_state=seed)
        pool = delay_days[tr[y[tr] == 1]]
        rng = np.random.default_rng(20261012 + int(seed))
        cut = event.where(c['Y_H'].eq(1), pd.NaT).copy()
        neg = np.where(y == 0)[0]
        prop = start.iloc[neg].to_numpy('datetime64[ns]') + pd.to_timedelta(
            rng.choice(pool, len(neg), replace=True), unit='D').to_numpy()
        cut.iloc[neg] = np.minimum(prop, T.iloc[neg].to_numpy('datetime64[ns]'))
        mat_s = matrix_at(cut, f'Bsym_seed{seed}')
        for learner in LEARNERS:
            pr = {b: _fit(mt, y, tr, te, learner, seed) for b, mt in (('A', mat_a), ('Bsym', mat_s), ('B', mat_b))}
            auc = {b: roc_auc_score(y[te], p) for b, p in pr.items()}
            rows.append(dict(seed=seed, learner=learner, AUC_A=auc['A'], AUC_Bsym=auc['Bsym'], AUC_B=auc['B'],
                             D_total_B_minus_A=auc['B'] - auc['A'], D_sym_Bsym_minus_A=auc['Bsym'] - auc['A'],
                             D_res_B_minus_Bsym=auc['B'] - auc['Bsym'],
                             tau_A_B=float(kendalltau(pr['A'], pr['B']).statistic),
                             tau_A_Bsym=float(kendalltau(pr['A'], pr['Bsym']).statistic)))
        pd.DataFrame(rows).to_csv(ckpt, index=False)
        print(f'H180 R50 {r}/50 seed={seed}  ({(time.monotonic() - t0) / 60:.1f} min)')
        del mat_s
        gc.collect()
    r50 = pd.DataFrame(rows)
    summ = []
    for learner, g in r50.groupby('learner'):
        for col in ['D_total_B_minus_A', 'D_sym_Bsym_minus_A', 'D_res_B_minus_Bsym', 'tau_A_B', 'tau_A_Bsym']:
            v = g[col].to_numpy()
            summ.append(dict(learner=learner, metric=col, R=len(v), mean=v.mean(), sd=v.std(ddof=1),
                             q025=np.quantile(v, .025), q975=np.quantile(v, .975),
                             n_gt_0=int((v > 0).sum()) if col.startswith('D_') else np.nan))
    summ = pd.DataFrame(summ)
    summ.to_csv(REV_DIR / 'H180_R50_summary.csv', index=False)
    print('\nH180 R50 SUMMARY\n', summ.round(4).to_string(index=False))
    print(f'Total {(time.monotonic() - t0) / 3600:.2f} h')
    return r50, summ


REV = {}
REV['H180_LAG'] = run_h180()        # A/B gate + B-sym-rs + floored lag curve + B->A
_ROWS_CACHE.clear(); gc.collect()
REV['ADMIN_LAG'] = run_admin()      # same for the administrative-endpoint cohort
_ROWS_CACHE.clear(); gc.collect()
REV['H180_R50'] = run_h180_r50()    # four learners x 50 refits on H180


Output: /content/drive/MyDrive/Fintect/ELAPS_RUN_AUDIT/REVISION_20261121T092646Z

VIB H180
H180: N=31,149; sự kiện=1,454; train/test sự kiện=1163/291
  Dựng B_H180: giao dịch / hoạt động / tiền gửi / khoản vay


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


H180: A=0.891002 (công bố 0.891002); B=0.906161 (công bố 0.906161)

=== H180 Bsym_rs ===
  Dựng Bsym_rs: giao dịch / hoạt động / tiền gửi / khoản vay
Bsym_rs: AUC=0.903357

=== H180 lag7 ===
  Dựng lag7: giao dịch / hoạt động / tiền gửi / khoản vay
lag7: AUC=0.899950

=== H180 lag30 ===
  Dựng lag30: giao dịch / hoạt động / tiền gửi / khoản vay
lag30: AUC=0.898577

=== H180 lag60 ===
  Dựng lag60: giao dịch / hoạt động / tiền gửi / khoản vay
lag60: AUC=0.893005

LAG FLOOR SHARE
  k  n_pos  share_floored
 7   2547         0.0432
30   2547         0.1437
60   2547         0.2662

CALIBRATION-IN-THE-LARGE
     branch  mean_pred  prevalence
         A    0.04370     0.04671
         B    0.04363     0.04671
      B->A    0.05246     0.04671
   Bsym_rs    0.04272     0.04671
Bsym_rs->A    0.03883     0.04671
      lag7    0.04296     0.04671
     lag30    0.04271     0.04671
     lag60    0.04366     0.04671
  bootstrap 250/1000
  bootstrap 500/1000
  bootstrap 750/1000
  bootstrap 1000/100

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


  A: số giá trị lệch = 0
  Dựng B_rebuilt: giao dịch / hoạt động / tiền gửi / khoản vay
  B (so với file đã lưu): số giá trị lệch = [('LOGIN_PER_ACTIVE_DAY', 12653), ('INTEREST_RATE_RATIO', 12749), ('TRANS_LV1_MODE', 60472), ('TRANS_LV2_MODE', 61012), ('TRANS_AMOUNT_MAX', 1962), ('TRANS_AMOUNT_MEAN', 22513), ('COUNT_CA_ACCT', 2357), ('AVG_CA_BALANCE', 71452), ('COUNT_TD_ACCT', 1842), ('AVG_TD_BALANCE', 3215), ('COUNT_OF_LOAN', 589), ('AVG_LOAN_AMOUNT', 4766), ('TOTAL_LOAN_AMOUNT', 4771)]
B_rebuilt: AUC=0.922914

=== ADMIN Bsym_rs ===
  Dựng Bsym_rs: giao dịch / hoạt động / tiền gửi / khoản vay
Bsym_rs: AUC=0.923381

=== ADMIN lag7 ===
  Dựng lag7: giao dịch / hoạt động / tiền gửi / khoản vay
lag7: AUC=0.921093

=== ADMIN lag30 ===
  Dựng lag30: giao dịch / hoạt động / tiền gửi / khoản vay
lag30: AUC=0.918151

=== ADMIN lag60 ===
  Dựng lag60: giao dịch / hoạt động / tiền gửi / khoản vay
lag60: AUC=0.914441

LAG FLOOR SHARE
  k  n_pos  share_floored
 7   4374         0.0725
30   4374   

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


... (97 outputs left out)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


H180 R50 49/50 seed=90  (36.5 min)
  Dựng Bsym_seed91: giao dịch / hoạt động / tiền gửi / khoản vay


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


H180 R50 50/50 seed=91  (37.2 min)

H180 R50 SUMMARY
  learner             metric  R    mean     sd    q025    q975  n_gt_0
      LR  D_total_B_minus_A 50  0.0065 0.0048 -0.0011  0.0168    47.0
      LR D_sym_Bsym_minus_A 50  0.0140 0.0050  0.0053  0.0242    50.0
      LR D_res_B_minus_Bsym 50 -0.0075 0.0018 -0.0109 -0.0044     0.0
      LR            tau_A_B 50  0.8100 0.0057  0.8018  0.8200     NaN
      LR         tau_A_Bsym 50  0.8519 0.0053  0.8429  0.8624     NaN
LightGBM  D_total_B_minus_A 50  0.0148 0.0063  0.0026  0.0265    49.0
LightGBM D_sym_Bsym_minus_A 50  0.0153 0.0062  0.0023  0.0261    50.0
LightGBM D_res_B_minus_Bsym 50 -0.0005 0.0037 -0.0066  0.0073    22.0
LightGBM            tau_A_B 50  0.7187 0.0095  0.7026  0.7386     NaN
LightGBM         tau_A_Bsym 50  0.7575 0.0101  0.7406  0.7805     NaN
      RF  D_total_B_minus_A 50  0.0257 0.0089  0.0108  0.0415    50.0
      RF D_sym_Bsym_minus_A 50  0.0236 0.0078  0.0115  0.0365    50.0
      RF D_res_B_minus_Bsym 50  0.00

## Section L — Legacy cells (historical Branch B matrix; not used for reported results)
These cells use `VIB_FLDC_60D.csv`, the earlier Branch B matrix taken from the historical extract. Its aggregation rules differ from Branch A, so its contrasts (for example B − A = 0.050446) are not reported. See the README, section on feature-pipeline equivalence.

A, B, C dùng đúng cùng 94.453 khách hàng.

Chia cùng một train/test 80:20, random_state=42.

Tạo cùng một 5-fold Stratified CV trên tập train.

Cấu hình mô hình giống nhau: XGBoost, không sampling, missing numeric → 0, categorical → "MISSING", One-Hot.

A = 18 biến; B = 18 biến; C = 19 biến với TENURE_AT_CUTOFF.
Test metrics đầy đủ: AUC, KS, Recall, Precision, F1, Accuracy, Brier, BSS, ECE, MCC, Specificity, NPV, TP/FP/TN/FN.
Ranking Top 10/20/30: Precision, Lift, Capture.
Decile D1–D10.

CV 5 fold đầy đủ tất cả chỉ số.

SHAP cho cả A/B/C, quy về feature gốc.

So sánh A–B, B–C, A–C bằng paired bootstrap 95% CI + paired DeLong.

Không dùng train_test_split() nữa.
Dùng trực tiếp:

FINAL_ABC_MATCHED_E2/exact_E2_shared_train_test_split.csv

FINAL_ABC_MATCHED_E2/exact_E2_shared_CV_folds.csv

Dùng _E2_ORDER để khôi phục đúng thứ tự TRAIN của E2 cũ.

A/B/C dùng đúng cùng 75,562 train + 18,891 test.

A/B/C dùng đúng cùng 5 CV folds.

Cấu hình mô hình đúng E2: XGBoost + None.

Dùng đúng 18 feature của A/B, C thêm TENURE_AT_CUTOFF.

Có kiểm tra A với kết quả E2 cũ.

Có đầy đủ CV, test metrics, ranking, decile, SHAP, bootstrap paired ΔAUC, DeLong và Excel.

In [ ]:
# ============================================================
# VIB FLDC-60D
# FINAL HARMONIZED A / B / C
# EXACT E2-LOCKED EXPERIMENT
# ============================================================
#
# GOLDEN E2 FILES:
#
#   FINAL_ABC_MATCHED_E2/
#       exact_E2_shared_train_test_split.csv
#       exact_E2_shared_CV_folds.csv
#
# ============================================================
#
# BRANCH A
#   Fixed 60-Day
#   18 predictors
#
# BRANCH B
#   Event-Anchored
#   18 predictors
#
# BRANCH C
#   Event-Anchored + TENURE_AT_CUTOFF
#   19 predictors
#
# ============================================================
#
# SAME MODEL:
#
#   XGBoost + None
#
#   n_estimators      = 300
#   max_depth         = 6
#   learning_rate     = 0.05
#   subsample         = 0.8
#   colsample_bytree  = 0.8
#   objective         = binary:logistic
#   eval_metric       = logloss
#   random_state      = 42
#
# ============================================================
#
# SAME EXPERIMENT:
#
#   Population = 94,453
#   Train      = 75,562
#   Test       = 18,891
#   CV         = exact E2 folds
#
# ============================================================
#
# OUTPUT:
#
#   1. Exact E2 split
#   2. Exact E2 CV
#   3. A/B/C fold-level CV
#   4. A/B/C CV mean ± SD
#   5. A/B/C test metrics
#   6. Confusion matrix metrics
#   7. Ranking 10/20/30%
#   8. Decile table
#   9. Test predictions
#   10. SHAP global importance
#   11. A vs E2 reference
#   12. Pairwise Delta AUC
#   13. Paired bootstrap 95% CI
#   14. Paired DeLong
#   15. Excel workbook
#
# GOOGLE COLAB — COPY AND RUN DIRECTLY
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip -q install xgboost scipy shap openpyxl


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import norm

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    matthews_corrcoef
)

from xgboost import XGBClassifier

import shap

from google.colab import drive

warnings.filterwarnings("ignore")


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

print("=" * 120)
print("MOUNT GOOGLE DRIVE")
print("=" * 120)

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Fintect"
)

if not ROOT.is_dir():
    raise RuntimeError(
        "Không truy cập được Google Drive:\n"
        f"{ROOT}"
    )


# ============================================================
# 3. GLOBAL CONFIGURATION
# ============================================================

SEED = 42

N_SPLITS = 5

DECISION_THRESHOLD = 0.50

SHAP_SAMPLE_SIZE = 5000

N_BOOTSTRAP = 5000

EXPECTED_CUSTOMERS = 94453
EXPECTED_TRAIN = 75562
EXPECTED_TEST = 18891


# ============================================================
# 4. GOLDEN E2 REFERENCE
# ============================================================

E2_REFERENCE = {
    "CV_AUC_MEAN": 0.909226,
    "CV_AUC_SD": 0.005908,
    "TEST_AUC": 0.910242,
    "TEST_KS": 0.673291,
    "TEST_RECALL": 0.168000,
    "PRECISION_AT_10": 0.296825,
    "LIFT_AT_10": 6.408376,
    "CAPTURE_AT_10": 0.641143
}

REFERENCE_TOLERANCE = 0.002


# ============================================================
# 5. FILE CONFIGURATION
# ============================================================

A_FILE = (
    ROOT
    / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

B_FILE = (
    ROOT
    / "VIB_FLDC_60D.csv"
)

C_FILE = (
    ROOT
    / "VIB_E1_BRANCH_C.csv"
)

E2_SPLIT_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

E2_CV_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_CV_folds.csv"
)

OUTPUT_DIR = (
    ROOT
    / "FINAL_ABC_EXACT_E2_FIXED"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 6. ID / TARGET
# ============================================================

ID_COL = "CUSTOMER_NUMBER"

TARGET_A = "TARGET_60D"
TARGET_B = "COUNT_CREDITCARD"
TARGET_C = "COUNT_CREDITCARD"

TENURE_COL = "TENURE_AT_CUTOFF"


# ============================================================
# 7. EXACT FEATURE SET
# ============================================================

BASE_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

FEATURES_A = list(BASE_FEATURES)
FEATURES_B = list(BASE_FEATURES)
FEATURES_C = BASE_FEATURES + [TENURE_COL]

if len(FEATURES_A) != 18:
    raise RuntimeError("A phải có 18 features.")

if len(FEATURES_B) != 18:
    raise RuntimeError("B phải có 18 features.")

if len(FEATURES_C) != 19:
    raise RuntimeError("C phải có 19 features.")

if FEATURES_A != FEATURES_B:
    raise RuntimeError("Feature set A và B không giống nhau.")


# ============================================================
# 8. XGBOOST CONFIGURATION
# ============================================================

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "logloss",
    "objective": "binary:logistic",
    "random_state": SEED,
    "n_jobs": -1
}


def get_xgb():
    return XGBClassifier(
        **XGB_PARAMS
    )


# ============================================================
# 9. STANDARDIZE CUSTOMER ID
# ============================================================

def standardize_customer_id(series):

    result = (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    invalid_values = [
        "",
        "NAN",
        "<NA>",
        "NONE",
        "NULL"
    ]

    result = result.mask(
        result.str.upper().isin(
            invalid_values
        ),
        pd.NA
    )

    return result


# ============================================================
# 10. LOAD BRANCH
# ============================================================

def load_branch(
    file_path,
    target_col,
    branch_name
):

    print("\n" + "-" * 110)
    print(f"LOAD {branch_name}")
    print(file_path)

    if not Path(file_path).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy:\n{file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    required = [
        ID_COL,
        target_col
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{branch_name} thiếu cột:\n{missing}"
        )

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df["TARGET_SHARED"] = pd.to_numeric(
        df[target_col],
        errors="coerce"
    )

    if TENURE_COL in df.columns:
        df[TENURE_COL] = pd.to_numeric(
            df[TENURE_COL],
            errors="coerce"
        )

    df = (
        df
        .dropna(
            subset=[
                ID_COL,
                "TARGET_SHARED"
            ]
        )
        .copy()
    )

    df["TARGET_SHARED"] = (
        df["TARGET_SHARED"]
        .astype(int)
    )

    if not df["TARGET_SHARED"].isin(
        [0, 1]
    ).all():
        raise ValueError(
            f"{branch_name}: TARGET không phải 0/1."
        )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{branch_name}: "
            f"{duplicate_count:,} duplicate customers."
        )

    print("Shape:", df.shape)
    print(
        "Customers:",
        f"{df[ID_COL].nunique():,}"
    )
    print(
        "Positive:",
        int(df["TARGET_SHARED"].sum())
    )

    return df


# ============================================================
# 11. LOAD A / B / C
# ============================================================

print("\n" + "=" * 120)
print("LOAD A / B / C")
print("=" * 120)

df_a = load_branch(
    A_FILE,
    TARGET_A,
    "A"
)

df_b = load_branch(
    B_FILE,
    TARGET_B,
    "B"
)

df_c = load_branch(
    C_FILE,
    TARGET_C,
    "C"
)


# ============================================================
# 12. LOAD EXACT E2 TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 120)
print("LOAD EXACT E2 TRAIN / TEST SPLIT")
print("=" * 120)

if not E2_SPLIT_FILE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_SPLIT_FILE}"
    )

e2_split = pd.read_csv(
    E2_SPLIT_FILE,
    low_memory=False
)

e2_split.columns = (
    e2_split.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_split_columns = [
    ID_COL,
    "TARGET",
    "SPLIT"
]

missing = [
    c
    for c in required_split_columns
    if c not in e2_split.columns
]

if missing:
    raise ValueError(
        "E2 split thiếu cột:\n"
        f"{missing}"
    )

e2_split[ID_COL] = (
    standardize_customer_id(
        e2_split[ID_COL]
    )
)

e2_split["TARGET"] = pd.to_numeric(
    e2_split["TARGET"],
    errors="coerce"
)

e2_split["SPLIT"] = (
    e2_split["SPLIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

e2_split = (
    e2_split
    .dropna(
        subset=[
            ID_COL,
            "TARGET"
        ]
    )
    .copy()
)

e2_split["TARGET"] = (
    e2_split["TARGET"]
    .astype(int)
)


# ============================================================
# 13. LOAD EXACT E2 CV
# ============================================================

print("\n" + "=" * 120)
print("LOAD EXACT E2 CV")
print("=" * 120)

if not E2_CV_FILE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_CV_FILE}"
    )

e2_cv = pd.read_csv(
    E2_CV_FILE,
    low_memory=False
)

e2_cv.columns = (
    e2_cv.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_cv_columns = [
    ID_COL,
    "TARGET",
    "_E2_ORDER",
    "CV_FOLD"
]

missing = [
    c
    for c in required_cv_columns
    if c not in e2_cv.columns
]

if missing:
    raise ValueError(
        "E2 CV thiếu cột:\n"
        f"{missing}"
    )

e2_cv[ID_COL] = (
    standardize_customer_id(
        e2_cv[ID_COL]
    )
)

e2_cv["TARGET"] = pd.to_numeric(
    e2_cv["TARGET"],
    errors="coerce"
)

e2_cv["_E2_ORDER"] = pd.to_numeric(
    e2_cv["_E2_ORDER"],
    errors="coerce"
)

e2_cv["CV_FOLD"] = pd.to_numeric(
    e2_cv["CV_FOLD"],
    errors="coerce"
)

e2_cv = (
    e2_cv
    .dropna(
        subset=[
            ID_COL,
            "TARGET",
            "_E2_ORDER",
            "CV_FOLD"
        ]
    )
    .copy()
)

e2_cv["TARGET"] = (
    e2_cv["TARGET"]
    .astype(int)
)

e2_cv["_E2_ORDER"] = (
    e2_cv["_E2_ORDER"]
    .astype(int)
)

e2_cv["CV_FOLD"] = (
    e2_cv["CV_FOLD"]
    .astype(int)
)


# ============================================================
# 14. POPULATION CHECK
# ============================================================

ids_a = set(df_a[ID_COL])
ids_b = set(df_b[ID_COL])
ids_c = set(df_c[ID_COL])
ids_split = set(e2_split[ID_COL])
ids_cv = set(e2_cv[ID_COL])

print("\n" + "=" * 120)
print("POPULATION CHECK")
print("=" * 120)

print("A:", f"{len(ids_a):,}")
print("B:", f"{len(ids_b):,}")
print("C:", f"{len(ids_c):,}")
print("E2 split:", f"{len(ids_split):,}")
print("E2 CV:", f"{len(ids_cv):,}")

if len(ids_a) != EXPECTED_CUSTOMERS:
    raise RuntimeError(
        "A không có đúng 94,453 customers."
    )

if ids_a != ids_b:
    raise RuntimeError(
        "B population khác A."
    )

if ids_a != ids_c:
    raise RuntimeError(
        "C population khác A."
    )

if ids_a != ids_split:
    raise RuntimeError(
        "E2 split population khác A."
    )


# ============================================================
# 15. TARGET CONSISTENCY
# ============================================================

target_a = (
    df_a[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_A"
        }
    )
)

target_b = (
    df_b[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_B"
        }
    )
)

target_c = (
    df_c[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_C"
        }
    )
)

target_check = (
    target_a
    .merge(
        target_b,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        target_c,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        e2_split[
            [ID_COL, "TARGET"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

target_mismatch = (
    (target_check["TARGET_A"] != target_check["TARGET_B"])
    |
    (target_check["TARGET_A"] != target_check["TARGET_C"])
    |
    (target_check["TARGET_A"] != target_check["TARGET"])
)

print(
    "Target mismatch:",
    int(target_mismatch.sum())
)

if target_mismatch.any():
    raise RuntimeError(
        "Target A/B/C/E2 không giống nhau."
    )


# ============================================================
# 16. EXACT E2 TRAIN / TEST
# ============================================================

train_ids = set(
    e2_split.loc[
        e2_split["SPLIT"] == "TRAIN",
        ID_COL
    ]
)

test_ids = set(
    e2_split.loc[
        e2_split["SPLIT"] == "TEST",
        ID_COL
    ]
)

print("\n" + "=" * 120)
print("EXACT E2 TRAIN / TEST")
print("=" * 120)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

print(
    "Train base rate:",
    f"{e2_split.loc[e2_split['SPLIT']=='TRAIN','TARGET'].mean():.4%}"
)

print(
    "Test base rate:",
    f"{e2_split.loc[e2_split['SPLIT']=='TEST','TARGET'].mean():.4%}"
)

if len(train_ids) != EXPECTED_TRAIN:
    raise RuntimeError(
        "TRAIN không phải 75,562."
    )

if len(test_ids) != EXPECTED_TEST:
    raise RuntimeError(
        "TEST không phải 18,891."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "TRAIN / TEST overlap."
    )

if (train_ids | test_ids) != ids_a:
    raise RuntimeError(
        "TRAIN + TEST không bao phủ population A."
    )


# ============================================================
# 17. EXACT E2 CV ORDER
# ============================================================

e2_cv = (
    e2_cv
    .sort_values("_E2_ORDER")
    .reset_index(drop=True)
)

expected_order = np.arange(
    len(e2_cv)
)

if not np.array_equal(
    e2_cv["_E2_ORDER"].to_numpy(),
    expected_order
):
    raise RuntimeError(
        "_E2_ORDER không liên tục từ 0 đến N-1."
    )

if len(e2_cv) != EXPECTED_TRAIN:
    raise RuntimeError(
        "E2 CV không có đúng 75,562 records."
    )

if set(e2_cv[ID_COL]) != train_ids:
    raise RuntimeError(
        "E2 CV customers khác E2 TRAIN."
    )

if e2_cv["CV_FOLD"].nunique() != N_SPLITS:
    raise RuntimeError(
        "E2 CV không có đủ 5 folds."
    )

print("\n" + "=" * 120)
print("EXACT E2 CV")
print("=" * 120)

print(
    e2_cv[
        "CV_FOLD"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 18. GOLDEN ORDER
# ============================================================

golden_train_order = (
    e2_cv[
        ID_COL
    ]
    .astype(str)
    .tolist()
)

golden_test_order = (
    e2_split.loc[
        e2_split["SPLIT"] == "TEST",
        ID_COL
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 19. PREPROCESSOR
# ============================================================

def build_preprocessor(
    X_train
):

    numeric_cols = (
        X_train
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = (
        X_train
        .select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric_cols
        ),
        (
            "cat",
            categorical_pipe,
            categorical_cols
        )
    ])

    return preprocessor


# ============================================================
# 20. METRICS
# ============================================================

def ks_score(
    y_true,
    prob
):

    fpr, tpr, _ = roc_curve(
        y_true,
        prob
    )

    return float(
        np.max(
            tpr - fpr
        )
    )


def bss_score(
    y_true,
    prob
):

    brier = brier_score_loss(
        y_true,
        prob
    )

    prevalence = float(
        np.mean(y_true)
    )

    reference = (
        prevalence *
        (1 - prevalence)
    )

    if reference <= 0:
        return np.nan

    return float(
        1 -
        brier / reference
    )


def ece_score(
    y_true,
    prob,
    n_bins=10
):

    y_true = np.asarray(y_true)
    prob = np.asarray(prob)

    bins = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:

            mask = (
                (prob >= bins[i])
                &
                (prob <= bins[i + 1])
            )

        else:

            mask = (
                (prob >= bins[i])
                &
                (prob < bins[i + 1])
            )

        if not mask.any():
            continue

        observed = (
            y_true[mask].mean()
        )

        predicted = (
            prob[mask].mean()
        )

        weight = (
            mask.sum() /
            len(y_true)
        )

        ece += (
            weight *
            abs(
                observed -
                predicted
            )
        )

    return float(ece)


def ranking_metrics(
    y_true,
    prob
):

    temp = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "PROB": np.asarray(prob)
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    base_rate = float(
        temp["Y_TRUE"].mean()
    )

    total_positive = float(
        temp["Y_TRUE"].sum()
    )

    rows = []

    for rate in [
        0.10,
        0.20,
        0.30
    ]:

        top_n = max(
            1,
            int(
                np.ceil(
                    len(temp) * rate
                )
            )
        )

        top_df = (
            temp
            .iloc[:top_n]
        )

        precision = float(
            top_df["Y_TRUE"].mean()
        )

        lift = (
            precision / base_rate
            if base_rate > 0
            else np.nan
        )

        capture = (
            float(
                top_df["Y_TRUE"].sum()
            ) /
            total_positive
            if total_positive > 0
            else np.nan
        )

        rows.append({
            "TOP_RATE": rate,
            "TOP_K": f"Top {int(rate * 100)}%",
            "CUSTOMERS": top_n,
            "PRECISION": precision,
            "LIFT": float(lift),
            "CAPTURE": float(capture)
        })

    return pd.DataFrame(rows)


def decile_metrics(
    y_true,
    prob
):

    temp = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "PROB": np.asarray(prob)
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    temp["DECILE"] = pd.qcut(
        np.arange(len(temp)),
        q=10,
        labels=[
            f"D{i}"
            for i in range(1, 11)
        ]
    )

    base_rate = (
        temp["Y_TRUE"].mean()
    )

    result = (
        temp
        .groupby(
            "DECILE",
            observed=False
        )
        .agg(
            CUSTOMERS=(
                "Y_TRUE",
                "count"
            ),
            POSITIVES=(
                "Y_TRUE",
                "sum"
            ),
            OBSERVED_RATE=(
                "Y_TRUE",
                "mean"
            ),
            MEAN_PROBABILITY=(
                "PROB",
                "mean"
            )
        )
        .reset_index()
    )

    result["LIFT"] = (
        result["OBSERVED_RATE"] /
        base_rate
    )

    return result


# ============================================================
# 21. SHAP HELPERS
# ============================================================

def map_transformed_features(
    transformed_names,
    original_features
):

    sorted_features = sorted(
        original_features,
        key=len,
        reverse=True
    )

    rows = []

    for transformed_name in transformed_names:

        clean_name = transformed_name

        if clean_name.startswith("num__"):
            clean_name = clean_name[5:]

        elif clean_name.startswith("cat__"):
            clean_name = clean_name[5:]

        matched = None

        for feature in sorted_features:

            if clean_name == feature:
                matched = feature
                break

            if clean_name.startswith(
                feature + "_"
            ):
                matched = feature
                break

        if matched is None:
            matched = clean_name

        rows.append({
            "TRANSFORMED_FEATURE":
                transformed_name,
            "ORIGINAL_FEATURE":
                matched
        })

    return pd.DataFrame(rows)


def aggregate_shap(
    shap_values,
    mapping,
    original_features
):

    labels = (
        mapping["ORIGINAL_FEATURE"]
        .tolist()
    )

    rows = []

    for feature in original_features:

        indices = [
            i
            for i, label in enumerate(labels)
            if label == feature
        ]

        if not indices:

            rows.append({
                "FEATURE": feature,
                "MEAN_ABS_SHAP": 0.0,
                "MEAN_SIGNED_SHAP": 0.0
            })

            continue

        selected = (
            shap_values[:, indices]
        )

        signed = (
            selected.sum(axis=1)
        )

        absolute = (
            np.abs(selected).sum(axis=1)
        )

        rows.append({
            "FEATURE": feature,
            "MEAN_ABS_SHAP": float(
                absolute.mean()
            ),
            "MEAN_SIGNED_SHAP": float(
                signed.mean()
            )
        })

    result = pd.DataFrame(rows)

    result = (
        result
        .sort_values(
            "MEAN_ABS_SHAP",
            ascending=False
        )
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "RANK",
        np.arange(
            1,
            len(result) + 1
        )
    )

    return result


# ============================================================
# 22. RUN ONE BRANCH
# ============================================================

def run_branch(
    branch_name,
    data,
    feature_cols
):

    print(
        "\n" + "=" * 120
    )
    print(
        f"RUN {branch_name}"
    )
    print(
        "=" * 120
    )

    # --------------------------------------------------------
    # Feature check
    # --------------------------------------------------------

    missing_features = [
        feature
        for feature in feature_cols
        if feature not in data.columns
    ]

    if missing_features:
        raise ValueError(
            f"{branch_name} thiếu features:\n"
            +
            "\n".join(missing_features)
        )

    branch = data.copy()

    branch[ID_COL] = (
        standardize_customer_id(
            branch[ID_COL]
        )
    )

    branch_indexed = (
        branch
        .set_index(
            ID_COL,
            drop=False
        )
    )

    # --------------------------------------------------------
    # Check indexed population
    # --------------------------------------------------------

    if branch_indexed.index.duplicated().any():
        raise RuntimeError(
            f"{branch_name}: duplicate CUSTOMER_NUMBER."
        )

    # --------------------------------------------------------
    # Exact E2 TRAIN order
    # --------------------------------------------------------

    missing_train = (
        set(golden_train_order)
        -
        set(branch_indexed.index)
    )

    if missing_train:
        raise RuntimeError(
            f"{branch_name}: "
            f"thiếu {len(missing_train)} "
            "customers trong TRAIN."
        )

    train_part = (
        branch_indexed
        .loc[golden_train_order]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Exact E2 TEST population
    # --------------------------------------------------------

    missing_test = (
        set(golden_test_order)
        -
        set(branch_indexed.index)
    )

    if missing_test:
        raise RuntimeError(
            f"{branch_name}: "
            f"thiếu {len(missing_test)} "
            "customers trong TEST."
        )

    test_part = (
        branch_indexed
        .loc[golden_test_order]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Exact TRAIN order check
    # --------------------------------------------------------

    actual_train_ids = (
        train_part[ID_COL]
        .astype(str)
        .to_numpy()
    )

    expected_train_ids = np.asarray(
        golden_train_order,
        dtype=str
    )

    if not np.array_equal(
        actual_train_ids,
        expected_train_ids
    ):

        mismatch_positions = np.where(
            actual_train_ids !=
            expected_train_ids
        )[0]

        print(
            "\nTRAIN ORDER MISMATCH"
        )

        print(
            "Mismatch count:",
            len(mismatch_positions)
        )

        for position in (
            mismatch_positions[:20]
        ):

            print(
                f"Position {position}: "
                f"actual={actual_train_ids[position]} | "
                f"expected={expected_train_ids[position]}"
            )

        raise RuntimeError(
            f"{branch_name}: "
            "TRAIN order không khớp _E2_ORDER."
        )

    print(
        f"{branch_name}: "
        "EXACT E2 TRAIN ORDER VERIFIED."
    )

    # --------------------------------------------------------
    # Check against E2 CV file
    # --------------------------------------------------------

    e2_cv_ids = (
        e2_cv[ID_COL]
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        actual_train_ids,
        e2_cv_ids
    ):
        raise RuntimeError(
            f"{branch_name}: "
            "TRAIN order không khớp E2 CV."
        )

    # --------------------------------------------------------
    # X / y
    # --------------------------------------------------------

    X_train = (
        train_part[feature_cols]
        .reset_index(drop=True)
    )

    X_test = (
        test_part[feature_cols]
        .reset_index(drop=True)
    )

    y_train = (
        train_part["TARGET_SHARED"]
        .astype(int)
        .reset_index(drop=True)
    )

    y_test = (
        test_part["TARGET_SHARED"]
        .astype(int)
        .reset_index(drop=True)
    )

    train_customer_ids = (
        train_part[ID_COL]
        .astype(str)
        .reset_index(drop=True)
    )

    test_customer_ids = (
        test_part[ID_COL]
        .astype(str)
        .reset_index(drop=True)
    )

    print(
        "Train:",
        f"{len(X_train):,}"
    )

    print(
        "Test:",
        f"{len(X_test):,}"
    )

    print(
        "Features:",
        len(feature_cols)
    )

    # --------------------------------------------------------
    # Exact E2 CV assignment
    # --------------------------------------------------------

    fold_assignment = (
        e2_cv["CV_FOLD"]
        .to_numpy(dtype=int)
    )

    if len(fold_assignment) != len(X_train):
        raise RuntimeError(
            f"{branch_name}: CV length mismatch."
        )

    # ========================================================
    # FIVE-FOLD CV
    # ========================================================

    cv_rows = []

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        valid_mask = (
            fold_assignment == fold
        )

        train_mask_fold = (
            ~valid_mask
        )

        X_fold_train = (
            X_train.loc[
                train_mask_fold
            ]
            .copy()
        )

        X_fold_valid = (
            X_train.loc[
                valid_mask
            ]
            .copy()
        )

        y_fold_train = (
            y_train.loc[
                train_mask_fold
            ]
            .to_numpy()
        )

        y_fold_valid = (
            y_train.loc[
                valid_mask
            ]
            .to_numpy()
        )

        # ----------------------------------------------------
        # Preprocess
        # ----------------------------------------------------

        preprocessor = build_preprocessor(
            X_fold_train
        )

        X_fold_train_processed = (
            preprocessor
            .fit_transform(
                X_fold_train
            )
        )

        X_fold_valid_processed = (
            preprocessor
            .transform(
                X_fold_valid
            )
        )

        # ----------------------------------------------------
        # XGBoost + None
        # ----------------------------------------------------

        model = get_xgb()

        model.fit(
            X_fold_train_processed,
            y_fold_train
        )

        fold_prob = (
            model
            .predict_proba(
                X_fold_valid_processed
            )[:, 1]
        )

        fold_pred = (
            fold_prob >=
            DECISION_THRESHOLD
        ).astype(int)

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        fold_auc = roc_auc_score(
            y_fold_valid,
            fold_prob
        )

        fold_ks = ks_score(
            y_fold_valid,
            fold_prob
        )

        fold_recall = recall_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_precision = precision_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_accuracy = accuracy_score(
            y_fold_valid,
            fold_pred
        )

        fold_brier = brier_score_loss(
            y_fold_valid,
            fold_prob
        )

        fold_bss = bss_score(
            y_fold_valid,
            fold_prob
        )

        fold_ece = ece_score(
            y_fold_valid,
            fold_prob
        )

        cv_rows.append({
            "FOLD":
                fold,

            "N_VALID":
                len(y_fold_valid),

            "AUC":
                float(fold_auc),

            "KS":
                float(fold_ks),

            "Recall":
                float(fold_recall),

            "Precision":
                float(fold_precision),

            "F1":
                float(fold_f1),

            "Accuracy":
                float(fold_accuracy),

            "Brier":
                float(fold_brier),

            "BSS":
                float(fold_bss),

            "ECE":
                float(fold_ece)
        })

        print(
            f"Fold {fold}: "
            f"AUC={fold_auc:.4f} | "
            f"KS={fold_ks:.4f} | "
            f"Recall={fold_recall:.4f} | "
            f"Precision={fold_precision:.4f} | "
            f"F1={fold_f1:.4f} | "
            f"Brier={fold_brier:.4f}"
        )

        del model
        del preprocessor
        del X_fold_train_processed
        del X_fold_valid_processed

        gc.collect()

    cv_results = pd.DataFrame(
        cv_rows
    )

    # ========================================================
    # FINAL MODEL
    # ========================================================

    final_preprocessor = build_preprocessor(
        X_train
    )

    X_train_processed = (
        final_preprocessor
        .fit_transform(
            X_train
        )
    )

    X_test_processed = (
        final_preprocessor
        .transform(
            X_test
        )
    )

    final_model = get_xgb()

    final_model.fit(
        X_train_processed,
        y_train.to_numpy()
    )

    test_prob = (
        final_model
        .predict_proba(
            X_test_processed
        )[:, 1]
    )

    test_pred = (
        test_prob >=
        DECISION_THRESHOLD
    ).astype(int)

    # ========================================================
    # TEST METRICS
    # ========================================================

    test_auc = roc_auc_score(
        y_test,
        test_prob
    )

    test_ks = ks_score(
        y_test,
        test_prob
    )

    test_recall = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_precision = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_accuracy = accuracy_score(
        y_test,
        test_pred
    )

    test_brier = brier_score_loss(
        y_test,
        test_prob
    )

    test_bss = bss_score(
        y_test,
        test_prob
    )

    test_ece = ece_score(
        y_test,
        test_prob
    )

    test_mcc = matthews_corrcoef(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )
        .ravel()
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    # ========================================================
    # RANKING / DECILES
    # ========================================================

    ranking = ranking_metrics(
        y_test,
        test_prob
    )

    deciles = decile_metrics(
        y_test,
        test_prob
    )

    # ========================================================
    # PREDICTIONS
    # ========================================================

    predictions = pd.DataFrame({
        ID_COL:
            test_customer_ids.to_numpy(),

        "Y_TRUE":
            y_test.to_numpy(),

        "Y_PROB":
            test_prob,

        "Y_PRED":
            test_pred
    })

    # ========================================================
    # SHAP
    # ========================================================

    shap_mapping = pd.DataFrame()

    try:

        shap_n = min(
            SHAP_SAMPLE_SIZE,
            len(X_test_processed)
        )

        rng = np.random.default_rng(
            SEED
        )

        shap_indices = rng.choice(
            len(X_test_processed),
            size=shap_n,
            replace=False
        )

        X_shap = (
            X_test_processed[
                shap_indices
            ]
        )

        explainer = shap.TreeExplainer(
            final_model
        )

        shap_raw = (
            explainer
            .shap_values(
                X_shap
            )
        )

        if isinstance(
            shap_raw,
            list
        ):
            shap_values = np.asarray(
                shap_raw[-1]
            )
        else:
            shap_values = np.asarray(
                shap_raw
            )

        if shap_values.ndim == 3:
            shap_values = (
                shap_values[:, :, -1]
            )

        transformed_names = (
            final_preprocessor
            .get_feature_names_out()
            .tolist()
        )

        shap_mapping = (
            map_transformed_features(
                transformed_names,
                feature_cols
            )
        )

        shap_result = (
            aggregate_shap(
                shap_values,
                shap_mapping,
                feature_cols
            )
        )

        top_shap = str(
            shap_result.iloc[0]["FEATURE"]
        )

    except Exception as e:

        print(
            f"\nSHAP warning for {branch_name}:",
            repr(e)
        )

        shap_result = pd.DataFrame({
            "RANK":
                np.arange(
                    1,
                    len(feature_cols) + 1
                ),

            "FEATURE":
                feature_cols,

            "MEAN_ABS_SHAP":
                np.nan,

            "MEAN_SIGNED_SHAP":
                np.nan
        })

        top_shap = None

    # ========================================================
    # CV SUMMARY
    # ========================================================

    cv_summary_rows = []

    for metric in [
        "AUC",
        "KS",
        "Recall",
        "Precision",
        "F1",
        "Accuracy",
        "Brier",
        "BSS",
        "ECE"
    ]:

        mean_value = float(
            cv_results[metric].mean()
        )

        sd_value = float(
            cv_results[metric].std(
                ddof=1
            )
        )

        cv_summary_rows.append({
            "METRIC":
                metric,

            "MEAN":
                mean_value,

            "SD":
                sd_value,

            "MEAN_PLUS_MINUS_SD":
                (
                    f"{mean_value:.4f} ± "
                    f"{sd_value:.4f}"
                )
        })

    cv_summary = pd.DataFrame(
        cv_summary_rows
    )

    # ========================================================
    # FULL METRICS
    # ========================================================

    metrics = {
        "Branch":
            branch_name,

        "Features":
            len(feature_cols),

        "N_Customers":
            len(data),

        "N_Train":
            len(X_train),

        "N_Test":
            len(X_test),

        "Train_Base_Rate":
            float(y_train.mean()),

        "Test_Base_Rate":
            float(y_test.mean()),

        # CV
        "CV_AUC_Mean":
            float(
                cv_results["AUC"].mean()
            ),

        "CV_AUC_SD":
            float(
                cv_results["AUC"].std(
                    ddof=1
                )
            ),

        "CV_KS_Mean":
            float(
                cv_results["KS"].mean()
            ),

        "CV_KS_SD":
            float(
                cv_results["KS"].std(
                    ddof=1
                )
            ),

        "CV_Recall_Mean":
            float(
                cv_results["Recall"].mean()
            ),

        "CV_Recall_SD":
            float(
                cv_results["Recall"].std(
                    ddof=1
                )
            ),

        "CV_Precision_Mean":
            float(
                cv_results["Precision"].mean()
            ),

        "CV_Precision_SD":
            float(
                cv_results["Precision"].std(
                    ddof=1
                )
            ),

        "CV_F1_Mean":
            float(
                cv_results["F1"].mean()
            ),

        "CV_F1_SD":
            float(
                cv_results["F1"].std(
                    ddof=1
                )
            ),

        "CV_Accuracy_Mean":
            float(
                cv_results["Accuracy"].mean()
            ),

        "CV_Accuracy_SD":
            float(
                cv_results["Accuracy"].std(
                    ddof=1
                )
            ),

        "CV_Brier_Mean":
            float(
                cv_results["Brier"].mean()
            ),

        "CV_Brier_SD":
            float(
                cv_results["Brier"].std(
                    ddof=1
                )
            ),

        "CV_BSS_Mean":
            float(
                cv_results["BSS"].mean()
            ),

        "CV_BSS_SD":
            float(
                cv_results["BSS"].std(
                    ddof=1
                )
            ),

        "CV_ECE_Mean":
            float(
                cv_results["ECE"].mean()
            ),

        "CV_ECE_SD":
            float(
                cv_results["ECE"].std(
                    ddof=1
                )
            ),

        # TEST
        "Test_AUC":
            float(test_auc),

        "Test_KS":
            float(test_ks),

        "Test_Recall":
            float(test_recall),

        "Test_Precision":
            float(test_precision),

        "Test_F1":
            float(test_f1),

        "Test_Accuracy":
            float(test_accuracy),

        "Test_Brier":
            float(test_brier),

        "Test_BSS":
            float(test_bss),

        "Test_ECE":
            float(test_ece),

        "Test_MCC":
            float(test_mcc),

        "Test_Specificity":
            float(specificity),

        "Test_NPV":
            float(npv),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "FN":
            int(fn),

        "Top_SHAP_Feature":
            top_shap
    }

    for _, row in ranking.iterrows():

        pct = int(
            row["TOP_RATE"] * 100
        )

        metrics[
            f"Precision@{pct}%"
        ] = float(
            row["PRECISION"]
        )

        metrics[
            f"Lift@{pct}%"
        ] = float(
            row["LIFT"]
        )

        metrics[
            f"Capture@{pct}%"
        ] = float(
            row["CAPTURE"]
        )

    # ========================================================
    # PRINT
    # ========================================================

    print("\nFINAL TEST RESULT")

    print(
        f"AUC         = {test_auc:.6f}"
    )

    print(
        f"KS          = {test_ks:.6f}"
    )

    print(
        f"Recall      = {test_recall:.6f}"
    )

    print(
        f"Precision   = {test_precision:.6f}"
    )

    print(
        f"F1          = {test_f1:.6f}"
    )

    print(
        f"Accuracy    = {test_accuracy:.6f}"
    )

    print(
        f"Brier       = {test_brier:.6f}"
    )

    print(
        f"BSS         = {test_bss:.6f}"
    )

    print(
        f"ECE         = {test_ece:.6f}"
    )

    print(
        f"MCC         = {test_mcc:.6f}"
    )

    print(
        f"Specificity = {specificity:.6f}"
    )

    print(
        f"NPV         = {npv:.6f}"
    )

    print(
        f"Precision@10 = "
        f"{metrics['Precision@10%']:.6f}"
    )

    print(
        f"Lift@10      = "
        f"{metrics['Lift@10%']:.6f}"
    )

    print(
        f"Capture@10   = "
        f"{metrics['Capture@10%']:.6f}"
    )

    print(
        f"Precision@20 = "
        f"{metrics['Precision@20%']:.6f}"
    )

    print(
        f"Lift@20      = "
        f"{metrics['Lift@20%']:.6f}"
    )

    print(
        f"Capture@20   = "
        f"{metrics['Capture@20%']:.6f}"
    )

    print(
        f"Precision@30 = "
        f"{metrics['Precision@30%']:.6f}"
    )

    print(
        f"Lift@30      = "
        f"{metrics['Lift@30%']:.6f}"
    )

    print(
        f"Capture@30   = "
        f"{metrics['Capture@30%']:.6f}"
    )

    print(
        "Top SHAP:",
        top_shap
    )

    return {
        "metrics":
            metrics,

        "cv":
            cv_results,

        "cv_summary":
            cv_summary,

        "ranking":
            ranking,

        "deciles":
            deciles,

        "predictions":
            predictions,

        "shap":
            shap_result,

        "shap_mapping":
            shap_mapping,

        "model":
            final_model,

        "preprocessor":
            final_preprocessor
    }


# ============================================================
# 23. RUN BRANCH A
# ============================================================

result_a = run_branch(
    "A — Fixed 60-Day",
    df_a,
    FEATURES_A
)

gc.collect()


# ============================================================
# 24. A VS GOLDEN E2
# ============================================================

a_metrics = result_a["metrics"]

reference_rows = [
    {
        "Metric": "CV AUC",
        "E2": E2_REFERENCE["CV_AUC_MEAN"],
        "Current A": a_metrics["CV_AUC_Mean"]
    },
    {
        "Metric": "CV AUC SD",
        "E2": E2_REFERENCE["CV_AUC_SD"],
        "Current A": a_metrics["CV_AUC_SD"]
    },
    {
        "Metric": "Test AUC",
        "E2": E2_REFERENCE["TEST_AUC"],
        "Current A": a_metrics["Test_AUC"]
    },
    {
        "Metric": "KS",
        "E2": E2_REFERENCE["TEST_KS"],
        "Current A": a_metrics["Test_KS"]
    },
    {
        "Metric": "Recall",
        "E2": E2_REFERENCE["TEST_RECALL"],
        "Current A": a_metrics["Test_Recall"]
    },
    {
        "Metric": "Precision@10",
        "E2": E2_REFERENCE["PRECISION_AT_10"],
        "Current A": a_metrics["Precision@10%"]
    },
    {
        "Metric": "Lift@10",
        "E2": E2_REFERENCE["LIFT_AT_10"],
        "Current A": a_metrics["Lift@10%"]
    },
    {
        "Metric": "Capture@10",
        "E2": E2_REFERENCE["CAPTURE_AT_10"],
        "Current A": a_metrics["Capture@10%"]
    }
]

comparison_reference = pd.DataFrame(
    reference_rows
)

comparison_reference[
    "Absolute_Difference"
] = (
    comparison_reference["Current A"]
    -
    comparison_reference["E2"]
).abs()

print(
    "\n" + "=" * 120
)

print(
    "A VS GOLDEN E2 REFERENCE"
)

print(
    "=" * 120
)

print(
    comparison_reference.to_string(
        index=False
    )
)

auc_diff = abs(
    a_metrics["Test_AUC"]
    -
    E2_REFERENCE["TEST_AUC"]
)

ks_diff = abs(
    a_metrics["Test_KS"]
    -
    E2_REFERENCE["TEST_KS"]
)

print(
    "\nTest AUC difference:",
    f"{auc_diff:.8f}"
)

print(
    "KS difference:",
    f"{ks_diff:.8f}"
)

if (
    auc_diff <= REFERENCE_TOLERANCE
    and
    ks_diff <= REFERENCE_TOLERANCE
):

    print(
        "\nA is within the E2 reference tolerance."
    )

else:

    print(
        "\nWARNING: A chưa khớp hoàn toàn số E2 cũ."
    )

    print(
        "Population, split và CV hiện đã được khóa."
    )

    print(
        "Không dừng B/C để giữ được bộ so sánh A/B/C."
    )


# ============================================================
# 25. RUN BRANCH B
# ============================================================

result_b = run_branch(
    "B — Event-Anchored",
    df_b,
    FEATURES_B
)

gc.collect()


# ============================================================
# 26. RUN BRANCH C
# ============================================================

result_c = run_branch(
    "C — Event-Anchored + Tenure",
    df_c,
    FEATURES_C
)

gc.collect()


# ============================================================
# 27. TEST PREDICTIONS
# ============================================================

pred_a = (
    result_a["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

pred_b = (
    result_b["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

pred_c = (
    result_c["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)


# ============================================================
# 28. SAME TEST CUSTOMERS
# ============================================================

test_set_a = set(pred_a[ID_COL])
test_set_b = set(pred_b[ID_COL])
test_set_c = set(pred_c[ID_COL])

print(
    "\n" + "=" * 120
)

print(
    "TEST CUSTOMER VERIFICATION"
)

print(
    "=" * 120
)

print(
    "A == B:",
    test_set_a == test_set_b
)

print(
    "B == C:",
    test_set_b == test_set_c
)

print(
    "A == C:",
    test_set_a == test_set_c
)

if not (
    test_set_a ==
    test_set_b ==
    test_set_c
):
    raise RuntimeError(
        "A/B/C không cùng TEST customer set."
    )


# ============================================================
# 29. SAME TEST LABELS
# ============================================================

label_check = (
    pred_a[
        [ID_COL, "Y_TRUE"]
    ]
    .merge(
        pred_b[
            [ID_COL, "Y_TRUE"]
        ],
        on=ID_COL,
        suffixes=(
            "_A",
            "_B"
        ),
        validate="one_to_one"
    )
    .merge(
        pred_c[
            [ID_COL, "Y_TRUE"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

if (
    (
        label_check["Y_TRUE_A"]
        !=
        label_check["Y_TRUE_B"]
    )
    |
    (
        label_check["Y_TRUE_A"]
        !=
        label_check["Y_TRUE"]
    )
).any():

    raise RuntimeError(
        "TARGET test A/B/C không giống nhau."
    )


# ============================================================
# 30. ALIGN PREDICTIONS FOR PAIRED COMPARISON
# ============================================================

common_test = (
    pred_a[
        [ID_COL, "Y_TRUE"]
    ]
    .copy()
)

aligned_a = (
    common_test
    .merge(
        pred_a[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

aligned_b = (
    common_test
    .merge(
        pred_b[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

aligned_c = (
    common_test
    .merge(
        pred_c[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

y_test_common = (
    common_test[
        "Y_TRUE"
    ]
    .to_numpy(dtype=int)
)

score_a = (
    aligned_a[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)

score_b = (
    aligned_b[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)

score_c = (
    aligned_c[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)


# ============================================================
# 31. PAIRED BOOTSTRAP
# ============================================================

def paired_bootstrap_delta_auc(
    y,
    score_1,
    score_2,
    n_bootstrap=5000,
    seed=42
):

    y = np.asarray(y)
    score_1 = np.asarray(score_1)
    score_2 = np.asarray(score_2)

    n = len(y)

    rng = np.random.default_rng(
        seed
    )

    deltas = []

    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        y_boot = (
            y[indices]
        )

        if np.unique(
            y_boot
        ).size < 2:
            continue

        auc_1 = roc_auc_score(
            y_boot,
            score_1[indices]
        )

        auc_2 = roc_auc_score(
            y_boot,
            score_2[indices]
        )

        deltas.append(
            auc_2 - auc_1
        )

    deltas = np.asarray(
        deltas
    )

    ci_low, ci_high = np.percentile(
        deltas,
        [2.5, 97.5]
    )

    return (
        float(ci_low),
        float(ci_high),
        deltas
    )


# ============================================================
# 32. DELONG
# ============================================================

def compute_midrank(x):

    x = np.asarray(
        x,
        dtype=float
    )

    order = np.argsort(x)

    sorted_x = x[order]

    midranks_sorted = np.empty(
        len(x),
        dtype=float
    )

    start = 0

    while start < len(x):

        end = start

        while (
            end + 1 < len(x)
            and
            sorted_x[end + 1]
            ==
            sorted_x[start]
        ):

            end += 1

        midranks_sorted[
            start:end + 1
        ] = (
            start +
            end +
            2
        ) / 2.0

        start = end + 1

    midranks = np.empty(
        len(x),
        dtype=float
    )

    midranks[order] = (
        midranks_sorted
    )

    return midranks


def fast_delong(
    predictions_sorted_transposed,
    label_1_count
):

    m = int(label_1_count)

    n = (
        predictions_sorted_transposed.shape[1]
        -
        m
    )

    k = (
        predictions_sorted_transposed.shape[0]
    )

    tx = np.empty(
        (k, m)
    )

    ty = np.empty(
        (k, n)
    )

    tz = np.empty(
        (k, m + n)
    )

    for r in range(k):

        prediction = (
            predictions_sorted_transposed[r]
        )

        tx[r] = compute_midrank(
            prediction[:m]
        )

        ty[r] = compute_midrank(
            prediction[m:]
        )

        tz[r] = compute_midrank(
            prediction
        )

    aucs = (
        tz[:, :m].sum(axis=1)
        /
        m
        /
        n
        -
        (m + 1.0) /
        (2.0 * n)
    )

    v01 = (
        tz[:, :m] / n
        -
        tx / n
    )

    v10 = (
        1.0
        -
        (
            tz[:, m:]
            -
            ty
        )
        / m
    )

    sx = np.cov(v01)
    sy = np.cov(v10)

    covariance = (
        sx / m
        +
        sy / n
    )

    return (
        aucs,
        covariance
    )


def paired_delong(
    y,
    score_1,
    score_2
):

    order = np.argsort(-y)

    y_sorted = y[order]

    predictions = np.vstack([
        score_1[order],
        score_2[order]
    ])

    m = int(
        y_sorted.sum()
    )

    aucs, covariance = fast_delong(
        predictions,
        m
    )

    contrast = np.array([
        -1.0,
        1.0
    ])

    delta = float(
        contrast @ aucs
    )

    variance = float(
        contrast
        @ covariance
        @ contrast
    )

    if variance <= 0:
        raise RuntimeError(
            "DeLong variance <= 0."
        )

    se = float(
        np.sqrt(variance)
    )

    z = float(
        delta / se
    )

    p_value = float(
        2 *
        norm.sf(
            abs(z)
        )
    )

    return (
        delta,
        se,
        z,
        p_value
    )


# ============================================================
# 33. PAIRWISE COMPARISON
# ============================================================

def compare_pair(
    name_1,
    name_2,
    score_1,
    score_2
):

    auc_1 = roc_auc_score(
        y_test_common,
        score_1
    )

    auc_2 = roc_auc_score(
        y_test_common,
        score_2
    )

    delta = (
        auc_2 -
        auc_1
    )

    (
        ci_low,
        ci_high,
        bootstrap_values
    ) = paired_bootstrap_delta_auc(
        y_test_common,
        score_1,
        score_2,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )

    (
        delong_delta,
        delong_se,
        delong_z,
        delong_p
    ) = paired_delong(
        y_test_common,
        score_1,
        score_2
    )

    return (
        {
            "Comparison":
                f"{name_2} - {name_1}",

            "AUC_1":
                float(auc_1),

            "AUC_2":
                float(auc_2),

            "Delta_AUC":
                float(delta),

            "Bootstrap_95CI_Lower":
                float(ci_low),

            "Bootstrap_95CI_Upper":
                float(ci_high),

            "DeLong_SE":
                float(delong_se),

            "DeLong_Delta":
                float(delong_delta),

            "DeLong_Z":
                float(delong_z),

            "DeLong_p":
                float(delong_p),

            "Bootstrap_N":
                len(bootstrap_values)
        },
        bootstrap_values
    )


ab_result, bootstrap_ab = compare_pair(
    "A",
    "B",
    score_a,
    score_b
)

bc_result, bootstrap_bc = compare_pair(
    "B",
    "C",
    score_b,
    score_c
)

ac_result, bootstrap_ac = compare_pair(
    "A",
    "C",
    score_a,
    score_c
)

pairwise_table = pd.DataFrame([
    ab_result,
    bc_result,
    ac_result
])


# ============================================================
# 34. FULL METRICS TABLE
# ============================================================

full_metrics = pd.DataFrame([
    result_a["metrics"],
    result_b["metrics"],
    result_c["metrics"]
])


# ============================================================
# 35. PUBLICATION TABLE
# ============================================================

publication_columns = [
    "Branch",
    "Features",
    "N_Customers",
    "N_Train",
    "N_Test",

    "Train_Base_Rate",
    "Test_Base_Rate",

    "CV_AUC_Mean",
    "CV_AUC_SD",

    "CV_KS_Mean",
    "CV_KS_SD",

    "CV_Recall_Mean",
    "CV_Recall_SD",

    "CV_Precision_Mean",
    "CV_Precision_SD",

    "CV_F1_Mean",
    "CV_F1_SD",

    "CV_Accuracy_Mean",
    "CV_Accuracy_SD",

    "CV_Brier_Mean",
    "CV_Brier_SD",

    "CV_BSS_Mean",
    "CV_BSS_SD",

    "CV_ECE_Mean",
    "CV_ECE_SD",

    "Test_AUC",
    "Test_KS",
    "Test_Recall",
    "Test_Precision",
    "Test_F1",
    "Test_Accuracy",
    "Test_Brier",
    "Test_BSS",
    "Test_ECE",
    "Test_MCC",
    "Test_Specificity",
    "Test_NPV",

    "Precision@10%",
    "Lift@10%",
    "Capture@10%",

    "Precision@20%",
    "Lift@20%",
    "Capture@20%",

    "Precision@30%",
    "Lift@30%",
    "Capture@30%",

    "TP",
    "FP",
    "TN",
    "FN",

    "Top_SHAP_Feature"
]

publication_table = (
    full_metrics[
        publication_columns
    ]
    .copy()
)


# ============================================================
# 36. SAVE POPULATION / SPLIT / CV
# ============================================================

population_table = (
    target_check[
        [
            ID_COL,
            "TARGET_A"
        ]
    ]
    .rename(
        columns={
            "TARGET_A":
                "TARGET"
        }
    )
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

population_table.to_csv(
    OUTPUT_DIR /
    "ABC_common_population_94453.csv",
    index=False,
    encoding="utf-8-sig"
)

e2_split.to_csv(
    OUTPUT_DIR /
    "EXACT_E2_shared_train_test_split_used.csv",
    index=False,
    encoding="utf-8-sig"
)

e2_cv.to_csv(
    OUTPUT_DIR /
    "EXACT_E2_shared_CV_folds_used.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 37. SAVE TEST PREDICTIONS
# ============================================================

pred_a.to_csv(
    OUTPUT_DIR /
    "Branch_A_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

pred_b.to_csv(
    OUTPUT_DIR /
    "Branch_B_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

pred_c.to_csv(
    OUTPUT_DIR /
    "Branch_C_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 38. SAVE CV / RANKING / DECILES / SHAP
# ============================================================

for branch_code, result in [
    ("A", result_a),
    ("B", result_b),
    ("C", result_c)
]:

    result["cv"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_CV.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["cv_summary"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_CV_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["ranking"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_ranking.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["deciles"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_deciles.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["shap"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_SHAP.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 39. SAVE MAIN METRICS
# ============================================================

full_metrics.to_csv(
    OUTPUT_DIR /
    "ABC_full_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

publication_table.to_csv(
    OUTPUT_DIR /
    "ABC_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)

pairwise_table.to_csv(
    OUTPUT_DIR /
    "ABC_pairwise_DeltaAUC.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_reference.to_csv(
    OUTPUT_DIR /
    "A_vs_E2_reference.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 40. SAVE BOOTSTRAP DISTRIBUTIONS
# ============================================================

pd.DataFrame({
    "Delta_AUC_B_minus_A":
        bootstrap_ab
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_B_minus_A.csv",
    index=False
)

pd.DataFrame({
    "Delta_AUC_C_minus_B":
        bootstrap_bc
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_C_minus_B.csv",
    index=False
)

pd.DataFrame({
    "Delta_AUC_C_minus_A":
        bootstrap_ac
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_C_minus_A.csv",
    index=False
)


# ============================================================
# 41. EXCEL WORKBOOK
# ============================================================

excel_file = (
    OUTPUT_DIR /
    "ABC_FINAL_EXACT_E2_RESULTS.xlsx"
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    population_table.to_excel(
        writer,
        sheet_name="Population",
        index=False
    )

    e2_split.to_excel(
        writer,
        sheet_name="Exact E2 Split",
        index=False
    )

    e2_cv.to_excel(
        writer,
        sheet_name="Exact E2 CV",
        index=False
    )

    comparison_reference.to_excel(
        writer,
        sheet_name="A vs E2",
        index=False
    )

    publication_table.to_excel(
        writer,
        sheet_name="Publication Table",
        index=False
    )

    full_metrics.to_excel(
        writer,
        sheet_name="Full Metrics",
        index=False
    )

    pairwise_table.to_excel(
        writer,
        sheet_name="Pairwise DeltaAUC",
        index=False
    )

    for branch_code, result in [
        ("A", result_a),
        ("B", result_b),
        ("C", result_c)
    ]:

        result["cv"].to_excel(
            writer,
            sheet_name=f"{branch_code} CV",
            index=False
        )

        result["cv_summary"].to_excel(
            writer,
            sheet_name=f"{branch_code} CV Summary",
            index=False
        )

        result["ranking"].to_excel(
            writer,
            sheet_name=f"{branch_code} Ranking",
            index=False
        )

        result["deciles"].to_excel(
            writer,
            sheet_name=f"{branch_code} Deciles",
            index=False
        )

        result["predictions"].to_excel(
            writer,
            sheet_name=f"{branch_code} Predictions",
            index=False
        )

        result["shap"].to_excel(
            writer,
            sheet_name=f"{branch_code} SHAP",
            index=False
        )


# ============================================================
# 42. FINAL VALIDATION
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "FINAL VALIDATION"
)

print(
    "=" * 150
)

# Population
if len(ids_a) != EXPECTED_CUSTOMERS:
    raise RuntimeError(
        "Population A không đúng."
    )

if len(train_ids) != EXPECTED_TRAIN:
    raise RuntimeError(
        "TRAIN population không đúng."
    )

if len(test_ids) != EXPECTED_TEST:
    raise RuntimeError(
        "TEST population không đúng."
    )

# Same test customers
if test_set_a != test_set_b:
    raise RuntimeError(
        "TEST A != TEST B."
    )

if test_set_b != test_set_c:
    raise RuntimeError(
        "TEST B != TEST C."
    )

# Same test labels
y_true_a = (
    pred_a["Y_TRUE"]
    .to_numpy(dtype=int)
)

y_true_b = (
    pred_b["Y_TRUE"]
    .to_numpy(dtype=int)
)

y_true_c = (
    pred_c["Y_TRUE"]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    y_true_a,
    y_true_b
):
    raise RuntimeError(
        "Y_TRUE A != B."
    )

if not np.array_equal(
    y_true_a,
    y_true_c
):
    raise RuntimeError(
        "Y_TRUE A != C."
    )

# Test duplicate check
for code, prediction in [
    ("A", pred_a),
    ("B", pred_b),
    ("C", pred_c)
]:

    duplicate_count = int(
        prediction[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise RuntimeError(
            f"Branch {code} có "
            f"{duplicate_count} "
            "duplicate test customers."
        )

# Exact E2 CV length
if len(e2_cv) != EXPECTED_TRAIN:
    raise RuntimeError(
        "E2 CV length không đúng."
    )

# Exact E2 order
if not np.array_equal(
    e2_cv["_E2_ORDER"].to_numpy(),
    np.arange(
        EXPECTED_TRAIN
    )
):
    raise RuntimeError(
        "E2_ORDER không đúng."
    )

print(
    "Population check       : PASS"
)

print(
    "TRAIN population       : PASS"
)

print(
    "TEST population        : PASS"
)

print(
    "A/B/C TEST customers   : PASS"
)

print(
    "A/B/C TEST labels      : PASS"
)

print(
    "No duplicate TEST IDs  : PASS"
)

print(
    "Exact E2 TRAIN order   : PASS"
)

print(
    "Exact E2 CV            : PASS"
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "FINAL A / B / C — EXACT E2"
)

print(
    "=" * 150
)

print(
    "\nPopulation:",
    f"{EXPECTED_CUSTOMERS:,}"
)

print(
    "Train:",
    f"{EXPECTED_TRAIN:,}"
)

print(
    "Test:",
    f"{EXPECTED_TEST:,}"
)

print(
    "\nA TEST AUC:",
    f"{result_a['metrics']['Test_AUC']:.6f}"
)

print(
    "B TEST AUC:",
    f"{result_b['metrics']['Test_AUC']:.6f}"
)

print(
    "C TEST AUC:",
    f"{result_c['metrics']['Test_AUC']:.6f}"
)

print(
    "\nA TEST KS:",
    f"{result_a['metrics']['Test_KS']:.6f}"
)

print(
    "B TEST KS:",
    f"{result_b['metrics']['Test_KS']:.6f}"
)

print(
    "C TEST KS:",
    f"{result_c['metrics']['Test_KS']:.6f}"
)

print(
    "\n" +
    "A/B/C same TEST customers:",
    test_set_a == test_set_b == test_set_c
)


# ============================================================
# 44. PAIRWISE RESULTS
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "PAIRWISE DELTA AUC"
)

print(
    "=" * 150
)

print(
    pairwise_table[
        [
            "Comparison",
            "AUC_1",
            "AUC_2",
            "Delta_AUC",
            "Bootstrap_95CI_Lower",
            "Bootstrap_95CI_Upper",
            "DeLong_Delta",
            "DeLong_Z",
            "DeLong_p"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 45. PUBLICATION SUMMARY
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "PUBLICATION SUMMARY"
)

print(
    "=" * 150
)

print(
    publication_table.to_string(
        index=False
    )
)


# ============================================================
# 46. A VS E2 FINAL
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "A VS GOLDEN E2 REFERENCE"
)

print(
    "=" * 150
)

print(
    comparison_reference.to_string(
        index=False
    )
)

print(
    "\nTest AUC absolute difference:",
    f"{auc_diff:.8f}"
)

print(
    "KS absolute difference:",
    f"{ks_diff:.8f}"
)


# ============================================================
# 47. OUTPUT
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "COMPLETED"
)

print(
    "=" * 150
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\nExcel:"
)

print(
    excel_file
)

print(
    "\nExact E2 split used:"
)

print(
    E2_SPLIT_FILE
)

print(
    "\nExact E2 CV used:"
)

print(
    E2_CV_FILE
)

print(
    "\nALL FINAL VALIDATION CHECKS PASSED."
)

MOUNT GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOAD A / B / C

--------------------------------------------------------------------------------------------------------------
LOAD A
/content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv
Shape: (94453, 21)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD B
/content/drive/MyDrive/Fintect/VIB_FLDC_60D.csv
Shape: (94453, 25)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD C
/content/drive/MyDrive/Fintect/VIB_E1_BRANCH_C.csv
Shape: (94453, 28)
Customers: 94,453
Positive: 4374

LOAD EXACT E2 TRAIN / TEST SPLIT

LOAD EXACT E2 CV

POPULATION CHECK
A: 94,453
B: 94,453
C: 94,453
E2 split: 94,453
E2 CV: 75,562


In [ ]:
# ============================================================
# TABLE 10 — SAME-POPULATION DECOMPOSITION
#
# Branch A : Fixed first 60 days       — 18 predictors
# Branch B : Event anchored            — 18 predictors
# Branch C : Event anchored + tenure   — 19 predictors
#
# CONTROLLED EXPERIMENT
# ------------------------------------------------------------
# FIXED ACROSS A / B / C:
#   - Same 94,453 customers
#   - Same target
#   - Same train/test customers
#   - Same 5-fold CV folds
#   - Same XGBoost configuration
#   - No sampling
#   - Same preprocessing
#
# ONLY DIFFERENCE:
#   A/B/C feature construction
#
# OUTPUT:
#   - Table 10
#   - A/B/C detailed CV
#   - A/B/C test predictions
#   - Pairwise Delta AUC
#   - Excel workbook
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip -q install xgboost openpyxl


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    brier_score_loss
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

print("=" * 110)
print("MOUNT GOOGLE DRIVE")
print("=" * 110)

drive.mount(
    "/content/drive",
    force_remount=False
)

MYDRIVE_ROOT = "/content/drive/MyDrive"

if not Path(MYDRIVE_ROOT).is_dir():
    raise RuntimeError(
        f"Không truy cập được Google Drive:\n{MYDRIVE_ROOT}"
    )


# ============================================================
# 3. CONFIGURATION
# ============================================================

ROOT = Path(
    MYDRIVE_ROOT
) / "Fintect"

SEED = 42

TEST_SIZE = 0.20
N_SPLITS = 5

DECISION_THRESHOLD = 0.50


# ============================================================
# 4. INPUT FILES
# ============================================================

BRANCH_A_FILE = (
    ROOT
    / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

BRANCH_B_FILE = (
    ROOT
    / "VIB_FLDC_60D.csv"
)

BRANCH_C_FILE = (
    ROOT
    / "VIB_E1_BRANCH_C.csv"
)


# ============================================================
# 5. GOLDEN E2 SPLIT / CV
# ============================================================
#
# Đây là các file đã được kiểm tra:
#
# exact_E2_shared_train_test_split.csv
# exact_E2_shared_CV_folds.csv
#
# Không tạo split mới.
# ============================================================

E2_SPLIT_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

E2_CV_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_CV_folds.csv"
)


# ============================================================
# 6. OUTPUT
# ============================================================

OUTPUT_DIR = (
    ROOT
    / "TABLE_10_SAME_POPULATION"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE10_FILE = (
    OUTPUT_DIR
    / "Table_10_same_population_decomposition.csv"
)

CV_A_FILE = (
    OUTPUT_DIR
    / "Branch_A_CV.csv"
)

CV_B_FILE = (
    OUTPUT_DIR
    / "Branch_B_CV.csv"
)

CV_C_FILE = (
    OUTPUT_DIR
    / "Branch_C_CV.csv"
)

PRED_A_FILE = (
    OUTPUT_DIR
    / "Branch_A_test_predictions.csv"
)

PRED_B_FILE = (
    OUTPUT_DIR
    / "Branch_B_test_predictions.csv"
)

PRED_C_FILE = (
    OUTPUT_DIR
    / "Branch_C_test_predictions.csv"
)

PAIRWISE_FILE = (
    OUTPUT_DIR
    / "Table_10_pairwise_delta_auc.csv"
)

EXCEL_FILE = (
    OUTPUT_DIR
    / "Table_10_same_population_decomposition.xlsx"
)


# ============================================================
# 7. COLUMN CONFIGURATION
# ============================================================

ID_COL = "CUSTOMER_NUMBER"

TENURE_COL = "TENURE_AT_CUTOFF"


BASE_FEATURES = [

    "CLIENT_SEX",

    "EB_REGISTER_CHANNEL",

    "SMS",

    "VERIFY_METHOD",

    "AGE",

    "LOGIN_PER_ACTIVE_DAY",

    "INTEREST_RATE_RATIO",

    "TRANS_LV1_MODE",

    "TRANS_LV2_MODE",

    "TRANS_AMOUNT_MAX",

    "TRANS_AMOUNT_MEAN",

    "COUNT_CA_ACCT",

    "AVG_CA_BALANCE",

    "COUNT_TD_ACCT",

    "AVG_TD_BALANCE",

    "COUNT_OF_LOAN",

    "AVG_LOAN_AMOUNT",

    "TOTAL_LOAN_AMOUNT"

]

FEATURES_A = list(
    BASE_FEATURES
)

FEATURES_B = list(
    BASE_FEATURES
)

FEATURES_C = (
    BASE_FEATURES
    + [TENURE_COL]
)

assert len(BASE_FEATURES) == 18
assert len(FEATURES_A) == 18
assert len(FEATURES_B) == 18
assert len(FEATURES_C) == 19


# ============================================================
# 8. XGBOOST
# ============================================================
#
# Phải dùng đúng cấu hình E2 / controlled A-B-C.
#
# ============================================================

XGB_PARAMS = {

    "n_estimators": 300,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "objective": "binary:logistic",

    "eval_metric": "logloss",

    "tree_method": "hist",

    "max_bin": 256,

    "random_state": SEED,

    "n_jobs": 2
}


def get_xgb():

    return XGBClassifier(
        **XGB_PARAMS
    )


# ============================================================
# 9. STANDARDIZE ID
# ============================================================

def standardize_customer_id(series):

    result = (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    invalid_values = [
        "",
        "NAN",
        "<NA>",
        "NONE",
        "NULL"
    ]

    result = result.mask(
        result.str.upper().isin(
            invalid_values
        ),
        pd.NA
    )

    return result


# ============================================================
# 10. LOAD BRANCH
# ============================================================

def load_branch(
    file_path,
    branch_name,
    target_col
):

    print("\n" + "-" * 110)
    print(f"LOAD {branch_name}")
    print(file_path)
    print("-" * 110)

    if not Path(file_path).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy file:\n{file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    required = [
        ID_COL,
        target_col
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{branch_name} thiếu cột:\n{missing}"
        )

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df["TARGET_SHARED"] = pd.to_numeric(
        df[target_col],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            ID_COL,
            "TARGET_SHARED"
        ]
    ).copy()

    df["TARGET_SHARED"] = (
        df["TARGET_SHARED"]
        .astype(int)
    )

    if not df["TARGET_SHARED"].isin(
        [0, 1]
    ).all():

        raise ValueError(
            f"{branch_name}: target không phải 0/1."
        )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{branch_name}: "
            f"{duplicate_count:,} duplicate customers."
        )

    print(
        "Shape:",
        df.shape
    )

    print(
        "Customers:",
        f"{df[ID_COL].nunique():,}"
    )

    print(
        "Positive:",
        int(df["TARGET_SHARED"].sum())
    )

    return df


# ============================================================
# 11. LOAD A / B / C
# ============================================================

print("\n" + "=" * 110)
print("LOAD A / B / C")
print("=" * 110)

df_a = load_branch(
    BRANCH_A_FILE,
    "A",
    "TARGET_60D"
)

df_b = load_branch(
    BRANCH_B_FILE,
    "B",
    "COUNT_CREDITCARD"
)

df_c = load_branch(
    BRANCH_C_FILE,
    "C",
    "COUNT_CREDITCARD"
)


# ============================================================
# 12. LOAD GOLDEN E2 SPLIT
# ============================================================

print("\n" + "=" * 110)
print("LOAD GOLDEN E2 TRAIN / TEST SPLIT")
print("=" * 110)

if not E2_SPLIT_FILE.is_file():

    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_SPLIT_FILE}"
    )

split_df = pd.read_csv(
    E2_SPLIT_FILE,
    low_memory=False
)

split_df.columns = (
    split_df.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_split_cols = [
    ID_COL,
    "TARGET",
    "SPLIT"
]

missing = [
    c
    for c in required_split_cols
    if c not in split_df.columns
]

if missing:

    raise ValueError(
        f"Golden split thiếu:\n{missing}"
    )

split_df[ID_COL] = (
    standardize_customer_id(
        split_df[ID_COL]
    )
)

split_df["TARGET"] = pd.to_numeric(
    split_df["TARGET"],
    errors="coerce"
).astype(int)

split_df["SPLIT"] = (
    split_df["SPLIT"]
    .astype("string")
    .str.upper()
    .str.strip()
)


# ============================================================
# 13. LOAD GOLDEN E2 CV
# ============================================================

print("\n" + "=" * 110)
print("LOAD GOLDEN E2 CV")
print("=" * 110)

if not E2_CV_FILE.is_file():

    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_CV_FILE}"
    )

cv_df = pd.read_csv(
    E2_CV_FILE,
    low_memory=False
)

cv_df.columns = (
    cv_df.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_cv_cols = [
    ID_COL,
    "TARGET",
    "CV_FOLD"
]

missing = [
    c
    for c in required_cv_cols
    if c not in cv_df.columns
]

if missing:

    raise ValueError(
        f"Golden CV thiếu:\n{missing}"
    )

cv_df[ID_COL] = (
    standardize_customer_id(
        cv_df[ID_COL]
    )
)

cv_df["TARGET"] = pd.to_numeric(
    cv_df["TARGET"],
    errors="coerce"
).astype(int)

cv_df["CV_FOLD"] = pd.to_numeric(
    cv_df["CV_FOLD"],
    errors="coerce"
).astype(int)


# ============================================================
# 14. POPULATION CHECK
# ============================================================

ids_a = set(
    df_a[ID_COL]
)

ids_b = set(
    df_b[ID_COL]
)

ids_c = set(
    df_c[ID_COL]
)

ids_split = set(
    split_df[ID_COL]
)

ids_cv = set(
    cv_df[ID_COL]
)

print("\n" + "=" * 110)
print("POPULATION CHECK")
print("=" * 110)

print(
    "A:",
    f"{len(ids_a):,}"
)

print(
    "B:",
    f"{len(ids_b):,}"
)

print(
    "C:",
    f"{len(ids_c):,}"
)

print(
    "E2 split:",
    f"{len(ids_split):,}"
)

print(
    "E2 CV:",
    f"{len(ids_cv):,}"
)

if not (
    ids_a
    == ids_b
    == ids_c
    == ids_split
):

    raise RuntimeError(
        "A/B/C/E2 split population không giống nhau."
    )


# ============================================================
# 15. TARGET CONSISTENCY
# ============================================================

target_a = (
    df_a[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_A"
        }
    )
)

target_b = (
    df_b[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_B"
        }
    )
)

target_c = (
    df_c[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_C"
        }
    )
)

targets = (
    target_a
    .merge(
        target_b,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        target_c,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        split_df[
            [ID_COL, "TARGET"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

target_mismatch = (
    (targets["TARGET_A"] != targets["TARGET_B"])
    |
    (targets["TARGET_A"] != targets["TARGET_C"])
    |
    (targets["TARGET_A"] != targets["TARGET"])
)

print(
    "Target mismatch:",
    int(target_mismatch.sum())
)

if target_mismatch.any():

    raise RuntimeError(
        "Target A/B/C/E2 không giống nhau."
    )


# ============================================================
# 16. SHARED TRAIN / TEST
# ============================================================

train_ids = set(
    split_df.loc[
        split_df["SPLIT"] == "TRAIN",
        ID_COL
    ]
)

test_ids = set(
    split_df.loc[
        split_df["SPLIT"] == "TEST",
        ID_COL
    ]
)

print("\n" + "=" * 110)
print("GOLDEN E2 TRAIN / TEST")
print("=" * 110)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

if train_ids & test_ids:

    raise RuntimeError(
        "TRAIN/TEST overlap."
    )

if (
    len(train_ids)
    +
    len(test_ids)
    !=
    len(ids_split)
):

    raise RuntimeError(
        "TRAIN + TEST không bằng population."
    )


# ============================================================
# 17. SHARED CV
# ============================================================

train_cv = cv_df[
    cv_df[ID_COL].isin(
        train_ids
    )
].copy()

if len(train_cv) != len(train_ids):

    raise RuntimeError(
        "Golden CV không chứa đủ TRAIN customers."
    )

print("\n" + "=" * 110)
print("GOLDEN E2 CV")
print("=" * 110)

print(
    train_cv["CV_FOLD"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 18. BUILD PREPROCESSOR
# ============================================================

def build_preprocessor(
    X_train
):

    numeric_cols = (
        X_train
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = (
        X_train
        .select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric_cols
        ),
        (
            "cat",
            categorical_pipe,
            categorical_cols
        )
    ])

    return preprocessor


# ============================================================
# 19. METRICS
# ============================================================

def ks_score(
    y_true,
    probability
):

    fpr, tpr, _ = roc_curve(
        y_true,
        probability
    )

    return float(
        np.max(
            tpr - fpr
        )
    )


def bss_score(
    y_true,
    probability
):

    brier = brier_score_loss(
        y_true,
        probability
    )

    prevalence = float(
        np.mean(
            y_true
        )
    )

    reference = (
        prevalence
        *
        (1 - prevalence)
    )

    if reference <= 0:
        return np.nan

    return float(
        1
        -
        brier / reference
    )


def ranking_metrics(
    y_true,
    probability
):

    temp = pd.DataFrame({

        "Y_TRUE":
            np.asarray(
                y_true
            ),

        "PROB":
            np.asarray(
                probability
            )
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    base_rate = float(
        temp["Y_TRUE"].mean()
    )

    total_positive = int(
        temp["Y_TRUE"].sum()
    )

    rows = []

    for rate in [
        0.10,
        0.20,
        0.30
    ]:

        n_top = max(
            1,
            int(
                np.ceil(
                    len(temp)
                    *
                    rate
                )
            )
        )

        top_df = temp.iloc[
            :n_top
        ]

        precision = float(
            top_df["Y_TRUE"].mean()
        )

        capture = (
            float(
                top_df["Y_TRUE"].sum()
            )
            /
            total_positive
            if total_positive > 0
            else np.nan
        )

        lift = (
            precision
            /
            base_rate
            if base_rate > 0
            else np.nan
        )

        rows.append({

            "TOP_RATE":
                rate,

            "PRECISION":
                precision,

            "LIFT":
                lift,

            "CAPTURE":
                capture
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 20. PREPARE BRANCH
# ============================================================

def prepare_branch(
    df,
    feature_cols
):

    missing = [
        c
        for c in feature_cols
        if c not in df.columns
    ]

    if missing:

        raise ValueError(
            f"Missing features:\n{missing}"
        )

    data = df[
        [ID_COL, "TARGET_SHARED"]
        + feature_cols
    ].copy()

    data = (
        data
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    return data


data_a = prepare_branch(
    df_a,
    FEATURES_A
)

data_b = prepare_branch(
    df_b,
    FEATURES_B
)

data_c = prepare_branch(
    df_c,
    FEATURES_C
)


# ============================================================
# 21. RUN ONE CONTROLLED BRANCH
# ============================================================

def run_branch(
    branch_name,
    data,
    feature_cols
):

    print("\n" + "=" * 110)
    print(f"RUN {branch_name}")
    print("=" * 110)

    # --------------------------------------------------------
    # Verify exact same population
    # --------------------------------------------------------

    branch_ids = set(
        data[ID_COL]
    )

    if branch_ids != ids_split:

        raise RuntimeError(
            f"{branch_name}: "
            "customer population không khớp E2."
        )

    # --------------------------------------------------------
    # Use EXACT E2 train/test IDs
    # --------------------------------------------------------

    train_mask = (
        data[ID_COL]
        .isin(train_ids)
    )

    test_mask = (
        data[ID_COL]
        .isin(test_ids)
    )

    X = data[
        feature_cols
    ].copy()

    y = (
        data["TARGET_SHARED"]
        .astype(int)
    )

    X_train = (
        X.loc[
            train_mask
        ]
        .reset_index(
            drop=True
        )
    )

    X_test = (
        X.loc[
            test_mask
        ]
        .reset_index(
            drop=True
        )
    )

    y_train = (
        y.loc[
            train_mask
        ]
        .reset_index(
            drop=True
        )
    )

    y_test = (
        y.loc[
            test_mask
        ]
        .reset_index(
            drop=True
        )
    )

    train_customer_ids = (
        data.loc[
            train_mask,
            ID_COL
        ]
        .reset_index(
            drop=True
        )
    )

    test_customer_ids = (
        data.loc[
            test_mask,
            ID_COL
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        "Train:",
        f"{len(X_train):,}"
    )

    print(
        "Test:",
        f"{len(X_test):,}"
    )

    print(
        "Features:",
        len(feature_cols)
    )

    # --------------------------------------------------------
    # FOLD RESULTS
    # --------------------------------------------------------

    cv_rows = []

    oof_probability = np.full(
        len(X_train),
        np.nan
    )

    # --------------------------------------------------------
    # EXACT GOLDEN FOLDS
    # --------------------------------------------------------

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        valid_ids = set(
            train_cv.loc[
                train_cv["CV_FOLD"] == fold,
                ID_COL
            ]
        )

        fold_train_ids = (
            train_ids
            -
            valid_ids
        )

        fold_train_mask = (
            train_customer_ids
            .isin(
                fold_train_ids
            )
            .to_numpy()
        )

        fold_valid_mask = (
            train_customer_ids
            .isin(
                valid_ids
            )
            .to_numpy()
        )

        X_fold_train = (
            X_train.loc[
                fold_train_mask
            ]
            .copy()
        )

        X_fold_valid = (
            X_train.loc[
                fold_valid_mask
            ]
            .copy()
        )

        y_fold_train = (
            y_train[
                fold_train_mask
            ]
        )

        y_fold_valid = (
            y_train[
                fold_valid_mask
            ]
        )

        # ----------------------------------------------------
        # PREPROCESS ONLY ON FOLD TRAIN
        # ----------------------------------------------------

        preprocessor = build_preprocessor(
            X_fold_train
        )

        X_fold_train_processed = (
            preprocessor
            .fit_transform(
                X_fold_train
            )
        )

        X_fold_valid_processed = (
            preprocessor
            .transform(
                X_fold_valid
            )
        )

        # ----------------------------------------------------
        # XGBOOST — NO SAMPLING
        # ----------------------------------------------------

        model = get_xgb()

        model.fit(
            X_fold_train_processed,
            y_fold_train
        )

        fold_probability = (
            model
            .predict_proba(
                X_fold_valid_processed
            )[:, 1]
        )

        # ----------------------------------------------------
        # OOF POSITION
        # ----------------------------------------------------

        valid_positions = np.where(
            fold_valid_mask
        )[0]

        oof_probability[
            valid_positions
        ] = fold_probability

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        fold_auc = roc_auc_score(
            y_fold_valid,
            fold_probability
        )

        fold_ks = ks_score(
            y_fold_valid,
            fold_probability
        )

        fold_brier = brier_score_loss(
            y_fold_valid,
            fold_probability
        )

        fold_bss = bss_score(
            y_fold_valid,
            fold_probability
        )

        cv_rows.append({

            "FOLD":
                fold,

            "AUC":
                fold_auc,

            "KS":
                fold_ks,

            "BRIER":
                fold_brier,

            "BSS":
                fold_bss
        })

        print(
            f"Fold {fold}: "
            f"AUC={fold_auc:.4f} | "
            f"KS={fold_ks:.4f} | "
            f"Brier={fold_brier:.4f}"
        )

        del model
        del preprocessor
        del X_fold_train_processed
        del X_fold_valid_processed

        gc.collect()

    cv_results = pd.DataFrame(
        cv_rows
    )

    # --------------------------------------------------------
    # FINAL MODEL ON EXACT 80% TRAIN
    # --------------------------------------------------------

    final_preprocessor = build_preprocessor(
        X_train
    )

    X_train_processed = (
        final_preprocessor
        .fit_transform(
            X_train
        )
    )

    X_test_processed = (
        final_preprocessor
        .transform(
            X_test
        )
    )

    final_model = get_xgb()

    final_model.fit(
        X_train_processed,
        y_train
    )

    test_probability = (
        final_model
        .predict_proba(
            X_test_processed
        )[:, 1]
    )

    # --------------------------------------------------------
    # TEST METRICS
    # --------------------------------------------------------

    test_auc = roc_auc_score(
        y_test,
        test_probability
    )

    test_ks = ks_score(
        y_test,
        test_probability
    )

    test_brier = brier_score_loss(
        y_test,
        test_probability
    )

    test_bss = bss_score(
        y_test,
        test_probability
    )

    ranking = ranking_metrics(
        y_test,
        test_probability
    )

    # --------------------------------------------------------
    # STORE TEST PREDICTIONS
    # --------------------------------------------------------

    predictions = pd.DataFrame({

        ID_COL:
            test_customer_ids
            .to_numpy(),

        "Y_TRUE":
            y_test
            .to_numpy(),

        "Y_PROB":
            test_probability
    })

    predictions = (
        predictions
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    result = {

        "Branch":
            branch_name,

        "Features":
            len(feature_cols),

        "CV_AUC":
            float(
                cv_results["AUC"].mean()
            ),

        "CV_AUC_SD":
            float(
                cv_results["AUC"].std(
                    ddof=1
                )
            ),

        "TEST_AUC":
            float(
                test_auc
            ),

        "KS":
            float(
                test_ks
            ),

        "BSS":
            float(
                test_bss
            ),

        "BRIER":
            float(
                test_brier
            ),

        "Precision@10":
            float(
                ranking.iloc[0]["PRECISION"]
            ),

        "Lift@10":
            float(
                ranking.iloc[0]["LIFT"]
            ),

        "Capture@10":
            float(
                ranking.iloc[0]["CAPTURE"]
            ),

        "Precision@20":
            float(
                ranking.iloc[1]["PRECISION"]
            ),

        "Lift@20":
            float(
                ranking.iloc[1]["LIFT"]
            ),

        "Capture@20":
            float(
                ranking.iloc[1]["CAPTURE"]
            ),

        "Precision@30":
            float(
                ranking.iloc[2]["PRECISION"]
            ),

        "Lift@30":
            float(
                ranking.iloc[2]["LIFT"]
            ),

        "Capture@30":
            float(
                ranking.iloc[2]["CAPTURE"]
            )
    }

    print("\nFINAL TEST RESULT")

    print(
        f"AUC        = {test_auc:.6f}"
    )

    print(
        f"KS         = {test_ks:.6f}"
    )

    print(
        f"Brier      = {test_brier:.6f}"
    )

    print(
        f"BSS        = {test_bss:.6f}"
    )

    print(
        f"Precision@10 = "
        f"{result['Precision@10']:.6f}"
    )

    print(
        f"Lift@10      = "
        f"{result['Lift@10']:.6f}"
    )

    print(
        f"Capture@10   = "
        f"{result['Capture@10']:.6f}"
    )

    return {
        "summary":
            result,

        "cv":
            cv_results,

        "ranking":
            ranking,

        "predictions":
            predictions,

        "model":
            final_model,

        "preprocessor":
            final_preprocessor
    }


# ============================================================
# 22. RUN A / B / C
# ============================================================

result_a = run_branch(
    "A — Fixed first 60 days",
    data_a,
    FEATURES_A
)

gc.collect()

result_b = run_branch(
    "B — Event anchored",
    data_b,
    FEATURES_B
)

gc.collect()

result_c = run_branch(
    "C — Event anchored + tenure",
    data_c,
    FEATURES_C
)

gc.collect()


# ============================================================
# 23. EXTRACT PREDICTIONS
# ============================================================

pred_a = (
    result_a["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

pred_b = (
    result_b["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

pred_c = (
    result_c["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 24. VERIFY SAME TEST CUSTOMERS
# ============================================================

test_set_a = set(
    pred_a[ID_COL]
)

test_set_b = set(
    pred_b[ID_COL]
)

test_set_c = set(
    pred_c[ID_COL]
)

print(
    "\n" + "=" * 110
)

print(
    "TEST CUSTOMER VERIFICATION"
)

print(
    "=" * 110
)

print(
    "A == B:",
    test_set_a == test_set_b
)

print(
    "B == C:",
    test_set_b == test_set_c
)

print(
    "A == C:",
    test_set_a == test_set_c
)

if not (
    test_set_a
    ==
    test_set_b
    ==
    test_set_c
):

    raise RuntimeError(
        "A/B/C không cùng test customers."
    )


# ============================================================
# 25. PREPARE TABLE 10
# ============================================================

summary_df = pd.DataFrame([

    result_a["summary"],

    result_b["summary"],

    result_c["summary"]

])


# ------------------------------------------------------------
# AUC DELTAS
# ------------------------------------------------------------

auc_a = (
    result_a["summary"]["TEST_AUC"]
)

auc_b = (
    result_b["summary"]["TEST_AUC"]
)

auc_c = (
    result_c["summary"]["TEST_AUC"]
)

delta_window = (
    auc_b
    -
    auc_a
)

delta_explicit = (
    auc_c
    -
    auc_b
)


# ============================================================
# 26. TABLE 10 — EXACT PAPER FORMAT
# ============================================================

table10 = pd.DataFrame({

    "Br.": [
        "A",
        "B",
        "C"
    ],

    "Feature construction": [

        "Fixed first 60 days",

        "Event anchored",

        "Event anchored + tenure"
    ],

    "AUC": [

        auc_a,

        auc_b,

        auc_c
    ],

    "ΔAUC": [

        np.nan,

        delta_window,

        delta_explicit
    ],

    "KS": [

        result_a["summary"]["KS"],

        result_b["summary"]["KS"],

        result_c["summary"]["KS"]
    ],

    "Lift@10%": [

        result_a["summary"]["Lift@10"],

        result_b["summary"]["Lift@10"],

        result_c["summary"]["Lift@10"]
    ],

    "C@10%": [

        result_a["summary"]["Capture@10"],

        result_b["summary"]["Capture@10"],

        result_c["summary"]["Capture@10"]
    ],

    "BSS": [

        result_a["summary"]["BSS"],

        result_b["summary"]["BSS"],

        result_c["summary"]["BSS"]
    ]
})


# ============================================================
# 27. PAIRWISE SUMMARY
# ============================================================

pairwise = pd.DataFrame({

    "Comparison": [

        "B - A",

        "C - B",

        "C - A"
    ],

    "AUC_1": [

        auc_a,

        auc_b,

        auc_a
    ],

    "AUC_2": [

        auc_b,

        auc_c,

        auc_c
    ],

    "Delta_AUC": [

        auc_b - auc_a,

        auc_c - auc_b,

        auc_c - auc_a
    ]
})


# ============================================================
# 28. PRINT TABLE 10
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE 10 — SAME-POPULATION DECOMPOSITION"
)

print(
    "=" * 110
)

print(
    table10.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nDwindow = "
    f"{delta_window:.6f}"
)

print(
    "Dexplicit = "
    f"{delta_explicit:.6f}"
)

print(
    "C - A = "
    f"{auc_c - auc_a:.6f}"
)


# ============================================================
# 29. SAVE CSV
# ============================================================

table10.to_csv(
    TABLE10_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_a["cv"].to_csv(
    CV_A_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_b["cv"].to_csv(
    CV_B_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_c["cv"].to_csv(
    CV_C_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_a.to_csv(
    PRED_A_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_b.to_csv(
    PRED_B_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_c.to_csv(
    PRED_C_FILE,
    index=False,
    encoding="utf-8-sig"
)

pairwise.to_csv(
    PAIRWISE_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 30. EXCEL
# ============================================================

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    table10.to_excel(
        writer,
        sheet_name="Table 10",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Full Summary",
        index=False
    )

    pairwise.to_excel(
        writer,
        sheet_name="Pairwise Delta AUC",
        index=False
    )

    result_a["cv"].to_excel(
        writer,
        sheet_name="A CV",
        index=False
    )

    result_b["cv"].to_excel(
        writer,
        sheet_name="B CV",
        index=False
    )

    result_c["cv"].to_excel(
        writer,
        sheet_name="C CV",
        index=False
    )

    pred_a.to_excel(
        writer,
        sheet_name="A Predictions",
        index=False
    )

    pred_b.to_excel(
        writer,
        sheet_name="B Predictions",
        index=False
    )

    pred_c.to_excel(
        writer,
        sheet_name="C Predictions",
        index=False
    )

    split_df.to_excel(
        writer,
        sheet_name="Golden E2 Split",
        index=False
    )

    cv_df.to_excel(
        writer,
        sheet_name="Golden E2 CV",
        index=False
    )


# ============================================================
# 31. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE 10 COMPLETED"
)

print(
    "=" * 110
)

print(
    "\nPopulation:",
    f"{len(ids_split):,}"
)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

print(
    "\nA AUC:",
    f"{auc_a:.6f}"
)

print(
    "B AUC:",
    f"{auc_b:.6f}"
)

print(
    "C AUC:",
    f"{auc_c:.6f}"
)

print(
    "\nDwindow = B - A:",
    f"{delta_window:.6f}"
)

print(
    "Dexplicit = C - B:",
    f"{delta_explicit:.6f}"
)

print(
    "\nSaved Table 10:"
)

print(
    TABLE10_FILE
)

print(
    "\nSaved Excel:"
)

print(
    EXCEL_FILE
)

MOUNT GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOAD A / B / C

--------------------------------------------------------------------------------------------------------------
LOAD A
/content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv
--------------------------------------------------------------------------------------------------------------
Shape: (94453, 21)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD B
/content/drive/MyDrive/Fintect/VIB_FLDC_60D.csv
--------------------------------------------------------------------------------------------------------------
Shape: (94453, 25)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD C
/content/dri

CI95%

In [ ]:

# ============================================================
# ELAPS VIB FLDC-60D — CI 95% FOR A / B / C
# PAIRED BOOTSTRAP — 5,000 REPLICATIONS
# GOOGLE COLAB: COPY ALL INTO ONE CELL AND RUN
# NO EXCEL / NO CSV EXPORT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from google.colab import drive

# ============================================================
# 1. GOOGLE DRIVE AND CONFIGURATION
# ============================================================

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Fintect")

N_BOOTSTRAP = 5000
SEED = 42
EXPECTED_TEST = 18891
THRESHOLD = 0.50

ID = "CUSTOMER_NUMBER"
BRANCHES = ("A", "B", "C")

METRICS = (
    "AUC",
    "KS",
    "Brier",
    "BSS",
    "Recall",
    "Precision",
    "F1",
    "Precision@10%",
    "Capture@10%",
    "Lift@10%"
)

PAIR_METRICS = (
    "AUC",
    "KS",
    "BSS",
    "Precision@10%",
    "Capture@10%"
)

# ============================================================
# 2. FIND EXISTING TEST PREDICTIONS
# ============================================================

FOLDERS = [
    ROOT / "FINAL_ABC_EXACT_E2_FIXED",
    ROOT / "TABLE_10_SAME_POPULATION"
]

REQUIRED_FILES = [
    f"Branch_{b}_test_predictions.csv"
    for b in BRANCHES
]

folder = next(
    (
        f for f in FOLDERS
        if all((f / x).is_file() for x in REQUIRED_FILES)
    ),
    None
)

if folder is None:
    for p in ROOT.rglob("Branch_A_test_predictions.csv"):
        if all(
            (p.parent / x).is_file()
            for x in REQUIRED_FILES
        ):
            folder = p.parent
            break

if folder is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ 3 file test predictions "
        "A/B/C trong " + str(ROOT)
    )

print("Prediction folder:", folder)

# ============================================================
# 3. LOAD AND VALIDATE PREDICTIONS
# ============================================================

def load_predictions(path):
    frame = pd.read_csv(path, low_memory=False)

    frame.columns = (
        frame.columns.astype(str)
        .str.strip()
        .str.upper()
    )

    required = {ID, "Y_TRUE", "Y_PROB"}

    if not required.issubset(frame.columns):
        raise ValueError(
            f"{path.name} thiếu cột: "
            f"{required - set(frame.columns)}"
        )

    frame = frame[
        [ID, "Y_TRUE", "Y_PROB"]
    ].copy()

    frame[ID] = (
        frame[ID]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    frame["Y_TRUE"] = pd.to_numeric(
        frame["Y_TRUE"], errors="coerce"
    )

    frame["Y_PROB"] = pd.to_numeric(
        frame["Y_PROB"], errors="coerce"
    )

    if frame.isna().any().any():
        raise ValueError(
            f"{path.name}: Có dữ liệu thiếu hoặc không hợp lệ."
        )

    if frame[ID].duplicated().any():
        raise ValueError(
            f"{path.name}: Trùng CUSTOMER_NUMBER."
        )

    if not frame["Y_TRUE"].isin([0, 1]).all():
        raise ValueError(
            f"{path.name}: Y_TRUE phải là 0 hoặc 1."
        )

    if not frame["Y_PROB"].between(0, 1).all():
        raise ValueError(
            f"{path.name}: Y_PROB phải nằm trong [0, 1]."
        )

    frame["Y_TRUE"] = frame["Y_TRUE"].astype(np.int8)

    return frame.set_index(ID)


predictions = {
    b: load_predictions(
        folder / f"Branch_{b}_test_predictions.csv"
    )
    for b in BRANCHES
}

base = predictions["A"]

if len(base) != EXPECTED_TEST:
    raise ValueError(
        f"Test size={len(base):,}, "
        f"expected={EXPECTED_TEST:,} (E2)."
    )

y = base["Y_TRUE"].to_numpy(dtype=np.int8)

scores = {}

for b in BRANCHES:
    df = predictions[b]

    if (
        len(df) != len(base)
        or set(df.index) != set(base.index)
    ):
        raise ValueError(
            f"Branch {b}: Test customers không giống A."
        )

    df = df.reindex(base.index)

    if not np.array_equal(
        y, df["Y_TRUE"].to_numpy(dtype=np.int8)
    ):
        raise ValueError(
            f"Branch {b}: Y_TRUE không giống A."
        )

    scores[b] = df["Y_PROB"].to_numpy(
        dtype=np.float64
    )

print("\nVALIDATION PASSED")
print("Test customers :", f"{len(y):,}")
print("Test positives :", f"{int(y.sum()):,}")
print("Prevalence     :", f"{y.mean():.4%}")
print("Same IDs/labels: YES")

# ============================================================
# 4. METRIC FUNCTION
# Handles tied scores correctly for AUC and KS
# ============================================================

def calculate_metrics(y_true, prob):

    n = len(y_true)
    n_pos = int(y_true.sum())
    n_neg = n - n_pos

    if n_pos == 0 or n_neg == 0:
        raise ValueError(
            "AUC/KS cần có cả hai lớp."
        )

    order = np.argsort(
        -prob, kind="mergesort"
    )

    yy = y_true[order]
    pp = prob[order]

    cum_pos = np.cumsum(yy)
    cum_neg = np.arange(1, n + 1) - cum_pos

    last = np.r_[
        np.flatnonzero(np.diff(pp)),
        n - 1
    ]

    tpr = np.r_[
        0.0, cum_pos[last] / n_pos
    ]

    fpr = np.r_[
        0.0, cum_neg[last] / n_neg
    ]

    auc = float(
        np.sum(
            np.diff(fpr)
            * (tpr[1:] + tpr[:-1]) / 2.0
        )
    )

    ks = float(np.max(tpr - fpr))

    brier = float(
        np.mean((prob - y_true) ** 2)
    )

    prevalence = n_pos / n

    bss = float(
        1.0 - brier / (
            prevalence * (1.0 - prevalence)
        )
    )

    predicted = prob >= THRESHOLD

    tp = int(np.count_nonzero(
        predicted & (y_true == 1)
    ))

    fp = int(np.count_nonzero(
        predicted & (y_true == 0)
    ))

    fn = n_pos - tp

    recall = tp / n_pos

    precision = (
        tp / (tp + fp)
        if tp + fp > 0 else 0.0
    )

    f1 = (
        2 * tp / (2 * tp + fp + fn)
        if 2 * tp + fp + fn > 0 else 0.0
    )

    top_n = max(
        1, int(np.ceil(n * 0.10))
    )

    positives_top10 = int(
        cum_pos[top_n - 1]
    )

    precision10 = positives_top10 / top_n
    capture10 = positives_top10 / n_pos
    lift10 = precision10 / prevalence

    return np.array([
        auc,
        ks,
        brier,
        bss,
        recall,
        precision,
        f1,
        precision10,
        capture10,
        lift10
    ], dtype=np.float64)


# ============================================================
# 5. ORIGINAL TEST METRICS
# ============================================================

original = {
    b: calculate_metrics(y, scores[b])
    for b in BRANCHES
}

# ============================================================
# 6. PAIRED BOOTSTRAP 95% CI
# Same sampled indices for A, B and C
# ============================================================

rng = np.random.default_rng(SEED)

boot = {
    b: np.empty(
        (N_BOOTSTRAP, len(METRICS)),
        dtype=np.float64
    )
    for b in BRANCHES
}

print("\nRUNNING PAIRED BOOTSTRAP...")
print("Replications:", N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):

    idx = rng.integers(
        0, len(y), size=len(y)
    )

    y_boot = y[idx]

    if y_boot.min() == y_boot.max():
        raise RuntimeError(
            "Bootstrap sample chỉ có một lớp."
        )

    for b in BRANCHES:
        boot[b][i] = calculate_metrics(
            y_boot,
            scores[b][idx]
        )

    if (
        (i + 1) % 500 == 0
        or i + 1 == N_BOOTSTRAP
    ):
        print(
            f"Completed: {i + 1:,}/{N_BOOTSTRAP:,}"
        )

# ============================================================
# 7. CI 95% FOR INDIVIDUAL BRANCHES
# ============================================================

rows = []

for b in BRANCHES:
    for j, metric in enumerate(METRICS):

        lo, hi = np.percentile(
            boot[b][:, j],
            [2.5, 97.5]
        )

        estimate = original[b][j]

        rows.append({
            "Branch": b,
            "Metric": metric,
            "Estimate": estimate,
            "CI_95_Lower": lo,
            "CI_95_Upper": hi,
            "Estimate [95% CI]": (
                f"{estimate:.4f} "
                f"[{lo:.4f}, {hi:.4f}]"
            )
        })

ci_by_branch = pd.DataFrame(rows)

# ============================================================
# 8. PAIRED DELTA CI 95%
# B - A, C - B, C - A
# ============================================================

rows = []

for first, second in [
    ("A", "B"),
    ("B", "C"),
    ("A", "C")
]:

    for metric in PAIR_METRICS:

        j = METRICS.index(metric)

        delta = (
            original[second][j]
            - original[first][j]
        )

        bootstrap_deltas = (
            boot[second][:, j]
            - boot[first][:, j]
        )

        lo, hi = np.percentile(
            bootstrap_deltas,
            [2.5, 97.5]
        )

        excludes_zero = (
            lo > 0 or hi < 0
        )

        rows.append({
            "Comparison": f"{second} - {first}",
            "Metric": "Delta " + metric,
            "Delta": delta,
            "CI_95_Lower": lo,
            "CI_95_Upper": hi,
            "Delta [95% CI]": (
                f"{delta:+.4f} "
                f"[{lo:+.4f}, {hi:+.4f}]"
            ),
            "CI excludes 0": (
                "YES" if excludes_zero else "NO"
            )
        })

ci_pairs = pd.DataFrame(rows)

# ============================================================
# 9. DISPLAY RESULTS ONLY — NO FILE EXPORT
# ============================================================

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

print("\n" + "=" * 80)
print("TABLE 1: INDIVIDUAL BRANCH 95% CI")
print("=" * 80)

display(
    ci_by_branch[
        [
            "Branch",
            "Metric",
            "Estimate [95% CI]"
        ]
    ]
)

print("\n" + "=" * 80)
print("TABLE 2: PAIRED DELTA 95% CI")
print("=" * 80)

display(
    ci_pairs[
        [
            "Comparison",
            "Metric",
            "Delta [95% CI]",
            "CI excludes 0"
        ]
    ]
)

print("\n" + "=" * 80)
print("PUBLICATION-READY AUC SUMMARY")
print("=" * 80)

display(
    ci_by_branch.loc[
        ci_by_branch["Metric"] == "AUC",
        ["Branch", "Estimate [95% CI]"]
    ].reset_index(drop=True)
)

display(
    ci_pairs.loc[
        ci_pairs["Metric"] == "Delta AUC",
        [
            "Comparison",
            "Delta [95% CI]",
            "CI excludes 0"
        ]
    ].reset_index(drop=True)
)

print("\nCOMPLETED — 95% CI COMPUTATION")
print("Method: paired non-stratified percentile bootstrap")
print("Bootstrap N:", N_BOOTSTRAP)
print("Seed:", SEED)
print("No model retraining.")
print("No Excel or CSV exported.")
print("C = Event-Anchored + Tenure, not B-sym.")


Mounted at /content/drive
Prediction folder: /content/drive/MyDrive/Fintect/FINAL_ABC_EXACT_E2_FIXED

VALIDATION PASSED
Test customers : 18,891
Test positives : 875
Prevalence     : 4.6318%
Same IDs/labels: YES

RUNNING PAIRED BOOTSTRAP...
Replications: 5000
Completed: 500/5,000
Completed: 1,000/5,000
Completed: 1,500/5,000
Completed: 2,000/5,000
Completed: 2,500/5,000
Completed: 3,000/5,000
Completed: 3,500/5,000
Completed: 4,000/5,000
Completed: 4,500/5,000
Completed: 5,000/5,000

TABLE 1: INDIVIDUAL BRANCH 95% CI


,Branch,Metric,Estimate [95% CI]
0,A,AUC,"0.9102 [0.9018, 0.9184]"
1,A,KS,"0.6733 [0.6534, 0.6980]"
2,A,Brier,"0.0346 [0.0326, 0.0366]"
3,A,BSS,"0.2163 [0.1900, 0.2415]"
4,A,Recall,"0.1680 [0.1432, 0.1933]"
5,A,Precision,"0.5288 [0.4691, 0.5890]"
6,A,F1,"0.2550 [0.2210, 0.2880]"
7,A,Precision@10%,"0.2968 [0.2762, 0.3201]"
8,A,Capture@10%,"0.6411 [0.6128, 0.6724]"
9,A,Lift@10%,"6.4084 [6.1250, 6.7204]"



TABLE 2: PAIRED DELTA 95% CI


,Comparison,Metric,Delta [95% CI],CI excludes 0
0,B - A,Delta AUC,"+0.0504 [+0.0428, +0.0582]",YES
1,B - A,Delta KS,"+0.1404 [+0.1153, +0.1625]",YES
2,B - A,Delta BSS,"+0.2100 [+0.1791, +0.2412]",YES
3,B - A,Delta Precision@10%,"+0.0968 [+0.0810, +0.1122]",YES
4,B - A,Delta Capture@10%,"+0.2091 [+0.1767, +0.2401]",YES
5,C - B,Delta AUC,"+0.0062 [+0.0039, +0.0085]",YES
6,C - B,Delta KS,"+0.0065 [-0.0019, +0.0176]",NO
7,C - B,Delta BSS,"+0.0554 [+0.0412, +0.0691]",YES
8,C - B,Delta Precision@10%,"+0.0074 [+0.0000, +0.0143]",NO
9,C - B,Delta Capture@10%,"+0.0160 [+0.0000, +0.0309]",NO



PUBLICATION-READY AUC SUMMARY


,Branch,Estimate [95% CI]
0,A,"0.9102 [0.9018, 0.9184]"
1,B,"0.9607 [0.9551, 0.9661]"
2,C,"0.9669 [0.9619, 0.9717]"


,Comparison,Delta [95% CI],CI excludes 0
0,B - A,"+0.0504 [+0.0428, +0.0582]",YES
1,C - B,"+0.0062 [+0.0039, +0.0085]",YES
2,C - A,"+0.0566 [+0.0490, +0.0645]",YES



COMPLETED — 95% CI COMPUTATION
Method: paired non-stratified percentile bootstrap
Bootstrap N: 5000
Seed: 42
No model retraining.
No Excel or CSV exported.
C = Event-Anchored + Tenure, not B-sym.


In [ ]:
# -*- coding: utf-8 -*-
"""Reproduce the R50 seed-42 B-sym path in VIB60ABC.ipynb.
Run in a single Colab cell. Does NOT overwrite any files.
Only calculates publication CI when exact E2 and R50 seed42 test IDs match,
and A/B/Bsym original AUCs have been reproduced.
"""
from pathlib import Path
import inspect
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Fintect')
ID = 'CUSTOMER_NUMBER'
SEED = 42
N_BOOTSTRAP = 5000
REF = {'A':0.910242, 'Bsym':0.923461, 'B':0.960688}
A_FILE = ROOT/'final_landmark_60d'/'final_dataset_landmark_60d_no_auto_job.csv'
E2_SPLIT = ROOT/'FINAL_ABC_MATCHED_E2'/'exact_E2_shared_train_test_split.csv'
E2_CV = ROOT/'FINAL_ABC_MATCHED_E2'/'exact_E2_shared_CV_folds.csv'
PRED = ROOT/'FINAL_ABC_EXACT_E2_FIXED'
CACHE = ROOT/'VIB_BSYM_CACHE_V1'
FILE_BSYM_E2 = CACHE/'PRIMARY_EXACT_E2'/'Bsym_seed_042.csv'
FILE_BSYM_R50 = CACHE/'PRIMARY_R50'/'Bsym_seed_042.csv'
FEATURES = [
    'CLIENT_SEX','EB_REGISTER_CHANNEL','SMS','VERIFY_METHOD','AGE',
    'LOGIN_PER_ACTIVE_DAY','INTEREST_RATE_RATIO','TRANS_LV1_MODE',
    'TRANS_LV2_MODE','TRANS_AMOUNT_MAX','TRANS_AMOUNT_MEAN',
    'COUNT_CA_ACCT','AVG_CA_BALANCE','COUNT_TD_ACCT',
    'AVG_TD_BALANCE','COUNT_OF_LOAN','AVG_LOAN_AMOUNT',
    'TOTAL_LOAN_AMOUNT'
]

def clean_id(s):
    s = s.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    s = s.mask(s.str.upper().isin(['','NAN','<NA>','NONE','NULL']),pd.NA)
    numeric = s.str.fullmatch(r'[0-9]+',na=False)
    s.loc[numeric] = s.loc[numeric].str.lstrip('0').replace('','0')
    return s

def read(path):
    if not path.is_file(): raise FileNotFoundError(str(path))
    h = pd.read_csv(path,nrows=0)
    raw_col = next((c for c in h.columns if str(c).strip().upper()==ID),None)
    if raw_col is None: raise ValueError(f'{path} missing {ID}')
    d = pd.read_csv(path,low_memory=False,dtype={raw_col:str})
    d.columns = d.columns.astype(str).str.strip().str.upper()
    d[ID] = clean_id(d[ID])
    if d[ID].isna().any() or d[ID].duplicated().any():
        raise ValueError(f'Invalid IDs: {path}')
    return d

# Exactly reproduce A's original row order, not the sorted order of IDs.
a = read(A_FILE)
if len(a)!=94453: raise ValueError('Wrong VIB cohort size')
y = pd.to_numeric(a['TARGET_60D'],errors='raise').to_numpy(dtype=np.int8)
if int(y.sum())!=4374: raise ValueError('Wrong VIB positives')
a_ids = a[ID].tolist()

split = read(E2_SPLIT).set_index(ID).reindex(a_ids)
if split[['SPLIT','TARGET']].isna().any().any():
    raise ValueError('Missing customers in E2 split')
split_y = pd.to_numeric(split['TARGET'],errors='raise').to_numpy(dtype=np.int8)
if not np.array_equal(split_y,y): raise ValueError('Target mismatch E2')
kind = split['SPLIT'].astype('string').str.upper().str.strip().to_numpy()
e2_train = np.flatnonzero(kind=='TRAIN')
e2_test = np.flatnonzero(kind=='TEST')
if (len(e2_train),len(e2_test))!=(75562,18891):
    raise ValueError('Bad E2 train/test sizes')

# The EXACT path to the published 0.923461 in the uploaded notebook:
# run_primary_r50() uses train_test_split(... stratify=y, random_state=42)
r50_train, r50_test = train_test_split(
    np.arange(len(a)),test_size=0.2,stratify=y,random_state=SEED
)
shared_train = set(a[ID].iloc[e2_train])==set(a[ID].iloc[r50_train])
shared_test = set(a[ID].iloc[e2_test])==set(a[ID].iloc[r50_test])
print('\n=== SPLIT/ORDER AUDIT ===')
print('E2 vs R50 seed42 same TRAIN customers:',shared_train)
print('E2 vs R50 seed42 same TEST customers :',shared_test)
print('E2 train order equals R50 order     :',np.array_equal(e2_train,r50_train))
print('E2 test order equals R50 order      :',np.array_equal(e2_test,r50_test))

# Independent E2 saved training order from the original CV file.
cv=read(E2_CV)
if '_E2_ORDER' not in cv: raise ValueError('Missing _E2_ORDER')
cv=cv.sort_values('_E2_ORDER')
if set(cv[ID])!=set(a[ID].iloc[e2_train]):
    raise ValueError('Golden E2 CV customers differ from E2 train set')
pos=pd.Series(np.arange(len(a)),index=a[ID])
golden_train = pos.loc[cv[ID].tolist()].to_numpy(dtype=int)
print('E2 golden order equals R50 order   :',np.array_equal(golden_train,r50_train))

# Check E2 vs R50 source B-sym matrix, without reconstructing features.
bsym={}
for label,path in [('E2',FILE_BSYM_E2),('R50',FILE_BSYM_R50)]:
    d=read(path)
    if set(d[ID])!=set(a_ids) or len(d)!=len(a):
        raise ValueError(f'{label} Bsym wrong customer population')
    d=d.set_index(ID).reindex(a_ids).reset_index()
    if not np.array_equal(
        pd.to_numeric(d['TARGET'],errors='raise').to_numpy(dtype=np.int8),y
    ): raise ValueError(f'{label} Bsym label mismatch')
    missing=set(FEATURES)-set(d.columns)
    if missing: raise ValueError(f'{label} missing {missing}')
    bsym[label]=d

mismatch={}
for col in FEATURES:
    x=bsym['E2'][col]
    z=bsym['R50'][col]
    if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(z):
        equal=np.isclose(x.to_numpy(dtype=float),z.to_numpy(dtype=float),
                         rtol=1e-12,atol=1e-12,equal_nan=True)
    else:
        equal=(x.astype('string').fillna('<NULL>') ==
               z.astype('string').fillna('<NULL>')).to_numpy(dtype=bool)
    mismatch[col]=int((~equal).sum())
print('\n=== B-SYM MATRIX COMPARISON ===')
print('Different features:',sum(n>0 for n in mismatch.values()))
display(pd.DataFrame(
    [{'Feature':k,'Different_customers':v} for k,v in mismatch.items() if v]
).head(18))

if not(shared_train and shared_test):
    raise RuntimeError(
        'R50 seed42 and E2 do not share the same customer split. '
        'Do not mix R50 B-sym with saved A/B E2 predictions.'
    )

# Reproduce original XGB code, preprocessing, train ROW ORDER, seed, and features.
def fit_bsym(matrix,train_idx,test_idx):
    Xtr=matrix.iloc[train_idx][FEATURES].copy()
    Xte=matrix.iloc[test_idx][FEATURES].copy()
    ytr=y[train_idx]
    yte=y[test_idx]
    numeric=Xtr.select_dtypes(include=np.number).columns.tolist()
    categorical=[c for c in FEATURES if c not in numeric]
    transforms=[]
    if numeric:
        transforms.append(('num',Pipeline([
            ('imputer',SimpleImputer(strategy='constant',fill_value=0))
        ]),numeric))
    kw={'handle_unknown':'ignore'}
    if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
        kw['sparse_output']=False
    else:
        kw['sparse']=False
    if categorical:
        transforms.append(('cat',Pipeline([
            ('imputer',SimpleImputer(strategy='constant',fill_value='MISSING')),
            ('onehot',OneHotEncoder(**kw))
        ]),categorical))
    pre=ColumnTransformer(transforms,remainder='drop')
    xxtr=pre.fit_transform(Xtr)
    xxte=pre.transform(Xte)
    model=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=.05,
                        subsample=.8,colsample_bytree=.8,
                        objective='binary:logistic',eval_metric='logloss',
                        tree_method='hist',max_bin=256,random_state=SEED,n_jobs=2)
    model.fit(xxtr,ytr)
    probs=model.predict_proba(xxte)[:,1]
    return float(roc_auc_score(yte,probs)),probs

# First, replicate the source notebook's R50 seed-42 run.
print('\n=== R50 SEED 42 — ORIGINAL REPRODUCTION ===')
auc,prob_r50=fit_bsym(bsym['R50'],r50_train,r50_test)
print(f'B-sym R50 seed42 = {auc:.6f} | notebook = {REF["Bsym"]:.6f}')

if not np.isclose(auc,REF['Bsym'],atol=5e-7,rtol=0):
    print('Investigate XGBoost/sklearn versions or source-code/environment changes.')
    print('This model will NOT be published as the 0.923461 reproduction.')
    print('For comparison, running ORIGINAL B-sym E2 cache with E2 golden order...')
    auc_e2,_=fit_bsym(bsym['E2'],golden_train,e2_test)
    print(f'B-sym E2 cache + golden order = {auc_e2:.6f}')
    raise RuntimeError('B-sym exact reproduction failed; CI intentionally skipped')

# Align to the published A/B TEST predictions (not positional test-order assumptions).
ids_r50=a.iloc[r50_test][ID].tolist()
pred={}
for branch in ('A','B'):
    z=read(PRED/f'Branch_{branch}_test_predictions.csv')
    if set(z[ID])!=set(ids_r50) or len(z)!=18891:
        raise RuntimeError(f'{branch} saved test customers are not R50 seed42 set')
    z=z.set_index(ID).reindex(ids_r50)
    if not np.array_equal(pd.to_numeric(z['Y_TRUE'],errors='raise').to_numpy(dtype=int),
                          y[r50_test]):
        raise RuntimeError(f'{branch} test labels differ')
    pred[branch]=pd.to_numeric(z['Y_PROB'],errors='raise').to_numpy(dtype=float)
    val=roc_auc_score(y[r50_test],pred[branch])
    print(f'Saved {branch} AUC = {val:.6f}')
    if not np.isclose(val,REF[branch],atol=5e-7,rtol=0):
        raise RuntimeError(f'{branch} differs from reference AUC')
pred['Bsym']=prob_r50

# Paired bootstrap: 5000 independent customer resamples; identical indices for 3 models.
print('\n=== PAIRED BOOTSTRAP 5000 ===')
rng=np.random.default_rng(SEED)
ytest=y[r50_test]
auc_point={k:roc_auc_score(ytest,v) for k,v in pred.items()}
reps={k:np.empty(N_BOOTSTRAP,dtype=float) for k in pred}
for i in range(N_BOOTSTRAP):
    ix=rng.integers(0,len(ytest),size=len(ytest))
    yy=ytest[ix]
    if yy.min()==yy.max():
        raise RuntimeError('One-class bootstrap draw')
    for k,v in pred.items():
        reps[k][i]=roc_auc_score(yy,v[ix])
    if (i+1)%500==0: print(f'{i+1:,}/{N_BOOTSTRAP:,}')

rows=[]
for k in ('A','Bsym','B'):
    lo,hi=np.percentile(reps[k],[2.5,97.5])
    rows.append({'Branch':k,'AUC':auc_point[k],'CI_Lower':lo,'CI_Upper':hi})
print('\nINDIVIDUAL AUC CI')
display(pd.DataFrame(rows).round(6))
rows=[]
for name,u,l in [('D_window','B','A'),('D_matched','Bsym','A'),
                 ('D_endpoint','B','Bsym')]:
    lo,hi=np.percentile(reps[u]-reps[l],[2.5,97.5])
    rows.append({'Contrast':name,'Delta_AUC':auc_point[u]-auc_point[l],
                 'CI_Lower':lo,'CI_Upper':hi,'CI_excludes_zero':lo>0 or hi<0})
print('\nPAPER TABLE 5 — CI 95%')
display(pd.DataFrame(rows).round(6))
print('SUCCESS: exact R50 seed42 method, matched IDs, paired 5000 resamples; no files saved')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== SPLIT/ORDER AUDIT ===
E2 vs R50 seed42 same TRAIN customers: True
E2 vs R50 seed42 same TEST customers : True
E2 train order equals R50 order     : False
E2 test order equals R50 order      : False
E2 golden order equals R50 order   : True

=== B-SYM MATRIX COMPARISON ===
Different features: 13


,Feature,Different_customers
0,LOGIN_PER_ACTIVE_DAY,21291
1,INTEREST_RATE_RATIO,21761
2,TRANS_LV1_MODE,4101
3,TRANS_LV2_MODE,4822
4,TRANS_AMOUNT_MAX,9196
5,TRANS_AMOUNT_MEAN,17296
6,COUNT_CA_ACCT,1316
7,AVG_CA_BALANCE,43102
8,COUNT_TD_ACCT,1341
9,AVG_TD_BALANCE,1940



=== R50 SEED 42 — ORIGINAL REPRODUCTION ===
B-sym R50 seed42 = 0.923461 | notebook = 0.923461
Saved A AUC = 0.910242
Saved B AUC = 0.960688

=== PAIRED BOOTSTRAP 5000 ===
500/5,000
1,000/5,000
1,500/5,000
2,000/5,000
2,500/5,000
3,000/5,000
3,500/5,000
4,000/5,000
4,500/5,000
5,000/5,000

INDIVIDUAL AUC CI


,Branch,AUC,CI_Lower,CI_Upper
0,A,0.910242,0.901659,0.918488
1,Bsym,0.923461,0.915531,0.930753
2,B,0.960688,0.954795,0.966144



PAPER TABLE 5 — CI 95%


,Contrast,Delta_AUC,CI_Lower,CI_Upper,CI_excludes_zero
0,D_window,0.050446,0.042733,0.058240,True
1,D_matched,0.013219,0.008389,0.018206,True
2,D_endpoint,0.037227,0.030422,0.044381,True


SUCCESS: exact R50 seed42 method, matched IDs, paired 5000 resamples; no files saved


In [ ]:

# ============================================================
# ELAPS — B-SYM REPRODUCIBILITY AUDIT
# SOURCE: VIB60ABC.ipynb
#
# 1. SPLIT / ORDER AUDIT
# 2. B-SYM MATRIX COMPARISON
# 3. R50 SEED 42 REPRODUCTION
#
# No bootstrap yet. No files exported.
# ============================================================

from pathlib import Path
import inspect
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")

# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path("/content/drive/MyDrive/Fintect")

ID = "CUSTOMER_NUMBER"
SEED = 42

REFERENCE_BSYM = 0.923461

A_FILE = (
    ROOT / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

E2_SPLIT = (
    ROOT / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

E2_CV = (
    ROOT / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_CV_folds.csv"
)

CACHE = ROOT / "VIB_BSYM_CACHE_V1"

BSYM_E2 = (
    CACHE / "PRIMARY_EXACT_E2"
    / "Bsym_seed_042.csv"
)

BSYM_R50 = (
    CACHE / "PRIMARY_R50"
    / "Bsym_seed_042.csv"
)

PRED_FOLDER = ROOT / "FINAL_ABC_EXACT_E2_FIXED"

FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

# ============================================================
# HELPER: READ CSV
# ============================================================

def clean_id(series):

    s = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return s


def read_csv(path):

    if not path.is_file():
        raise FileNotFoundError(str(path))

    header = pd.read_csv(path, nrows=0)

    raw_id = next(
        (
            c for c in header.columns
            if str(c).strip().upper() == ID
        ),
        None
    )

    if raw_id is None:
        raise ValueError(f"Missing ID: {path}")

    df = pd.read_csv(
        path,
        dtype={raw_id: str},
        low_memory=False
    )

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
    )

    df[ID] = clean_id(df[ID])

    if df[ID].isna().any():
        raise ValueError("Missing customer IDs")

    if df[ID].duplicated().any():
        raise ValueError("Duplicate customer IDs")

    return df


# ============================================================
# AUDIT 1 — SPLIT AND ROW ORDER
# ============================================================

print("=" * 75)
print("AUDIT 1 — SPLIT / ORDER")
print("=" * 75)

a = read_csv(A_FILE)

if len(a) != 94453:
    raise ValueError("Wrong VIB cohort size")

y = pd.to_numeric(
    a["TARGET_60D"],
    errors="raise"
).to_numpy(dtype=int)

if y.sum() != 4374:
    raise ValueError("Wrong positive count")

ids = a[ID].tolist()

split = read_csv(E2_SPLIT)

if set(ids) != set(split[ID]):
    raise ValueError("E2 customer population differs")

split = (
    split.set_index(ID)
    .reindex(ids)
)

y_split = pd.to_numeric(
    split["TARGET"],
    errors="raise"
).to_numpy(dtype=int)

if not np.array_equal(y, y_split):
    raise ValueError("E2 target mismatch")

split_labels = (
    split["SPLIT"]
    .astype(str)
    .str.strip()
    .str.upper()
    .to_numpy()
)

if not np.isin(
    split_labels, ["TRAIN", "TEST"]
).all():
    raise ValueError("Invalid E2 split labels")

e2_train = np.flatnonzero(
    split_labels == "TRAIN"
)

e2_test = np.flatnonzero(
    split_labels == "TEST"
)

if (len(e2_train), len(e2_test)) != (75562, 18891):
    raise ValueError("Wrong E2 split sizes")

# R50 seed 42: exactly as original notebook
r50_train, r50_test = train_test_split(
    np.arange(len(a)),
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

same_train = (
    set(e2_train) == set(r50_train)
)

same_test = (
    set(e2_test) == set(r50_test)
)

same_train_order = np.array_equal(
    e2_train, r50_train
)

same_test_order = np.array_equal(
    e2_test, r50_test
)

# Golden E2 train order
cv = read_csv(E2_CV)

cv["_E2_ORDER"] = pd.to_numeric(
    cv["_E2_ORDER"], errors="raise"
)

cv = cv.sort_values("_E2_ORDER")

if (
    len(cv) != len(e2_train)
    or set(cv[ID]) != set(a.iloc[e2_train][ID])
):
    raise ValueError("Golden E2 train IDs mismatch")

id_to_pos = pd.Series(
    np.arange(len(a)),
    index=a[ID].to_numpy()
)

golden_train = (
    id_to_pos.loc[cv[ID]]
    .to_numpy(dtype=int)
)

audit_split = pd.DataFrame([
    {
        "Check": "Same train customers",
        "Result": same_train
    },
    {
        "Check": "Same test customers",
        "Result": same_test
    },
    {
        "Check": "Same train row order",
        "Result": same_train_order
    },
    {
        "Check": "Same test row order",
        "Result": same_test_order
    },
    {
        "Check": "Golden E2 train = R50 order",
        "Result": np.array_equal(
            golden_train, r50_train
        )
    }
])

display(audit_split)

print("Population:", len(a))
print("E2 train/test:", len(e2_train), len(e2_test))
print("R50 train/test:", len(r50_train), len(r50_test))
print("E2 test positives:", int(y[e2_test].sum()))
print("R50 test positives:", int(y[r50_test].sum()))


# ============================================================
# AUDIT 2 — B-SYM FEATURE MATRICES
# ============================================================

print("\n" + "=" * 75)
print("AUDIT 2 — B-SYM MATRIX COMPARISON")
print("=" * 75)

matrices = {}

for label, path in [
    ("E2", BSYM_E2),
    ("R50", BSYM_R50)
]:

    df = read_csv(path)

    if len(df) != 94453:
        raise ValueError(f"{label}: wrong population")

    if set(df[ID]) != set(ids):
        raise ValueError(f"{label}: different IDs")

    df = (
        df.set_index(ID)
        .reindex(ids)
    )

    target = pd.to_numeric(
        df["TARGET"], errors="raise"
    ).to_numpy(dtype=int)

    if not np.array_equal(target, y):
        raise ValueError(f"{label}: target mismatch")

    if not set(FEATURES).issubset(df.columns):
        raise ValueError(f"{label}: missing features")

    matrices[label] = df

rows = []

for col in FEATURES:

    x = matrices["E2"][col]
    z = matrices["R50"][col]

    if (
        pd.api.types.is_numeric_dtype(x)
        and pd.api.types.is_numeric_dtype(z)
    ):

        equal = np.isclose(
            x.to_numpy(dtype=float),
            z.to_numpy(dtype=float),
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True
        )

    else:

        equal = (
            x.astype("string").fillna("<NULL>")
            ==
            z.astype("string").fillna("<NULL>")
        ).to_numpy(dtype=bool)

    rows.append({
        "Feature": col,
        "Different_customers": int((~equal).sum()),
        "Difference_rate": float((~equal).mean())
    })

matrix_audit = pd.DataFrame(rows)

display(matrix_audit)

print(
    "\nNumber of different features:",
    int((matrix_audit["Different_customers"] > 0).sum())
)

print(
    "Identical matrices:",
    bool((matrix_audit["Different_customers"] == 0).all())
)


# ============================================================
# AUDIT 3 — REPRODUCE B-SYM XGBOOST
# ============================================================

print("\n" + "=" * 75)
print("AUDIT 3 — R50 SEED 42 REPRODUCTION")
print("=" * 75)


def fit_bsym(matrix, train_idx, test_idx):

    X_train = (
        matrix.iloc[train_idx][FEATURES].copy()
    )

    X_test = (
        matrix.iloc[test_idx][FEATURES].copy()
    )

    y_train = y[train_idx]
    y_test = y[test_idx]

    numeric = (
        X_train.select_dtypes(include=np.number)
        .columns.tolist()
    )

    categorical = [
        c for c in FEATURES
        if c not in numeric
    ]

    transforms = []

    if numeric:
        transforms.append((
            "numeric",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            numeric
        ))

    encoder_args = {
        "handle_unknown": "ignore"
    }

    if "sparse_output" in inspect.signature(
        OneHotEncoder
    ).parameters:
        encoder_args["sparse_output"] = False
    else:
        encoder_args["sparse"] = False

    if categorical:
        transforms.append((
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="MISSING"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(**encoder_args)
                )
            ]),
            categorical
        ))

    preprocessor = ColumnTransformer(
        transforms,
        remainder="drop"
    )

    Xtr = preprocessor.fit_transform(X_train)
    Xte = preprocessor.transform(X_test)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        random_state=SEED,
        n_jobs=2
    )

    model.fit(Xtr, y_train)

    probabilities = model.predict_proba(
        Xte
    )[:, 1]

    return roc_auc_score(
        y_test, probabilities
    )


print("Training R50 B-sym seed 42...")

auc_r50 = fit_bsym(
    matrices["R50"],
    r50_train,
    r50_test
)

print("Training E2 B-sym with golden order...")

auc_e2 = fit_bsym(
    matrices["E2"],
    golden_train,
    e2_test
)

comparison = pd.DataFrame([
    {
        "Experiment": "Published R50 seed 42",
        "AUC": REFERENCE_BSYM
    },
    {
        "Experiment": "Reproduced R50 seed 42",
        "AUC": auc_r50
    },
    {
        "Experiment": "E2 cache + golden order",
        "AUC": auc_e2
    },
    {
        "Experiment": "Previous reproduction",
        "AUC": 0.921921
    }
])

print("\nAUC COMPARISON")
display(comparison.round(6))

print("\nR50 reproduction error:",
      f"{abs(auc_r50 - REFERENCE_BSYM):.8f}")

if abs(auc_r50 - REFERENCE_BSYM) <= 0.0000005:
    print("R50 SEED 42: AUC REPRODUCED")
else:
    print("R50 SEED 42: NOT EXACTLY REPRODUCED")

if not (same_train and same_test):
    print(
        "WARNING: R50 and E2 have different "
        "customers in train/test."
    )

print("\n" + "=" * 75)
print("AUDIT COMPLETED")
print("=" * 75)

print("No bootstrap performed.")
print("No CSV or Excel exported.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AUDIT 1 — SPLIT / ORDER


,Check,Result
0,Same train customers,True
1,Same test customers,True
2,Same train row order,False
3,Same test row order,False
4,Golden E2 train = R50 order,True


Population: 94453
E2 train/test: 75562 18891
R50 train/test: 75562 18891
E2 test positives: 875
R50 test positives: 875

AUDIT 2 — B-SYM MATRIX COMPARISON


,Feature,Different_customers,Difference_rate
0,CLIENT_SEX,0,0.000000
1,EB_REGISTER_CHANNEL,0,0.000000
2,SMS,0,0.000000
3,VERIFY_METHOD,0,0.000000
4,AGE,0,0.000000
5,LOGIN_PER_ACTIVE_DAY,21291,0.225414
6,INTEREST_RATE_RATIO,21761,0.230390
7,TRANS_LV1_MODE,4101,0.043418
8,TRANS_LV2_MODE,4822,0.051052
9,TRANS_AMOUNT_MAX,9196,0.097361



Number of different features: 13
Identical matrices: False

AUDIT 3 — R50 SEED 42 REPRODUCTION
Training R50 B-sym seed 42...
Training E2 B-sym with golden order...

AUC COMPARISON


,Experiment,AUC
0,Published R50 seed 42,0.923461
1,Reproduced R50 seed 42,0.923461
2,E2 cache + golden order,0.921921
3,Previous reproduction,0.921921



R50 reproduction error: 0.00000030
R50 SEED 42: AUC REPRODUCED

AUDIT COMPLETED
No bootstrap performed.
No CSV or Excel exported.


In [ ]:

# ============================================================
# ELAPS — R50 SEED 42 B-SYM CUTOFF AUDIT
# Compare R50 canonical vs EXACT E2 cache
# Display only — no files exported
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Fintect")

CACHE = ROOT / "VIB_BSYM_CACHE_V1"
SPLIT_FILE = (
    ROOT / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

FILES = {
    "R50_seed42": (
        CACHE / "PRIMARY_R50"
        / "Bsym_cutoff_seed_042.csv"
    ),
    "EXACT_E2": (
        CACHE / "PRIMARY_EXACT_E2"
        / "Bsym_cutoff_seed_042.csv"
    )
}

ID = "CUSTOMER_NUMBER"


def read_data(path):
    df = pd.read_csv(
        path, dtype={ID: str}, low_memory=False
    )
    df.columns = df.columns.str.strip().str.upper()
    df[ID] = df[ID].str.strip().str.replace(
        r"\.0$", "", regex=True
    )

    if df[ID].isna().any() or df[ID].duplicated().any():
        raise ValueError(f"Invalid IDs: {path}")

    return df.set_index(ID)


split = read_data(SPLIT_FILE)

split["TARGET"] = pd.to_numeric(
    split["TARGET"], errors="raise"
).astype(int)

split["SPLIT"] = (
    split["SPLIT"].astype(str).str.upper().str.strip()
)

if len(split) != 94453:
    raise ValueError("Wrong E2 population")


def load_audit(path):
    df = read_data(path)

    if len(df) != len(split):
        raise ValueError("Wrong audit cohort")

    if set(df.index) != set(split.index):
        raise ValueError("Customer IDs differ")

    df = df.reindex(split.index)

    target = pd.to_numeric(
        df["TARGET"], errors="raise"
    ).astype(int)

    if not np.array_equal(
        target.to_numpy(),
        split["TARGET"].to_numpy()
    ):
        raise ValueError("Target mismatch")

    df["SPLIT"] = split["SPLIT"]

    for col in [
        "RELATIONSHIP_DATE",
        "FIRST_CARD_DATE",
        "PROPOSED_CUTOFF",
        "BSYM_CUTOFF",
        "T_END"
    ]:
        df[col] = pd.to_datetime(
            df[col], errors="coerce"
        )

    df["EVENT_DELAY"] = (
        df["FIRST_CARD_DATE"]
        - df["RELATIONSHIP_DATE"]
    ).dt.total_seconds() / 86400

    df["REAL_DURATION"] = (
        df["BSYM_CUTOFF"]
        - df["RELATIONSHIP_DATE"]
    ).dt.total_seconds() / 86400

    df["SAMPLED_DURATION_DAYS"] = pd.to_numeric(
        df["SAMPLED_DURATION_DAYS"],
        errors="coerce"
    )

    df["CAP"] = (
        df["CAPPED_AT_TEND"]
        .astype(str).str.strip().str.lower()
        .map({
            "true": 1, "false": 0,
            "1": 1, "0": 0,
            "1.0": 1, "0.0": 0
        })
    )

    return df


audits = {
    name: load_audit(path)
    for name, path in FILES.items()
}


def describe(x):
    x = np.asarray(x, dtype=float)

    if (
        len(x) == 0
        or not np.isfinite(x).all()
        or (x < 0).any()
    ):
        raise ValueError("Invalid duration values")

    return {
        "Mean": x.mean(),
        "Median": np.median(x),
        "Q1": np.quantile(x, 0.25),
        "Q3": np.quantile(x, 0.75)
    }


rows = []

for version, full in audits.items():

    for scope in ["ALL", "TRAIN", "TEST"]:

        df = (
            full if scope == "ALL"
            else full[full["SPLIT"] == scope]
        )

        pos_df = df[df["TARGET"] == 1]
        neg_df = df[df["TARGET"] == 0]

        pos = pos_df["EVENT_DELAY"].to_numpy(
            dtype=float
        )

        neg_raw = neg_df[
            "SAMPLED_DURATION_DAYS"
        ].to_numpy(dtype=float)

        neg_real = neg_df[
            "REAL_DURATION"
        ].to_numpy(dtype=float)

        cap = neg_df["CAP"].to_numpy(dtype=float)

        if not np.isfinite(cap).all():
            raise ValueError("Invalid CAP values")

        pos_stats = describe(pos)
        raw_stats = describe(neg_raw)
        real_stats = describe(neg_real)

        ks_before = ks_2samp(
            pos, neg_raw
        ).statistic

        ks_after = ks_2samp(
            pos, neg_real
        ).statistic

        rows.append({
            "Version": version,
            "Scope": scope,
            "N_positive": len(pos),
            "N_negative": len(neg_real),
            "Pos_median": pos_stats["Median"],
            "Pos_Q1": pos_stats["Q1"],
            "Pos_Q3": pos_stats["Q3"],
            "Neg_raw_median": raw_stats["Median"],
            "Neg_real_median": real_stats["Median"],
            "Neg_real_Q1": real_stats["Q1"],
            "Neg_real_Q3": real_stats["Q3"],
            "KS_before_cap": ks_before,
            "KS_after_cap": ks_after,
            "Capped_N": int(cap.sum()),
            "Cap_rate": cap.mean()
        })


print("B-SYM DURATION BALANCE: R50 vs E2")
display(pd.DataFrame(rows).round(6))

# ------------------------------------------------------------
# Compare the cutoff decisions for the same customers
# ------------------------------------------------------------

r50 = audits["R50_seed42"]
e2 = audits["EXACT_E2"]

negative = split["TARGET"] == 0

cutoff_diff = (
    r50.loc[negative, "BSYM_CUTOFF"]
    != e2.loc[negative, "BSYM_CUTOFF"]
)

cap_diff = (
    r50.loc[negative, "CAP"]
    != e2.loc[negative, "CAP"]
)

print("\nR50 / E2 CUTOFF COMPARISON")

print(
    "Negative customers with different cutoff:",
    int(cutoff_diff.sum())
)

print(
    "Negative customers with different cap flag:",
    int(cap_diff.sum())
)

print(
    "R50 negative cap rate:",
    round(float(r50.loc[negative, "CAP"].mean()), 6)
)

print(
    "E2 negative cap rate:",
    round(float(e2.loc[negative, "CAP"].mean()), 6)
)

print("\nAUDIT COMPLETED")
print("Canonical version: R50 seed 42")
print("No files exported")


B-SYM DURATION BALANCE: R50 vs E2


,Version,Scope,N_positive,N_negative,Pos_median,Pos_Q1,Pos_Q3,Neg_raw_median,Neg_real_median,Neg_real_Q1,Neg_real_Q3,KS_before_cap,KS_after_cap,Capped_N,Cap_rate
0,R50_seed42,ALL,4374,90079,136.0,91.0,240.0,133.0,106.0,76.0,156.0,0.012276,0.188748,36155,0.401370
1,R50_seed42,TRAIN,3499,72063,134.0,90.0,240.0,133.0,106.0,76.0,156.0,0.004528,0.183379,28860,0.400483
2,R50_seed42,TEST,875,18016,144.0,93.5,242.5,133.5,105.0,76.0,156.0,0.048321,0.215177,7295,0.404918
3,EXACT_E2,ALL,4374,90079,136.0,91.0,240.0,134.0,106.0,76.0,157.0,0.007225,0.187449,36147,0.401281
4,EXACT_E2,TRAIN,3499,72063,134.0,90.0,240.0,134.0,106.0,77.0,156.5,0.002932,0.182219,28891,0.400913
5,EXACT_E2,TEST,875,18016,144.0,93.5,242.5,135.0,106.0,76.0,157.0,0.043769,0.212651,7256,0.402753



R50 / E2 CUTOFF COMPARISON
Negative customers with different cutoff: 67891
Negative customers with different cap flag: 29328
R50 negative cap rate: 0.40137
E2 negative cap rate: 0.401281

AUDIT COMPLETED
Canonical version: R50 seed 42
No files exported
